# Data cleaning and consolidation

Aim of this notebook is to analyse, clean and consolidate the raw data collected for the project.

We will operate on a total of 21 sources openly available on the internet, including official Italian and Spanish statistics, crunchbase financing data and data we have scraped in the previous notebook "00_Web_Scraping" included in the same repository.

The detail and link to the specific source is included in the excel file "Data_sources" included in the repository and valid as per the time of writing (May 2026).

All the raw data is available in the "Raw" subfolder inside the "Data" main one.

The output of the cleaning and consolidation process is stored in the subfolder "Cleaned" and wil be used both for purposes of visualization and creation of machine learning models.

In [1]:
# Importing libraries

import pandas as pd
import numpy as np
import datetime as dt

from pathlib import Path
import os

import matplotlib.pyplot as plt
import seaborn as sns

import re
import unicodedata
from difflib import SequenceMatcher
from rapidfuzz import process, fuzz

Functions

In [2]:
# Dataframe general description

def df_describe(df):
    print("DataFrame shape:", df.shape)
    print("\nDataFrame info:")
    print(df.info())
    print("\nDataFrame description:")
    print(df.describe(include='all'))
    print("\nNull values summary:", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())

In [3]:
# Column-specific description

def print_column_description(df,df_col,column_index):
    print(df_col[column_index])
    df_describe(df[df_col[column_index]])

## Set-up of the cleaning process

In [4]:
# Define path of data sources list

repo_root = Path.cwd().parent
file_name = "Data_sources.xlsx"
file_path = repo_root / "Data" / file_name

excel_path_bio = Path(os.getenv(file_name, file_path))

if not excel_path_bio.exists():
    raise FileNotFoundError(f"Excel file not found at {excel_path_bio}")

In [5]:
# Importing data source

df = pd.read_excel(excel_path_bio, sheet_name="Raw", header=0)
df.head()

,ID,Name,Category,Description,Source,Comments
0,1,PIB_Comunidades_autonomas.xlsx,Economics,"GDP at current market prices 2000-2024, detail...",https://www.ine.es/,NaN
1,2,I_D_Comunidades_autonomas.xlsx,Economics,Indicators of investment and employment in R&d,https://www.ine.es/,NaN
2,3,Población_Comunidades_autonomas.xlsx,Demographics,Population 2021-2023 for region and age-group,https://www.ine.es/,NaN
3,4,recintos_autonomicas_inspire_peninbal_etrs89.shp,Geography,shp for spacial visualisation,https://centrodedescargas.cnig.es/CentroDescar...,NaN
4,5,limits_IT_regions.geojson,Geography,GEOJSON for spacial visualization,https://github.com/openpolis/geojson-italy/blo...,NaN


In [6]:
# Directorio de archivos como diccionario (Nombre, rutas)

files_names = []

for i in df['Name'].values:
    files_names.append(i)

path_list = []
base_path = repo_root / "Data" / "00_Raw"

for i in df['Name'].values:
    path_list.append(str(base_path / i))

files_dict = dict(zip(files_names, path_list))

Definimos categorías de fuentes tal como listado en la biografía, intentando agrupar siempre y donde sea posible y con finalidad de simplificar el analísis.

In [7]:
# Identifico categorías

df['Category'].value_counts()

Category
Finance                 6
Economics               5
Geography               5
Demographics            2
Start-up                2
Crunchbase_start-ups    1
Name: count, dtype: int64

Definimos finalmente una carpeta de exportación de los archivos limpios

In [8]:
export_path = repo_root / "Data" / "01_Cleaned"

## Data cleaning

### Crunchbase_start_ups

#### Initial description

In [9]:
# Importing csv

df = pd.read_csv(files_dict['investments_VC.csv'],encoding="latin1")
df_1 = df.copy()
df.head()

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,"15,00,000",operating,GBR,NaN,London,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,"60,000",operating,USA,TX,Dallas,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
df_describe(df)

DataFrame shape: (54294, 39)

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 54294 entries, 0 to 54293
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             49438 non-null  str    
 1   name                  49437 non-null  str    
 2   homepage_url          45989 non-null  str    
 3   category_list         45477 non-null  str    
 4    market               45470 non-null  str    
 5    funding_total_usd    49438 non-null  str    
 6   status                48124 non-null  str    
 7   country_code          44165 non-null  str    
 8   state_code            30161 non-null  str    
 9   region                44165 non-null  str    
 10  city                  43322 non-null  str    
 11  funding_rounds        49438 non-null  float64
 12  founded_at            38554 non-null  str    
 13  founded_month         38482 non-null  str    
 14  founded_quarter       38482 non-nul

#### Cleaning

In [11]:
# Deleting empty spaces in column names

df.columns = df.columns.str.strip()

# Transforming numerical values to int/long

numeric_columns = ['funding_total_usd', 'funding_rounds','seed','venture','equity_crowdfunding','undisclosed','convertible_note','debt_financing',
                   'angel','grant','private_equity','post_ipo_equity','post_ipo_debt','secondary_market','product_crowdfunding','round_A','round_B','round_C','round_D','round_E','round_F','round_G','round_H']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Unify nan in a coherent format

df.replace(0, np.nan, inplace=True)
df.replace('-', np.nan, inplace=True)
df.replace('n/a', np.nan, inplace=True)
df.replace('N/A', np.nan, inplace=True)
df.replace('NA', np.nan, inplace=True)
df.replace('na', np.nan, inplace=True)
df.replace('NaN', np.nan, inplace=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54294 entries, 0 to 54293
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             49438 non-null  str    
 1   name                  49437 non-null  str    
 2   homepage_url          45989 non-null  str    
 3   category_list         45477 non-null  str    
 4   market                45470 non-null  str    
 5   funding_total_usd     42 non-null     float64
 6   status                48124 non-null  str    
 7   country_code          44165 non-null  str    
 8   state_code            30161 non-null  str    
 9   region                44165 non-null  str    
 10  city                  43322 non-null  str    
 11  funding_rounds        49438 non-null  float64
 12  founded_at            38554 non-null  str    
 13  founded_month         38482 non-null  str    
 14  founded_quarter       38482 non-null  str    
 15  founded_year          38482 no

We notice that the column "funding_total_usd" doesn't contain relevant information for the analysis we aim to make, only 42 lines present information over a total 54k.

We therefore try to create an alternative metrics by using the detailled financing features.

We also compare this newly created measure with the lines where funding_total_usd is indeed present to check for coeherence and confirm quality of the indicator.

In [12]:
# Creating an alternative indicator by sum the detailled financing features

df['funding_calc'] = df[['seed','venture','equity_crowdfunding','undisclosed','convertible_note','debt_financing',
                   'angel','grant','private_equity','post_ipo_equity','post_ipo_debt','secondary_market','product_crowdfunding',
                   'round_A','round_B','round_C','round_D','round_E','round_F','round_G','round_H']].sum(axis=1)

df['funding_calc'].describe()

count    5.429400e+04
mean     1.678213e+07
std      1.535549e+08
min      0.000000e+00
25%      3.500000e+03
50%      7.268220e+05
75%      7.900600e+06
max      3.007950e+10
Name: funding_calc, dtype: float64

In [13]:
# Calculating variation between the new indicator and the information contained in the "total_funding_usd" variable

df_filtered = df[df['funding_total_usd'].notna() & df['funding_calc'].notna()]
df_filtered['var'] = (df_filtered['funding_total_usd'] - df_filtered['funding_calc'])

df_filtered[['funding_total_usd','funding_calc','var']].describe()

,funding_total_usd,funding_calc,var
count,42.000000,42.000000,42.000000
mean,329.309524,336.238095,-6.928571
std,286.158023,288.719275,44.902275
min,1.000000,1.000000,-291.000000
25%,100.000000,100.000000,0.000000
50%,253.000000,257.500000,0.000000
75%,500.000000,500.000000,0.000000
max,929.000000,929.000000,0.000000


We notice that the avg. difference is minimal, it seems that the newly created indicator is valid and we therefore decide to keep it for subsequent analysis.

In [14]:
# Filtering the dataset for entries with presence of financing information

df['funding_calc'] = df['funding_calc'].replace(0, np.nan)

df_funding = df[df['funding_calc'].notna()]
df_funding.info()

<class 'pandas.DataFrame'>
Index: 40907 entries, 0 to 49437
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             40907 non-null  str    
 1   name                  40906 non-null  str    
 2   homepage_url          38593 non-null  str    
 3   category_list         38404 non-null  str    
 4   market                38399 non-null  str    
 5   funding_total_usd     42 non-null     float64
 6   status                39802 non-null  str    
 7   country_code          37088 non-null  str    
 8   state_code            25619 non-null  str    
 9   region                37088 non-null  str    
 10  city                  36402 non-null  str    
 11  funding_rounds        40907 non-null  float64
 12  founded_at            32201 non-null  str    
 13  founded_month         32135 non-null  str    
 14  founded_quarter       32135 non-null  str    
 15  founded_year          32135 non-nul

In [15]:
# Dropping columns not relevant to the analysis

col_to_drop = ['permalink', 'homepage_url', 'category_list',
               'founded_month','founded_quarter','founded_year','funding_total_usd']
df_funding.drop(columns=col_to_drop, inplace=True)

In [16]:
# Reordering and renaming columns

df_col = df_funding.columns.to_list()

new_order = ['name','country_code','state_code','region','city','market','founded_at','funding_calc',
             'funding_rounds','first_funding_at','last_funding_at']
new_order = new_order + [col for col in df_col if col not in new_order]

df_funding = df_funding[new_order]

df_funding.rename(columns={'funding_calc': 'funding_total_usd'}, inplace=True)

In [17]:
# Dropping nan and and duplicate "name" entries

df_funding.dropna(subset=['name'], inplace=True)
df_funding.drop_duplicates(subset=['name'], inplace=True)

In [18]:
# Changing dates to date_time format

date_columns = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_columns:
    df_funding[col] = pd.to_datetime(df_funding[col], errors='coerce')

In [19]:
# Reset_index and final dataset structure

df_funding.reset_index(drop=True, inplace=True)
df_funding.info()

<class 'pandas.DataFrame'>
RangeIndex: 40845 entries, 0 to 40844
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   name                  40845 non-null  str           
 1   country_code          37033 non-null  str           
 2   state_code            25580 non-null  str           
 3   region                37033 non-null  str           
 4   city                  36347 non-null  str           
 5   market                38340 non-null  str           
 6   founded_at            32151 non-null  datetime64[us]
 7   funding_total_usd     40845 non-null  float64       
 8   funding_rounds        40845 non-null  float64       
 9   first_funding_at      40845 non-null  datetime64[us]
 10  last_funding_at       40845 non-null  datetime64[us]
 11  status                39740 non-null  str           
 12  seed                  13802 non-null  float64       
 13  venture               23252

#### Simplifications of `market` → `market_category`

We define 17 macro-categories + `Other` from the originally defined variable "market", with a keyword matching strategy.  
Rules are evaluated in order: the first match wins (more specific to less specific).

In [20]:
# Exploring distribution of "market" variable

vc = df_funding['market'].value_counts()

print(f"Total rows: {len(df_funding)} | Null markets: {df_funding['market'].isna().sum()}")
print(f"\nTop 40 markets:")
print(vc.head(40).to_string())
print(f"\nCumulative coverage:")

for n in [10, 15, 20, 30, 50]:
    print(f"  Top {n:>2}: {vc.head(n).sum() / len(df_funding) * 100:.1f}%")

Total rows: 40845 | Null markets: 2505

Top 40 markets:
market
Software                 4049
Biotechnology            3519
Mobile                   1707
E-Commerce               1459
Curated Web              1335
Enterprise Software      1137
Health Care              1114
Clean Technology         1067
Hardware + Software       992
Games                     957
Advertising               916
Health and Wellness       812
Social Media              741
Education                 711
Finance                   705
Manufacturing             558
Analytics                 540
Security                  476
Semiconductors            472
Web Hosting               394
Consulting                316
Hospitality               308
Travel                    284
Fashion                   277
News                      276
Messaging                 255
Real Estate               246
Music                     239
Search                    234
Technology                224
SaaS                      213
Interne

In [21]:
# Taxonomy of the 17 categories identified (More specific to less specific).

market_taxonomy = [
    ('Biotechnology',            ['biotech', 'bioinformatics', 'life science', 'genomics',
                                  'biomedical', 'biochem', 'biopharmaceutical']),
    ('Healthcare',               ['health care', 'healthcare', 'health and wellness',
                                  'medical device', 'medical', 'wellness', 'pharmaceutical',
                                  'pharma', 'hospital', 'clinical', 'therapy', 'therapeutics',
                                  'diagnostics', 'dental', 'optical', 'nursing',
                                  ' health ', 'fitness', 'mental health', 'telemedicine',                                  'femtech', 'medtech', 'ehealth', 'e-health',
                                  'cannabis', 'senior care', 'elder care', 'home care']),
    ('Clean Tech & Energy',      ['clean tech', 'clean technology', 'renewable energy',
                                  'renewable', 'solar', 'wind', 'greentech', 'green tech',
                                  'energy efficiency', 'smart grid', 'energy storage',
                                  'biofuel', 'hydroelectric', 'geothermal', 'environmental',
                                  'sustainability', 'circular economy', 'cleantech',
                                  'climatetech', 'climate tech', 'carbon',
                                  'electric vehicle', 'ev ', 'recycling', 'waste',
                                  'water tech', 'watertech']),
    ('Energy',                   ['energy', 'oil and gas', 'oil & gas', 'petroleum',
                                  'mining', 'utilities', 'power generation']),
    ('Finance & FinTech',        ['finance', 'fintech', 'financial', 'banking', 'insurance',
                                  'payments', 'accounting', 'investment', 'lending', 'credit',
                                  'trading', 'wealth management', 'cryptocurrency', 'blockchain',
                                  'mortgage', 'asset management', 'venture capital',
                                  'private equity', 'insurtech', 'crowdfunding',
                                  ' bitcoin', ' crypto', 'defi', 'nft', 'web3',
                                  'fundraising']),
    ('Analytics & AI',           ['analytics', 'big data', 'data science', 'machine learning',
                                  'artificial intelligence', 'business intelligence',
                                  'predictive', 'computer vision', 'deep learning',
                                  ' ai ', 'bigdata', ' data ', 'nlp', 'automation',
                                  'data analytics', 'generative', 'augmented reality',
                                  'virtual reality', 'mixed reality', 'extended reality',
                                  ' ar ', ' vr ', ' xr ']),
    ('Security',                 ['security', 'cybersecurity', 'cyber security', 'privacy',
                                  'fraud', 'compliance', 'identity management', 'surveillance',
                                  'regtech']),
    ('Education',                ['education', 'edtech', 'e-learning', 'elearning', 'learning',
                                  'training', 'academic', 'tutoring', 'skills']),
    ('Advertising & Marketing',  ['advertising', 'marketing', 'public relations',
                                  'digital marketing', 'branding', 'seo', 'crm',
                                  'lead generation', 'growth hacking', 'content marketing',
                                  'performance marketing', 'influencer', 'adtech',
                                  ' sales ', 'customer acquisition']),
    ('Social & Communication',   ['social media', 'social network', 'messaging',
                                  'communication', 'community', 'communities', 'collaboration',
                                  'chat', 'video conferencing', ' events', 'networking',
                                  'event management', 'social ']),
    ('Manufacturing & Logistics',['manufacturing', 'industrial', 'logistics', 'supply chain',
                                  'transportation', 'automotive', 'aerospace', 'shipping',
                                  'warehouse', 'packaging', 'agriculture', 'farming',
                                  'food processing', 'textile', 'mobility', 'smart city',
                                  'agritech', 'smart manufacturing', 'industry 4',
                                  'delivery', 'last mile', 'fleet', 'freight',
                                  'construction tech', ' booking ']),
    ('Entertainment & Media',    ['games', 'gaming', 'entertainment', 'music', 'video',
                                  'sports', 'media', 'publishing', 'news', 'photography',
                                  'film', 'television', 'broadcast', 'podcast', 'streaming',
                                  'esports', ' sport ', ' content ', ' art ', 'creative',
                                  'boating', 'outdoor', 'hobby']),
    ('E-Commerce & Retail',      ['e-commerce', 'ecommerce', 'retail', 'fashion', 'marketplace',
                                  'shopping', 'curated web', 'consumer goods', 'apparel',
                                  'luxury', 'beauty', 'cosmetics', 'd2c', 'direct to consumer',
                                  'furniture', ' rental', 'renting', 'subscription',
                                  'sharing economy', 'home decor', 'consumer ']),
    ('Travel & Hospitality',     ['travel', 'hospitality', 'tourism', 'hotel', 'restaurant',
                                  'food and beverage', 'food & beverage', 'catering', 'leisure',
                                  'airline', 'cruise', 'accommodation', 'foodtech', 'food tech',
                                  ' food ', ' coffee ']),
    ('Real Estate',              ['real estate', 'property', 'construction', 'architecture',
                                  'proptech', 'building material', 'smart building',
                                  'home ', 'interior']),
    ('Hardware & Semiconductors',['hardware', 'semiconductor', 'electronics', 'electrical',
                                  'iot', 'robotics', 'embedded', 'microprocessor', 'sensor',
                                  'wearable', 'drone', '3d printing', 'deep tech', 'deeptech',
                                  'nanotechnology', 'quantum', 'android', 'ios',
                                  'connected devices', 'edge computing']),
    ('Mobile',                   ['mobile', ' app ', ' apps ']),
    ('Software & SaaS',          ['software', 'saas', 'web hosting', 'internet',
                                  'cloud', 'platform', 'technology', 'tech',
                                  'information technology', 'open source', 'api', 'developer',
                                  'devops', 'digital', 'consulting', 'it services',
                                  'information services', 'b2b', 'b2c', 'hr tech',
                                  'human resources', ' hr ', 'enterprise', 'no-code', 'nocode',
                                  'low-code', ' computer ', 'software as',
                                  'business development', 'business services',
                                  ' design ', 'ux ', 'product management', 'project management',
                                  'workplace', 'productivity', 'workflow',
                                  'customer service', 'customer support', 'customer success',
                                  'legal services', 'legal ', 'legaltech', 'legal tech',
                                  'consumer services', ' it ', 'managed services',
                                  'professional services', 'outsourcing']),
]

def classify_market(market_val):
    if pd.isna(market_val):
        return pd.NA
    m = ' ' + market_val.strip().lower() + ' '  # padding para word-boundary matching
    for category, keywords in market_taxonomy:
        if any(kw in m for kw in keywords):
            return category
    return 'Other'

df_funding['market_category'] = df_funding['market'].apply(classify_market)

In [22]:
# Validating distribution of market_category

vc_cat = df_funding['market_category'].value_counts(dropna=False)

total = len(df_funding)

print(f"{'Categoría':<30} {'N':>7}  {'%':>6}")
print("-" * 50)
for cat, cnt in vc_cat.items():
    print(f"  {str(cat):<28} {cnt:>7}  ({cnt/total*100:>5.1f}%)")
print("-" * 50)
other_pct = (vc_cat.get('Other', 0) + vc_cat.get(pd.NA, 0)) / total * 100
print(f"  {'Other + NaN (total)':<28} {'':>7}  ({other_pct:>5.1f}%)")

Categoría                            N       %
--------------------------------------------------
  Software & SaaS                 7721  ( 18.9%)
  E-Commerce & Retail             3728  (  9.1%)
  Other                           3707  (  9.1%)
  Biotechnology                   3539  (  8.7%)
  Entertainment & Media           2824  (  6.9%)
  Healthcare                      2773  (  6.8%)
  nan                             2505  (  6.1%)
  Mobile                          2022  (  5.0%)
  Hardware & Semiconductors       1820  (  4.5%)
  Social & Communication          1790  (  4.4%)
  Advertising & Marketing         1570  (  3.8%)
  Finance & FinTech               1246  (  3.1%)
  Clean Tech & Energy             1199  (  2.9%)
  Analytics & AI                  1091  (  2.7%)
  Manufacturing & Logistics       1033  (  2.5%)
  Education                        801  (  2.0%)
  Security                         584  (  1.4%)
  Travel & Hospitality             448  (  1.1%)
  Real Estate       

In [23]:
# Re-ordering columns

col_order = ["name", "country_code", "state_code", "region", "city","market_category", "market", "founded_at", "funding_total_usd", "funding_rounds", "first_funding_at", "last_funding_at", "status"]
col_order = col_order + [x for x in df_funding.columns if x not in col_order]

df_funding = df_funding[col_order]
df_funding.head()

,name,country_code,state_code,region,city,market_category,market,founded_at,funding_total_usd,funding_rounds,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,#waywire,USA,NY,New York City,New York,Entertainment & Media,News,2012-06-01,1750000.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,&TV Communications,USA,CA,Los Angeles,Los Angeles,Entertainment & Media,Games,NaT,4000000.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,'Rock' Your Paper,EST,NaN,Tallinn,Tallinn,Entertainment & Media,Publishing,2012-10-26,40000.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(In)Touch Network,GBR,NaN,London,London,Hardware & Semiconductors,Electronics,2011-04-01,1500000.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-R- Ranch and Mine,USA,TX,Dallas,Fort Worth,Travel & Hospitality,Tourism,2014-01-01,60000.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Comparison and consolidation

##### Startup_global

We first export the dataset as it is to capture an overall picture of the information provided, indipendently of the geographical presence.

Main aim of this dataset is the creation of machine learning moodel.

In [24]:
# Exporting csv

startups_funding_global = df_funding.copy()
startups_funding_global.to_csv(export_path / "startups_funding_global.csv", index=False)

##### Startups IT_ES

We then extract a selection of the listed start-ups for Italy and Spain.

Aim of this selected dataset is mainly to compare the financing profile, which we'll integrate with geographical information to facilitate comparision and visualization:

- Country
- Region
- Year

In [25]:
# Filtering for IT and ES

df_es_it = startups_funding_global[startups_funding_global['country_code'].isin(['ESP','ITA'])]
df_describe(df_es_it)

DataFrame shape: (706, 34)

DataFrame info:
<class 'pandas.DataFrame'>
Index: 706 entries, 74 to 40831
Data columns (total 34 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   name                  706 non-null    str           
 1   country_code          706 non-null    str           
 2   state_code            0 non-null      str           
 3   region                706 non-null    str           
 4   city                  698 non-null    str           
 5   market_category       673 non-null    str           
 6   market                673 non-null    str           
 7   founded_at            569 non-null    datetime64[us]
 8   funding_total_usd     706 non-null    float64       
 9   funding_rounds        706 non-null    float64       
 10  first_funding_at      706 non-null    datetime64[us]
 11  last_funding_at       706 non-null    datetime64[us]
 12  status                684 non-null    str      

In [26]:
# Evaluating country distribution

print(f"ITA:{df_es_it[df_es_it['country_code'] == 'ITA'].shape[0]}")
print(f"ESP:{df_es_it[df_es_it['country_code'] == 'ESP'].shape[0]}")

ITA:244
ESP:462


In [27]:
# Mapping Italian and Spanish regions

# Evaluating information in 'region' column

df_es_it['region'].value_counts()

region
Barcelona                   178
Madrid                      150
Milan                        65
Rome                         36
ITA - Other                  36
                           ... 
Logrono                       1
Veneto                        1
Oviedo                        1
Cordoba                       1
San Casciano Val Di Pesa      1
Name: count, Length: 69, dtype: int64

In [28]:
# Evaluating information in column 'city'

df_es_it['city'].value_counts()

city
Barcelona                   178
Madrid                      142
Milan                        65
Valencia                     34
Rome                         25
                           ... 
Farra Di Soligo               1
Ferrara                       1
Córdoba                       1
San Casciano Val Di Pesa      1
Prato                         1
Name: count, Length: 130, dtype: int64

City seems to be containing an higer amount of information.

We'll proceed with the mapping by comparing to a standardise list of regions we'll use throughout the whole analysis.

In this way we aim to map entries where the region is not clearly specified, see "ITA - Other".

In [29]:
# Visualizing nan values in column "city"

df_es_it[['city','region']][df_es_it['city'].isna()]

,city,region
138,NaN,ESP - Other
185,NaN,Navarra
2065,NaN,Navarra
3971,NaN,ITA - Other
13542,NaN,Navarra
33529,NaN,ITA - Other
36565,NaN,Veneto
39882,NaN,ITA - Other


In [30]:
# Mapping of the missing city using the region's capital city

region_to_capital = {
    'Navarra':'Pamplona',
    'Cantabria':'Santander',
    'Veneto':'Venezia'
}

for region, capital in region_to_capital.items():
    df_es_it.loc[(df_es_it['region'] == region) & (df_es_it['city'].isna()), 'city'] = capital

# Dropping unspecified entries

df_es_it = df_es_it.dropna(subset=['city'])

In [31]:
# Creating mapping function of city -> region for Spain

def normalize(s):
    """Lowercase + strip accents."""
    s = str(s).lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

# Importing reference list file

df_poblaciones_es = pd.read_csv(files_dict['MUNICIPIOS.csv'], sep=';', encoding='latin1', decimal=',')
df_provincias_es  = pd.read_csv(files_dict['PROVINCIAS.csv'],  sep=';', encoding='latin1', decimal=',')

df_poblaciones_es = df_poblaciones_es[['COD_PROV', 'NOMBRE_ACTUAL']]
df_provincias_es  = df_provincias_es[['COD_PROV', 'COMUNIDAD_AUTONOMA']]

df_poblaciones_es = df_poblaciones_es.merge(df_provincias_es, on='COD_PROV', how='left')
df_poblaciones_es.columns = ['COD_PROV', 'City', 'Region']

# Creating dict of normalized regions

es_city_dict = {
    normalize(row['City']): row['Region']
    for _, row in df_poblaciones_es.iterrows()
}

# Adding provinces to match city -> region

df_provincias_es_full = pd.read_csv(files_dict['PROVINCIAS.csv'], sep=';', encoding='latin1', decimal=',')
for _, row in df_provincias_es_full.iterrows():
    key = normalize(row.get('NOMBRE_PROVINCIA', row.get('NOMBRE', '')))
    if key and key not in es_city_dict:
        es_city_dict[key] = row['COMUNIDAD_AUTONOMA']


# Creaing dict of typical spelling variatons (English)

manual_aliases_es = {
    'seville':                'Andalucía',
    'cordoba':                'Andalucía',
    'granada':                'Andalucía',
    'malaga':                 'Andalucía',
    'cadiz':                  'Andalucía',
    'almeria':                'Andalucía',
    'huelva':                 'Andalucía',
    'jerez':                  'Andalucía',
    'zaragoza':               'Aragón',
    'saragossa':              'Aragón',
    'oviedo':                 'Principado de Asturias',
    'gijon':                  'Principado de Asturias',
    'palma':                  'Illes Balears',
    'palma de mallorca':      'Illes Balears',
    'las palmas':             'Canarias',
    'santa cruz de tenerife': 'Canarias',
    'santander':              'Cantabria',
    'toledo':                 'Castilla-La Mancha',
    'albacete':               'Castilla-La Mancha',
    'valladolid':             'Castilla y León',
    'salamanca':              'Castilla y León',
    'burgos':                 'Castilla y León',
    'leon':                   'Castilla y León',
    'barcelona':              'Cataluña/Catalunya',
    'catalonia':              'Cataluña/Catalunya',
    'cataluna':               'Cataluña/Catalunya',
    'tarragona':              'Cataluña/Catalunya',
    'girona':                 'Cataluña/Catalunya',
    'lleida':                 'Cataluña/Catalunya',
    'merida':                 'Extremadura',
    'badajoz':                'Extremadura',
    'caceres':                'Extremadura',
    'a coruna':               'Galicia',
    'la coruna':              'Galicia',
    'coruna':                 'Galicia',
    'vigo':                   'Galicia',
    'santiago de compostela': 'Galicia',
    'logrono':                'La Rioja',
    'madrid':                 'Comunidad de Madrid',
    'alcala de henares':      'Comunidad de Madrid',
    'getafe':                 'Comunidad de Madrid',
    'murcia':                 'Región de Murcia',
    'cartagena':              'Región de Murcia',
    'pamplona':               'Comunidad Foral de Navarra',
    'navarra':                'Comunidad Foral de Navarra',
    'san sebastian':          'País Vasco/Euskadi',
    'donostia':               'País Vasco/Euskadi',
    'bilbao':                 'País Vasco/Euskadi',
    'vitoria':                'País Vasco/Euskadi',
    'vitoria gasteiz':        'País Vasco/Euskadi',
    'basque country':         'País Vasco/Euskadi',
    'valencia':               'Comunitat Valenciana',
    'alicante':               'Comunitat Valenciana',
    'castellon':              'Comunitat Valenciana',
    'ceuta':                  'Ceuta',
    'melilla':                'Melilla',
}

# Creating function

es_ref_keys = list(es_city_dict.keys())

FUZZY_THRESHOLD = 88

def map_city_to_region_es(city_raw):
    if pd.isna(city_raw):
        return np.nan

    city_norm = normalize(city_raw)

    # Layer 1: manual_dict
    if city_norm in manual_aliases_es:
        return manual_aliases_es[city_norm]
    # Layer 2: exact_match
    if city_norm in es_city_dict:
        return es_city_dict[city_norm]
    # Layer 3: fuzzy_match
    result = process.extractOne(city_norm, es_ref_keys, scorer=fuzz.WRatio, score_cutoff=FUZZY_THRESHOLD)
    if result:
        return es_city_dict[result[0]]

    return np.nan

df_es = df_es_it[df_es_it['country_code'] == 'ESP'].copy()
df_es['mapped_region'] = df_es['city'].apply(map_city_to_region_es)


# mapping summary

total   = len(df_es)
mapped  = df_es['mapped_region'].notna().sum()
missing = df_es['mapped_region'].isna().sum()
print(f"Mapped:  {mapped}/{total} ({mapped/total*100:.1f}%)")
print(f"Missing: {missing}/{total} ({missing/total*100:.1f}%)")
print("\nTop unmapped cities:")
print(df_es[df_es['mapped_region'].isna()]['city'].value_counts().head(20))


Mapped:  459/461 (99.6%)
Missing: 2/461 (0.4%)

Top unmapped cities:
city
Puerto De Andraitx    1
Miñano Menor          1
Name: count, dtype: int64


In [32]:
# Mapping city -> region for Italy

# Import reference file of Italian cities and regions

df_ciudades_it = pd.read_excel(files_dict['gi_comuni_cap.xlsx'])
df_ciudades_it.columns = df_ciudades_it.iloc[0]

df_ciudades_it = df_ciudades_it[1:]
df_ciudades_it = df_ciudades_it[['denominazione_ita', 'denominazione_regione', 'lat', 'lon']]
df_ciudades_it.columns = ['City', 'Region', 'Lat', 'Lon']

df_ciudades_it = df_ciudades_it.drop_duplicates(subset='City')

# Dict of normalized cities → Regions (from gi_comuni_cap)
it_city_dict = {
    normalize(row['City']): row['Region']
    for _, row in df_ciudades_it.iterrows()
}

# Dict of common variations (English, accents, etc.)
manual_aliases_it = {
    'milan':                    'Lombardia',
    'milano':                   'Lombardia',
    'rome':                     'Lazio',
    'roma':                     'Lazio',
    'naples':                   'Campania',
    'napoli':                   'Campania',
    'turin':                    'Piemonte',
    'torino':                   'Piemonte',
    'florence':                 'Toscana',
    'firenze':                  'Toscana',
    'bologna':                  'Emilia-Romagna',
    'venice':                   'Veneto',
    'venezia':                  'Veneto',
    'genoa':                    'Liguria',
    'genova':                   'Liguria',
    'palermo':                  'Sicilia',
    'catania':                  'Sicilia',
    'bari':                     'Puglia',
    'cagliari':                 'Sardegna',
    'sardinia':                 'Sardegna',
    'sicily':                   'Sicilia',
    'trieste':                  'Friuli-Venezia Giulia',
    'trento':                   'Trentino-Alto Adige/Südtirol',
    'bolzano':                  'Trentino-Alto Adige/Südtirol',
    'bozen':                    'Trentino-Alto Adige/Südtirol',
    'perugia':                  'Umbria',
    'ancona':                   'Marche',
    "l'aquila":                 'Abruzzo',
    'aquila':                   'Abruzzo',
    'potenza':                  'Basilicata',
    'catanzaro':                'Calabria',
    'reggio calabria':          'Calabria',
    'aosta':                    "Valle d'Aosta",
    'bergamo':                  'Lombardia',
    'brescia':                  'Lombardia',
    'monza':                    'Lombardia',
    'verona':                   'Veneto',
    'padova':                   'Veneto',
    'padua':                    'Veneto',
    'vicenza':                  'Veneto',
    'pisa':                     'Toscana',
    'siena':                    'Toscana',
    'modena':                   'Emilia-Romagna',
    'parma':                    'Emilia-Romagna',
    'reggio emilia':            'Emilia-Romagna',
    "reggio nell'emilia":       'Emilia-Romagna',
    'ferrara':                  'Emilia-Romagna',
    'ravenna':                  'Emilia-Romagna',
    'novara':                   'Piemonte',
    'como':                     'Lombardia',
    'varese':                   'Lombardia',
    'lecce':                    'Puglia',
    'brindisi':                 'Puglia',
    'taranto':                  'Puglia',
    'foggia':                   'Puglia',
    'cosenza':                  'Calabria',
    'salerno':                  'Campania',
    'caserta':                  'Campania',
}

# Mapping functions (Italy)
it_ref_keys = list(it_city_dict.keys())

def map_city_to_region_it(city_raw):
    if pd.isna(city_raw):
        return np.nan

    city_norm = normalize(city_raw)

    # Layer 1: manual alias
    if city_norm in manual_aliases_it:
        return manual_aliases_it[city_norm]
    # Layer 2: exact normalised match
    if city_norm in it_city_dict:
        return it_city_dict[city_norm]
    # Layer 3: fuzzy match
    result = process.extractOne(city_norm, it_ref_keys, scorer=fuzz.WRatio, score_cutoff=FUZZY_THRESHOLD)
    if result:
        return it_city_dict[result[0]]

    return np.nan

df_it = df_es_it[df_es_it['country_code'] == 'ITA'].copy()
df_it['mapped_region'] = df_it['city'].apply(map_city_to_region_it)

# Summary
total   = len(df_it)
mapped  = df_it['mapped_region'].notna().sum()
missing = df_it['mapped_region'].isna().sum()
print(f"Mapped:  {mapped}/{total} ({mapped/total*100:.1f}%)")
print(f"Missing: {missing}/{total} ({missing/total*100:.1f}%)")
print("\nTop unmapped cities:")
print(df_it[df_it['mapped_region'].isna()]['city'].value_counts().head(20))


Mapped:  240/241 (99.6%)
Missing: 1/241 (0.4%)

Top unmapped cities:
city
Navacchio    1
Name: count, dtype: int64


In [33]:
# Unify both dataframes and evaluate results

df_es_it_mapped = pd.concat([df_es, df_it], ignore_index=True)
df_es_it_mapped['mapped_region'].value_counts()

mapped_region
Cataluña/Catalunya              188
Comunidad de Madrid             156
Lombardia                        78
Comunitat Valenciana             46
Veneto                           36
Lazio                            35
Andalucía                        20
País Vasco/Euskadi               18
Emilia-Romagna                   16
Toscana                          15
Piemonte                         15
Campania                         12
Trentino-Alto Adige/Südtirol      7
Sicilia                           7
Galicia                           6
Sardegna                          5
Comunidad Foral de Navarra        4
Aragón                            4
Illes Balears                     4
Castilla y León                   3
Abruzzo                           3
Marche                            3
La Rioja                          2
Región de Murcia                  2
Castilla-La Mancha                2
Friuli-Venezia Giulia             2
Liguria                           2
Umbria        

In [34]:
# Map cities without manually assigned region

df_es_it_mapped[df_es_it_mapped['mapped_region'].isna()][['city','country_code']]

,city,country_code
282,Puerto De Andraitx,ESP
289,Miñano Menor,ESP
613,Navacchio,ITA


In [35]:
# Puerto de Andraitx: 'Illes Balears'

df_es_it_mapped.loc[df_es_it_mapped['city'] == 'Puerto de Andraitx', 'mapped_region'] = 'Illes Balears'

# Miñano Menor: 'País Vasco'

df_es_it_mapped.loc[df_es_it_mapped['city'] == 'Miñano Menor', 'mapped_region'] = 'País Vasco/Euskadi'

# Navacchio: 'Toscana'

df_es_it_mapped.loc[df_es_it_mapped['city'] == 'Navacchio', 'mapped_region'] = 'Toscana'

In [36]:
# Uniform country code with other datasets (ITA --> IT, ESP --> ES)

df_es_it_mapped['country_code'] = df_es_it_mapped['country_code'].replace({'ITA': 'IT', 'ESP': 'ES'})

# Remove, rename, and reorder columns

cols = df_es_it_mapped.columns.tolist()

cols.remove('region')
cols.remove('city')
cols.remove('state_code')
cols.remove('name')

col_order = ['country_code','mapped_region'] + [x for x in cols if x not in ['country_code','region','mapped_region']]
df_es_it_mapped = df_es_it_mapped[col_order]
df_es_it_mapped.rename(columns={'mapped_region': 'region'}, inplace=True)
df_es_it_mapped.rename(columns={'country_code': 'country'}, inplace=True)

Ahora empezamos la agrupación en un formato comparable a los de los dataset que utilizaremos más adelante.

Decidimos optar para la siguiente estrategía de agrupación:

- Aproximación de fechas considendo fecha de creación como fecha de financiación de la start-up
- Integramos fechas de creación que faltan con fechas de first_founding presentes para complementar cuanto más datos posibles
- Simplificamos rounds en una métrica de financiación única (Financing_rounds)
- Calculamos las siguientes métricas: avg. funding per start/up, avg. funding per round, avg. funding cycle + métricas de financiación presentes
- Convertimos a formato long para simplificar visualización y analísis

In [37]:
# Evaluate rows without creation dates

df_no_date = df_es_it_mapped[df_es_it_mapped['founded_at'].isna()]
len(df_no_date)

137

In [38]:
# Evaluate rows without creation dates but with first_funding date

df_no_fund_first = df_es_it_mapped[df_es_it_mapped['founded_at'].isna() & df_es_it_mapped['first_funding_at'].notna()]
len(df_no_fund_first)

137

In [39]:
# Fill rows without creation date with first_funding date, assuming creation date is earlier than first funding

df_es_it_mapped.loc[df_es_it_mapped['founded_at'].isna() & df_es_it_mapped['first_funding_at'].notna(), 'founded_at'] = df_es_it_mapped['first_funding_at'] 
df_no_date = df_es_it_mapped[df_es_it_mapped['founded_at'].isna()]
len(df_no_date)

0

In [40]:
# Calculate startup-level metric: funding cycle as the difference between last funding date and first funding date

df_es_it_mapped['funding_cycle_days'] = (df_es_it_mapped['last_funding_at'] - df_es_it_mapped['first_funding_at']).dt.days
df_es_it_mapped['funding_cycle_days'].describe()

count     702.000000
mean      208.770655
std       434.357055
min         0.000000
25%         0.000000
50%         0.000000
75%       275.500000
max      2905.000000
Name: funding_cycle_days, dtype: float64

In [41]:
# Remove columns not relevant for analysis

col_to_drop = ['first_funding_at', 'last_funding_at','market','funding_total_usd','post_ipo_equity','post_ipo_debt']
df_es_it_mapped = df_es_it_mapped.drop(columns=col_to_drop)

# Add ID column

df_es_it_mapped['ID'] = df_es_it_mapped.index + 1

# Reorder columns

df_es_it_mapped = df_es_it_mapped[['ID','country', 'region', 'market_category', 'founded_at','funding_rounds','funding_cycle_days','status', 'seed', 'venture','equity_crowdfunding','undisclosed','convertible_note','debt_financing','angel','grant','private_equity','secondary_market','product_crowdfunding','round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']]

In [42]:
# Convert to long format

df_long = pd.melt(df_es_it_mapped, id_vars=['ID','country', 'region', 'market_category', 'founded_at','funding_rounds','funding_cycle_days','status'], var_name='Financing_type', value_name='Value')
df_long.head()


,ID,country,region,market_category,founded_at,funding_rounds,funding_cycle_days,status,Financing_type,Value
0,1,ES,Comunidad Foral de Navarra,Biotechnology,2007-06-29,2.0,221,operating,seed,NaN
1,2,ES,Cataluña/Catalunya,Software & SaaS,2007-12-01,2.0,1100,operating,seed,800000.0
2,3,ES,Cataluña/Catalunya,Education,2007-01-01,1.0,0,operating,seed,NaN
3,4,ES,Cataluña/Catalunya,Mobile,2012-01-01,2.0,104,closed,seed,NaN
4,5,ES,Comunitat Valenciana,Advertising & Marketing,2009-08-01,2.0,365,operating,seed,NaN


In [43]:
df_long['Financing_type'].unique()

<ArrowStringArray>
[                'seed',              'venture',  'equity_crowdfunding',
          'undisclosed',     'convertible_note',       'debt_financing',
                'angel',                'grant',       'private_equity',
     'secondary_market', 'product_crowdfunding',              'round_A',
              'round_B',              'round_C',              'round_D',
              'round_E',              'round_F',              'round_G',
              'round_H']
Length: 19, dtype: str

In [44]:
# Add supercategory of financing for maturity funnel analysis

map_funnel = {
    'seed': 'Early Stage',
    'equity_crowdfunding': 'Early Stage',
    'angel': 'Early Stage',
    'round_A': 'Series A',
    'round_B': 'Series B',
    'round_C': 'Series C+',
    'round_D': 'Series C+',
    'round_E': 'Series C+',
    'round_F': 'Series C+',
    'round_G': 'Series C+',
    'round_H': 'Series C+',
    'private_equity': 'Late Stage',
    'secundary_market': 'Late Stage',
}

df_long['Funnel_stage'] = df_long['Financing_type'].map(map_funnel)

In [45]:
# Add supercategory of financing for volume analysis

map_volume = {
    'venture': 'Venture Capital',
    'round_A': 'Venture Capital',
    'round_B': 'Venture Capital',
    'round_C': 'Venture Capital',
    'round_D': 'Venture Capital',
    'round_E': 'Venture Capital',
    'round_F': 'Venture Capital',
    'round_G': 'Venture Capital',
    'round_H': 'Venture Capital',
    'seed': 'Early Stage Equity',
    'angel': 'Early Stage Equity',
    'equity_crowdfunding': 'Early Stage Equity',
    'debt_financing': 'Debt',
    'convertible_note': 'Debt',
    'grant':'Public Funding',
    'private_equity': 'Private Equity',
    'secondary_market': 'Private Equity',
    'product_crowdfunding': 'Other',
    'undisclosed': 'Other',
}

df_long['Volume_category'] = df_long['Financing_type'].map(map_volume)

In [46]:
# Export CSV

df_long.to_csv(export_path / "startups_funding_it_es.csv", index=False)

### Start-ups

This dataset is charachterised by a wider and more recent list of Italian and Spanish start-ups.

Whereas no information about financing is contained, we'll use this dataset to analyse trends in number of start-ups founded, annual growth, sectors and geographical coverage.

In [49]:
# List of files in this category

df = pd.read_excel(excel_path_bio, sheet_name="Raw", header=0)
df.query('Category == "Start-up"')['Name'].values


<ArrowStringArray>
['EU_Startups_Full.xlsx', 'EU_Startups_Financial_Institution.xlsx']
Length: 2, dtype: str

#### EU_Startups_Full.xlsx

In [50]:
# Importing excel

df = pd.read_excel(files_dict['EU_Startups_Full.xlsx'])

In [51]:
# Exploring dataset structure

df.head(20)

,Company Name,URL,Country,Based In,Tags,Founded
0,Save Your Travel,https://www.eu-startups.com/directory/save-you...,Italy,Rome,"Travel, Marketplace, Hotel, Train, Flight",2025
1,Gemini Security,https://www.eu-startups.com/directory/gemini-s...,Italy,"Bergamo, Italy","Software, Bot & Fraud Prevention, SaaS, Stop Bot",2023
2,JB Ecotex,https://www.eu-startups.com/directory/jb-ecotex/,Italy,Paris,Recycler,2012
3,Shopthelook,https://www.eu-startups.com/directory/shopthel...,Italy,Genoa,"vto, genAI, virtualtryon",2021
4,ClothesChanger,https://www.eu-startups.com/directory/clothesc...,Italy,Milan,AI clothes changer,2025
5,InvestinGoal,https://www.eu-startups.com/directory/investin...,Italy,Milano,"investing, trading, broker, compare, choose",2013
6,Proxima Security: Thynk Unlimited,https://www.eu-startups.com/directory/proxima-...,Italy,Napoli,"Cybersecurity, Artificial Intelligence, Deep-T...",2025
7,SiWeGO – Smart Eco Transport,https://www.eu-startups.com/directory/siwego-s...,Italy,Trento and Milan,"Smart Mobility, Logistics, Smart City",2021
8,suddo,https://www.eu-startups.com/directory/suddo/,Italy,Florence,"media,it,tech,news",2024
9,zornade,https://www.eu-startups.com/directory/zornade/,Italy,Trieste,"gis,saas,geospatial,data",2025


In [52]:
df.tail(10)

,Company Name,URL,Country,Based In,Tags,Founded
4357,Doist,https://www.eu-startups.com/directory/doist/,Spain,Barcelona,"Doist, Barcelona, HR Tech, remote work",2008
4358,busuu,https://www.eu-startups.com/directory/busuu/,Spain,Madrid,"Language Learning, Online Community, EdTech",2008
4359,ArtRatio,https://www.eu-startups.com/directory/artratio/,Spain,"Elche, Spain","smartglass, light-sensitive art, IoT",2008
4360,Dato Capital,https://www.eu-startups.com/directory/dato-cap...,Spain,Madrid,"Info Portal, Company Reports, AI",2007
4361,Cooltra,https://www.eu-startups.com/directory/cooltra/,Spain,Barcelona,"Consumer Services, Transportation, Rental",2006
4362,PuntoSeguro,https://www.eu-startups.com/directory/puntoseg...,Spain,Madrid,"life insurance, insurance, insurtech",2005
4363,BeTranslated,https://www.eu-startups.com/directory/betransl...,Spain,Valencia,"Translation, Translation Services, Localizatio...",2004
4364,Digital Samba,https://www.eu-startups.com/directory/digital-...,Spain,Barcelona,Video Conferencing Software,2004
4365,BriskBard,https://www.eu-startups.com/directory/briskbard/,Spain,Zaragoza,"Integration Tool, Web Developing Tools, Internet",2004
4366,La Novena Nube,https://www.eu-startups.com/directory/la-noven...,Spain,Valencia,"Home Decor, Textiles, Shopping, Sheets",2004


In [53]:
# Dropping rows with non relevant information

df.columns = df.columns.astype(str).str.strip()
df.columns = df.columns.astype(str).str.lower().str.replace(' ', '_')

df = df.drop(columns=['url'])

In [54]:
df_describe(df)

DataFrame shape: (4367, 5)

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 4367 entries, 0 to 4366
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   company_name  4367 non-null   str  
 1   country       4347 non-null   str  
 2   based_in      4365 non-null   str  
 3   tags          4348 non-null   str  
 4   founded       4367 non-null   int64
dtypes: int64(1), str(4)
memory usage: 457.8 KB
None

DataFrame description:
       company_name country   based_in  tags      founded
count          4367    4347       4365  4348  4367.000000
unique         4235       3        640  4103          NaN
top           Spain   Spain  Barcelona  SaaS          NaN
freq             16    2819       1002    20          NaN
mean            NaN     NaN        NaN   NaN  2018.983513
std             NaN     NaN        NaN   NaN     2.978039
min             NaN     NaN        NaN   NaN  2004.000000
25%             NaN     NaN      

In [55]:
# Analizying country column

# List unique values

df['country'].unique()

<ArrowStringArray>
['Italy', nan, 'Spain', 'UK']
Length: 4, dtype: str

In [56]:
# Analyzing companies in the UK

df[df['country'] == 'UK']

,company_name,country,based_in,tags,founded
3815,Spain,UK,London,"Care, Families, Vulnerable",2017


In [57]:
# Removing UK companies

df = df[df['country'] != 'UK']

In [58]:
# Analyzing NaN values in country column

df[df['country'].isna()]

,company_name,country,based_in,tags,founded
169,Italy,NaN,Terranova Da Sibari,Education,2017
174,Italy,NaN,Rimini,"AI, Big Data, IoT",2018
176,Italy,NaN,Fasano,AeroTech,2022
1439,Italy,NaN,Milan,"kitchen, hardware, goods",2015
1528,Italy,NaN,Milano,"Design, Manufacture, Distribution",2009
1758,Spain,NaN,Madrid,"Real Estate, Purpose",2023
1762,Spain,NaN,Madrid,Transportation,2022
1763,Spain,NaN,Madrid,Sharing Economy,2020
1780,Spain,NaN,Roquetas de Mar,"Health, IoT",2021
1782,Spain,NaN,Madrid,Health,2023


In [59]:
# We can infer the country of origin for all from the city

df_filtered = df[df['country'].isna()]
cities = df_filtered['based_in'].unique()

cities

<ArrowStringArray>
['Terranova Da Sibari',              'Rimini',              'Fasano',
               'Milan',              'Milano',              'Madrid',
     'Roquetas de Mar',            'Valencia',           'Barcelona',
               'Palma',               'Spain',            'Zaragoza']
Length: 12, dtype: str

In [60]:
# Mapping cities to countries

city_to_country = {
    'Terranova Da Sibari': 'IT',
    'Rimini': 'IT',
    'Fasano': 'IT',
    'Milan': 'IT',
    'Milano': 'IT',
    'Madrid': 'ES',
    'Roquetas de Mar':'ES',
    'Valencia':'ES',
    'Barcelona':'ES',
    'Palma':'ES',
    'Spain':'ES',
    'Zaragoza':'ES'
}

def map_city_to_country(city):
    if pd.isna(city):
        return np.nan
    city_norm = city.strip().lower()
    for key in city_to_country:
        if key.lower() == city_norm:
            return city_to_country[key]
    return np.nan

mask = df['country'].isna()
df.loc[mask, 'country'] = df.loc[mask, 'based_in'].apply(map_city_to_country)

# Renaming country (Italy --> IT, Spain --> ES)

df['country'] = df['country'].replace({'Italy': 'IT', 'Spain': 'ES'})
df['country'].unique()

<ArrowStringArray>
['IT', 'ES']
Length: 2, dtype: str

In [61]:
# Analyzing NaN values in based_in column

df[df['based_in'].isna()]

,company_name,country,based_in,tags,founded
2571,Coconut Jobs,ES,NaN,"Staffing, Recruiting, Social Networking",2021
2573,Big Together,ES,NaN,"provides optimization, get savings, high volum...",2021


In [62]:
# We search online to try to infer the missing information

# Coconut jobs seems to be based in Dubai, UAE

# We couldn't find relevant information about Big Together

# We remove these two records

df = df.dropna(subset=['based_in'])

In [63]:
# Analyzing NaN values in 'tags' column

df[df['tags'].isna()]

,company_name,country,based_in,tags,founded
679,Groupbmore,IT,Milan,NaN,2019
870,veeco srl,IT,Milan,NaN,2018
989,WeScriba,IT,Grottaglie,NaN,2017
994,Wexplore,IT,Milan,NaN,2017
1453,respectlife,IT,Pavia,NaN,2015
1518,Stamplay,IT,Rome,NaN,2012
1520,Vivocha,IT,Milan - Cagliari - San Francisco,NaN,2012
3087,Your VR Experience,ES,Cerdanyola,NaN,2019
3222,Wowzers.io,ES,Las Palmas,NaN,2019
3339,Propify,ES,Barcelona,NaN,2018


In [64]:
# We search online to try to infer the industry information

df['company_name'] = df['company_name'].str.strip()

map_company_to_tags = {
    'Groupbmore': 'Finance',
    'veeco srl':'Consulting',
    'WeScriba':'Saas',
    'Wexplore':	'Education',
    'respectlife':'Textile',
    'Stamplay':'Saas',
    'Vivocha':'Saas',
    'Your VR Experience':'Entertainment',
    'Wowzers.io':'Consulting',
    'Propify':'Housing',
    'Onyze':'Finance',
    'smartable IoT, SLU':'IoT',
    'Polaroo':'Energy',
    'Social&Care':'Healthcare',
    'PlusGuests':'Tourism',
    'TheTool':'Marketing',
    'Taiga.io':'SaaS',
    'TheySay.me':'SaaS',
    'viCloning':'SaaS'
}

def map_company_to_tags_func(company):
    if pd.isna(company):
        return np.nan
    company_norm = company.strip().lower()
    for key in map_company_to_tags:
        if key.lower() == company_norm:
            return map_company_to_tags[key]
    return np.nan

mask = df['tags'].isna()
df.loc[mask, 'tags'] = df.loc[mask, 'company_name'].apply(map_company_to_tags_func)

df[df['tags'].isna()]

,company_name,country,based_in,tags,founded


In [65]:
# Analyzing 'founded' column

df['founded'].unique()

array([2025, 2023, 2012, 2021, 2013, 2024, 2017, 2022, 2020, 2018, 2019,
       2015, 2016, 2009, 2014, 2004, 2011, 2010, 2007, 2008, 2006, 2005])

In [66]:
# Mapping region based on based_in, using the reference list for IT and ES

df_es_eu = df[df['country'] == 'ES'].copy()
df_es_eu['mapped_region'] = df_es_eu['based_in'].apply(map_city_to_region_es)

df_it_eu = df[df['country'] == 'IT'].copy()
df_it_eu['mapped_region'] = df_it_eu['based_in'].apply(map_city_to_region_it)

df = pd.concat([df_es_eu, df_it_eu], ignore_index=True)

# Mapping summary
total   = len(df)
mapped  = df['mapped_region'].notna().sum()
missing = df['mapped_region'].isna().sum()
print(f"Mapped:  {mapped}/{total} ({mapped/total*100:.1f}%)")
print(f"Missing: {missing}/{total} ({missing/total*100:.1f}%)")
print("\nTop unmapped cities:")
print(df[df['mapped_region'].isna()]['based_in'].value_counts().head(20))


Mapped:  4278/4364 (98.0%)
Missing: 86/4364 (2.0%)

Top unmapped cities:
based_in
Italy                 15
Galicia                6
London                 5
Lisbon                 3
Brixen                 3
Valence                2
Dublin                 2
Milan, Italy           2
Amsterdam              2
Ljubljana              1
Stockholm              1
Seoul                  1
Castilla-La Mancha     1
Corunna                1
ShowMB                 1
Buenos Aires           1
Corralejo              1
Alayor                 1
Vergara                1
Montroig               1
Name: count, dtype: int64


In [67]:
# We create a manual mapping to try to integrate as much information as possible and remove irrelevant companies

df['based_in'] = df['based_in'].str.strip()

manual_mapping = {
    'Galicia':'Galicia',
    'Milan, Italy':'Lombardia',
    'Castilla-La Mancha':'Castilla-La Mancha',
    'Corunna':'Galicia',
    'Corralejo':'Canarias',
    'Vergara':'País Vasco/Euskadi',
}

def map_based_in_to_region(based_in):
    if pd.isna(based_in):
        return np.nan
    based_in_norm = based_in.strip().lower()
    for key in manual_mapping:
        if key.lower() == based_in_norm:
            return manual_mapping[key]
    return np.nan

mask = df['mapped_region'].isna()
df.loc[mask, 'mapped_region'] = df.loc[mask, 'based_in'].apply(map_based_in_to_region)

# We remove rows without mapped region

df.dropna(subset=['mapped_region'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [68]:
# Now we try to unify the industry information with the previously analyzed funding database

df['tags'] = df['tags'].str.strip()
df['tags'] = df['tags'].str.lower()

ref_markets = startups_funding_global['market_category'].unique()
df_markets = df['tags'].unique()

ref_markets


<ArrowStringArray>
[    'Entertainment & Media', 'Hardware & Semiconductors',
      'Travel & Hospitality',           'Software & SaaS',
   'Advertising & Marketing',       'E-Commerce & Retail',
                'Healthcare',                 'Education',
                     'Other',            'Analytics & AI',
                    'Mobile', 'Manufacturing & Logistics',
             'Biotechnology',    'Social & Communication',
                         nan,         'Finance & FinTech',
                  'Security',       'Clean Tech & Energy',
               'Real Estate',                    'Energy']
Length: 20, dtype: str

In [69]:
# We extract only the first tag (primary tag) from the comma-separated list
df['tags_primary'] = df['tags'].str.split(',').str[0].str.strip()

In [70]:
# We apply the same market_category taxonomy directly on tags_primary
# (replaces the previous 3-layer fuzzy/TF-IDF pipeline for simplicity and consistency)

df['market_category'] = df['tags_primary'].apply(classify_market)

total   = len(df)
mapped  = df['market_category'].notna().sum()
other   = (df['market_category'] == 'Other').sum()
print(f"Assigned categories: {mapped}/{total} ({mapped/total*100:.1f}%)")
print(f"'Other':              {other}  ({other/total*100:.1f}%)")
print(f"Nulls (no tags):      {df['market_category'].isna().sum()}")
print(f"Unique values:        {df['market_category'].nunique()}")


Assigned categories: 4290/4290 (100.0%)
'Other':              760  (17.7%)
Nulls (no tags):      0
Unique values:        19


In [71]:
# Distribution of market_category in df
vc_df = df['market_category'].value_counts(dropna=False)
total = len(df)
print(f"{'Category':<30} {'N':>7}  {'%':>6}")
print("-" * 50)
for cat, cnt in vc_df.items():
    print(f"  {str(cat):<28} {cnt:>7}  ({cnt/total*100:>5.1f}%)")


Category                             N       %
--------------------------------------------------
  Other                            760  ( 17.7%)
  Software & SaaS                  643  ( 15.0%)
  Analytics & AI                   397  (  9.3%)
  Finance & FinTech                359  (  8.4%)
  E-Commerce & Retail              354  (  8.3%)
  Healthcare                       256  (  6.0%)
  Manufacturing & Logistics        228  (  5.3%)
  Travel & Hospitality             184  (  4.3%)
  Entertainment & Media            169  (  3.9%)
  Advertising & Marketing          158  (  3.7%)
  Mobile                           146  (  3.4%)
  Education                        115  (  2.7%)
  Clean Tech & Energy              110  (  2.6%)
  Real Estate                       97  (  2.3%)
  Social & Communication            79  (  1.8%)
  Hardware & Semiconductors         74  (  1.7%)
  Biotechnology                     73  (  1.7%)
  Security                          45  (  1.0%)
  Energy            

In [72]:
# Top tags that fall into 'Other' — useful for refining the taxonomy if needed
other_tags = df.loc[df['market_category'] == 'Other', 'tags_primary'].value_counts()
print(f"Distinct tags in 'Other': {len(other_tags)}")
other_tags.head(20)


Distinct tags in 'Other': 664


tags_primary
purpose               5
telecom               4
cms                   3
local                 3
cycling               3
cooking               3
aviation              3
animal feed           3
animation             3
pets                  3
battery               3
coworking             3
recruiting            3
kids                  3
career planning       3
password manager      2
revenue management    2
bank                  2
seguridad             2
batteries             2
Name: count, dtype: int64

In [73]:
# Remove duplicates

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# Remove columns without relevant information

cols_to_drop = ['company_name', 'tags', 'tags_primary','based_in']
df.drop(columns=cols_to_drop, inplace=True)

# Reorder and rename columns

col_order = ['country', 'mapped_region', 'market_category', 'founded']
df = df[col_order]

df.rename(columns={'mapped_region':'region'}, inplace=True)

# Transform 'founded' to date (year)

df['founded'] = pd.to_datetime(df['founded'], format='%Y')

In [74]:
# Export clean csv

df.to_csv(export_path / "startups_it_es.csv", index=False)

#### EU_Startups_Financial_Institutions.xlsx

In [75]:
# Import csv

df = pd.read_excel(files_dict['EU_Startups_Financial_Institution.xlsx'])

In [76]:
# Explore structure

df.head(20)

,Investor Name,Website URL,HQ Location,Investor Type,Investment Areas,Funding Stage
0,Patrimon Cube,https://www.eu-startups.com/investor/patrimon-...,Italy,VC Firm,"Fintech, SaaS, AI, HRtech, CRM",Pre-Seed / Seed
1,Novaterra,https://www.eu-startups.com/investor/novaterra/,Italy,VC Firm,"Health Tech, Life Sciences, SpaceTech, Energy ...",Pre-Seed / Seed
2,Generali Ventures,https://www.eu-startups.com/investor/generali-...,Italy,Corporate VC,Insurtech Fintech Proptech Mobility Cybersecur...,Series C & Beyond
3,Techshop,https://www.eu-startups.com/investor/techshop/,Italy,VC Firm,"B2B, tech, software",Pre-Seed / Seed
4,BlackSheep Fund,https://www.eu-startups.com/investor/blackshee...,Italy,VC Firm,"AI, Big Data, Automation, B2B, Marketing & Adv...",Series A/B
5,The Hive FVB Srl,https://www.eu-startups.com/investor/the-hive-...,Italy,Accelerator,"Developing innovative businesses, Promoting, S...",Pre-Seed / Seed
6,Eden Ventures,https://www.eu-startups.com/investor/eden-vent...,Italy,VC Firm,"cross sector, AI, industry agnostic",Pre-Seed / Seed
7,Moonstone,https://www.eu-startups.com/investor/moonstone/,Italy,VC Firm,"Healthtech, Energy, E-Commerce, Future of Work...",Pre-Seed / Seed
8,Neva SGR,https://www.eu-startups.com/investor/neva-sgr/,Italy,VC Firm,"InsurTech, RegTech, PropTech, FinTech, Aerospa...","Pre-Seed / Seed, Series A/B, Series C & Beyond"
9,NovaCapital,https://www.eu-startups.com/investor/novacapital/,Italy,VC Firm,"BioTech, GreenTech, Healthtech, FinTech, DeepT...","Pre-Seed / Seed, Series A/B, Series C & Beyond"


In [77]:
df.tail(10)

,Investor Name,Website URL,HQ Location,Investor Type,Investment Areas,Funding Stage
26,HWK Techinvestment,https://www.eu-startups.com/investor/hwk-techi...,Spain,VC Firm,Cybersecurity Software Artificial Intelligence...,Series A/B
27,Fierce Capital,https://www.eu-startups.com/investor/fierce-ca...,Spain,Corporate VC,"Venture Capital, Real Estate, Investment, Deve...","Pre-Seed / Seed, Series A/B, Series C & Beyond"
28,Capsa Vida,https://www.eu-startups.com/investor/capsa-vida/,Spain,Corporate VC,"Venture Capital, Investment Management, FoodTe...","Pre-Seed / Seed, Series A/B"
29,Enagas Emprende,https://www.eu-startups.com/investor/enagas-em...,Spain,Corporate VC,"Business Development, CleanTech, Energy, Renew...","Pre-Seed / Seed, Series A/B"
30,FI Nvest,https://www.eu-startups.com/investor/fi-nvest/,Spain,Corporate VC,"Finance, Financial Services, Funding Platform,...","Pre-Seed / Seed, Series A/B"
31,Capital Energy Quantum,https://www.eu-startups.com/investor/capital-e...,Spain,Corporate VC,"Renewable Energy, Independent Power Producer, ...","Pre-Seed / Seed, Series A/B, Series C & Beyond"
32,LLYC Venturing,https://www.eu-startups.com/investor/llyc-vent...,Spain,Corporate VC,"Funding Platform, Venture Capital, Public Rela...","Pre-Seed / Seed, Series A/B"
33,Adevinta Ventures,https://www.eu-startups.com/investor/adevinta-...,Spain,Corporate VC,"Venture Capital, Internet, Marketplace, Platfo...","Pre-Seed / Seed, Series A/B"
34,Cuatrecasas Ventures,https://www.eu-startups.com/investor/cuatrecas...,Spain,Corporate VC,"Business law, Legal, Lawyers, Law Practice","Pre-Seed / Seed, Series A/B, Series C & Beyond"
35,Cognitive Corporate Finance,https://www.eu-startups.com/investor/cognitive...,Spain,Corporate VC,"Finance, Financial Services, Health Care, Info...","Pre-Seed / Seed, Series A/B, Series C & Beyond"


The dataset contains a limited amount of financia entities for the 2 countries (36 in total).

We decide to exclude this specific source since the information is minimal and we have another dataset with a more general list of financial institutions.

### Finance

In [80]:
# Llamo archivos en esta categoría

df = pd.read_excel(excel_path_bio, sheet_name="Raw", header=0)
df.query('Category == "Finance"')['Name'].values

<ArrowStringArray>
[               'IT_Solidez_financiera.xlsx',
                'ES_Solidez_financiera.xlsx',
                     'IT_Tipos_interes.xlsx',
                     'ES_Tipos_interes.xlsx',
 'IT_Listado_instituciones_financieras.xlsx',
 'ES_Listado_instituciones_financieras.xlsx']
Length: 6, dtype: str

#### Financial soundness

##### IT_Solidez_financiera.xlsx

In [81]:
# Importing excel

df = pd.read_excel(files_dict['IT_Solidez_financiera.xlsx'])
df.head()

c:\Users\albet\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,SDDS_FS.Q.3700001.T1WA.902,Tier 1 su attività ponderate per il rischio,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,SDDS_FS.Q.3700001.T1TA.902,Tier 1 su totale attività,NaN,NaN,NaN,NaN,NaN,NaN
1,SDDS_FS.Q.3700001.NPLC.902,Rapporto tra prestiti deteriorati al netto del...,NaN,NaN,NaN,NaN,NaN,NaN
2,SDDS_FS.Q.3700001.NPLL.902,Rapporto tra prestiti deteriorati lordi e tota...,NaN,NaN,NaN,NaN,NaN,NaN
3,SDDS_FS.Q.3700001.ROA.902,Return on assets,NaN,NaN,NaN,NaN,NaN,NaN
4,SDDS_FS.Q.3700001.LIQSL.902,Attività liquide su passività a breve termine,NaN,NaN,NaN,NaN,NaN,NaN


In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 8 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   SDDS_FS.Q.3700001.T1WA.902                   67 non-null     str   
 1   Tier 1 su attività ponderate per il rischio  5 non-null      str   
 2   Unnamed: 2                                   53 non-null     object
 3   Unnamed: 3                                   53 non-null     object
 4   Unnamed: 4                                   53 non-null     object
 5   Unnamed: 5                                   53 non-null     object
 6   Unnamed: 6                                   53 non-null     object
 7   Unnamed: 7                                   53 non-null     object
dtypes: object(6), str(2)
memory usage: 5.3+ KB


In [83]:
# Exploring dataset structure

df.head(20)

,SDDS_FS.Q.3700001.T1WA.902,Tier 1 su attività ponderate per il rischio,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,SDDS_FS.Q.3700001.T1TA.902,Tier 1 su totale attività,NaN,NaN,NaN,NaN,NaN,NaN
1,SDDS_FS.Q.3700001.NPLC.902,Rapporto tra prestiti deteriorati al netto del...,NaN,NaN,NaN,NaN,NaN,NaN
2,SDDS_FS.Q.3700001.NPLL.902,Rapporto tra prestiti deteriorati lordi e tota...,NaN,NaN,NaN,NaN,NaN,NaN
3,SDDS_FS.Q.3700001.ROA.902,Return on assets,NaN,NaN,NaN,NaN,NaN,NaN
4,SDDS_FS.Q.3700001.LIQSL.902,Attività liquide su passività a breve termine,NaN,NaN,NaN,NaN,NaN,NaN
5,SERIE STORICA,NaN,Attività liquide su passività a breve termine,Rapporto tra prestiti deteriorati al netto del...,Rapporto tra prestiti deteriorati lordi e tota...,Return on assets,Tier 1 su totale attività,Tier 1 su attività ponderate per il rischio
6,Data dell'osservazione,NaN,Valore,Valore,Valore,Valore,Valore,Valore
7,30/09/2025,NaN,114.2,12.3,2.6,0.9,6.5,17.1
8,30/06/2025,NaN,117.4,12.2,2.6,0.7,6.8,17.4
9,31/03/2025,NaN,119.4,12.2,2.7,0.3,6.9,17.5


In [84]:
df.tail(10)

,SDDS_FS.Q.3700001.T1WA.902,Tier 1 su attività ponderate per il rischio,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
57,31/03/2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58,31/12/2012,NaN,76.2,75.7,13.4,-0.1,5.6,10.7
59,30/09/2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,30/06/2012,NaN,67.3,70.8,12.4,0.1,5.6,10.5
61,31/03/2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,31/12/2011,NaN,59.9,62.1,11.2,-0.8,5.5,9.6
63,30/09/2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,30/06/2011,NaN,73.6,58.2,10.5,0.2,5.5,9.5
65,31/03/2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,Prospetto estratto il 02-01-2026 01:16,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
# Consider row 4 as header

df.columns = df.iloc[5]

# Remove rows before header and empty rows at the end
df = df.iloc[7:-1]

# Remove empty columns

df = df.dropna(axis=1, how='all')
df.reset_index(drop=True, inplace=True)

# Modify column names

lista_columnas = df.columns.tolist()
lista_columnas[0] = "Date"

df.columns = [col.strip() for col in lista_columnas]

df.head()


,Date,Attività liquide su passività a breve termine,Rapporto tra prestiti deteriorati al netto delle rettifiche e totale dei mezzi propri,Rapporto tra prestiti deteriorati lordi e totale prestiti,Return on assets,Tier 1 su totale attività,Tier 1 su attività ponderate per il rischio
0,30/09/2025,114.2,12.3,2.6,0.9,6.5,17.1
1,30/06/2025,117.4,12.2,2.6,0.7,6.8,17.4
2,31/03/2025,119.4,12.2,2.7,0.3,6.9,17.5
3,31/12/2024,124,12.7,2.7,1.2,6.8,17.2
4,30/09/2024,118.1,13,2.8,0.9,6.7,17.5


In [86]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 7 columns):
 #   Column                                                                                 Non-Null Count  Dtype 
---  ------                                                                                 --------------  ----- 
 0   Date                                                                                   59 non-null     str   
 1   Attività liquide su passività a breve termine                                          51 non-null     object
 2   Rapporto tra prestiti deteriorati al netto delle rettifiche e totale dei mezzi propri  51 non-null     object
 3   Rapporto tra prestiti deteriorati lordi e totale prestiti                              51 non-null     object
 4   Return on assets                                                                       51 non-null     object
 5   Tier 1 su totale attività                                                              51 non-null 

In [87]:
# Modify data types

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

C:\Users\albet\AppData\Local\Temp\ipykernel_5520\2653673130.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


In [88]:
# Save column names for later analysis

df_it = df.copy()
col_IT = df.columns.tolist()

##### ES_Solidez_financiera.xlsx

In [89]:
# Importing excel

df = pd.read_excel(files_dict['ES_Solidez_financiera.xlsx'])
df.head()

,CÓDIGO DE LA SERIE,DSI.S122.FSKRC._Z._Z._Z.RAT.T,DSI.S122.FSKRTC._Z._Z._Z.RAT.T,DSI.S122.FSKNL._Z._Z._Z.RAT.T,DSI.S122.CET._Z._Z._Z.RAT.T,DSI.S122.KA._Z._Z._Z.RAT.T,DSI.S122.ANL._Z._Z._Z.PRT.T,DSI.S122.CLE._Z._Z._Z.PRT.T,DSI.S122.PN._Z._Z._Z.RAT.T,DSI.S122.ERA._Z._Z._Z.RAT.T,DSI.S122.ERE._Z._Z._Z.RAT.T,DSI.S122.EIM._Z._Z._Z.RAT.T,DSI.S122.ENE._Z._Z._Z.RAT.T,DSI.S122.LT._Z._Z._Z.PRT.T,DSI.S122.LS._Z._Z._Z.RAT.T,DSI.S122.LCR._Z._Z._Z.RAT.T,DSI.S122.NSF._Z._Z._Z.RAT.T
0,NÚMERO SECUENCIAL,4641896,4641897,4641898,4641899,4641900,4641901,4641902,4641903,4641904,4641905,4641906,4641907,4641908,4641909,4641910,4641911
1,ALIAS DE LA SERIE,SI_1_6A.1,SI_1_6A.2,SI_1_6A.3,SI_1_6A.4,SI_1_6A.5,SI_1_6A.6,SI_1_6A.7,SI_1_6A.8,SI_1_6A.9,SI_1_6A.10,SI_1_6A.11,SI_1_6A.12,SI_1_6A.13,SI_1_6A.14,SI_1_6A.15,SI_1_6A.16
2,DESCRIPCIÓN DE LA SERIE,Indicadores de solidez financiera. Capital reg...,Indicadores de solidez financiera. Capital de ...,Indicadores de solidez financiera. Préstamos d...,Indicadores de solidez financiera. Capital ord...,Indicadores de solidez financiera. Capital de ...,Indicadores de solidez financiera. Préstamos d...,Indicadores de solidez financiera. Indicador d...,Indicadores de solidez financiera. Provisiones...,Indicadores de solidez financiera. Rendimiento...,Indicadores de solidez financiera. Rendimiento...,Indicadores de solidez financiera. Margen de i...,Indicadores de solidez financiera. Gastos no f...,Indicadores de solidez financiera. Activos líq...,Indicadores de solidez financiera. Activos líq...,Indicadores de solidez financiera. Coeficiente...,Indicadores de solidez financiera. Coeficiente...
3,DESCRIPCIÓN DE LAS UNIDADES,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje
4,FRECUENCIA,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL


In [90]:
# Exploring structure

df.head(20)

,CÓDIGO DE LA SERIE,DSI.S122.FSKRC._Z._Z._Z.RAT.T,DSI.S122.FSKRTC._Z._Z._Z.RAT.T,DSI.S122.FSKNL._Z._Z._Z.RAT.T,DSI.S122.CET._Z._Z._Z.RAT.T,DSI.S122.KA._Z._Z._Z.RAT.T,DSI.S122.ANL._Z._Z._Z.PRT.T,DSI.S122.CLE._Z._Z._Z.PRT.T,DSI.S122.PN._Z._Z._Z.RAT.T,DSI.S122.ERA._Z._Z._Z.RAT.T,DSI.S122.ERE._Z._Z._Z.RAT.T,DSI.S122.EIM._Z._Z._Z.RAT.T,DSI.S122.ENE._Z._Z._Z.RAT.T,DSI.S122.LT._Z._Z._Z.PRT.T,DSI.S122.LS._Z._Z._Z.RAT.T,DSI.S122.LCR._Z._Z._Z.RAT.T,DSI.S122.NSF._Z._Z._Z.RAT.T
0,NÚMERO SECUENCIAL,4641896,4641897,4641898,4641899,4641900,4641901,4641902,4641903,4641904,4641905,4641906,4641907,4641908,4641909,4641910,4641911
1,ALIAS DE LA SERIE,SI_1_6A.1,SI_1_6A.2,SI_1_6A.3,SI_1_6A.4,SI_1_6A.5,SI_1_6A.6,SI_1_6A.7,SI_1_6A.8,SI_1_6A.9,SI_1_6A.10,SI_1_6A.11,SI_1_6A.12,SI_1_6A.13,SI_1_6A.14,SI_1_6A.15,SI_1_6A.16
2,DESCRIPCIÓN DE LA SERIE,Indicadores de solidez financiera. Capital reg...,Indicadores de solidez financiera. Capital de ...,Indicadores de solidez financiera. Préstamos d...,Indicadores de solidez financiera. Capital ord...,Indicadores de solidez financiera. Capital de ...,Indicadores de solidez financiera. Préstamos d...,Indicadores de solidez financiera. Indicador d...,Indicadores de solidez financiera. Provisiones...,Indicadores de solidez financiera. Rendimiento...,Indicadores de solidez financiera. Rendimiento...,Indicadores de solidez financiera. Margen de i...,Indicadores de solidez financiera. Gastos no f...,Indicadores de solidez financiera. Activos líq...,Indicadores de solidez financiera. Activos líq...,Indicadores de solidez financiera. Coeficiente...,Indicadores de solidez financiera. Coeficiente...
3,DESCRIPCIÓN DE LAS UNIDADES,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje
4,FRECUENCIA,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL,TRIMESTRAL
5,DIC 2005,11.99,8.12,-,-,-,-,-,-,-,-,-,-,-,-,-,-
6,MAR 2006,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
7,JUN 2006,12.02,7.61,-,-,-,-,-,-,-,19.79,-,-,-,-,-,-
8,SEP 2006,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
9,DIC 2006,11.91,7.56,-,-,-,-,-,-,-,-,-,-,-,-,-,-


In [91]:
df.tail()

,CÓDIGO DE LA SERIE,DSI.S122.FSKRC._Z._Z._Z.RAT.T,DSI.S122.FSKRTC._Z._Z._Z.RAT.T,DSI.S122.FSKNL._Z._Z._Z.RAT.T,DSI.S122.CET._Z._Z._Z.RAT.T,DSI.S122.KA._Z._Z._Z.RAT.T,DSI.S122.ANL._Z._Z._Z.PRT.T,DSI.S122.CLE._Z._Z._Z.PRT.T,DSI.S122.PN._Z._Z._Z.RAT.T,DSI.S122.ERA._Z._Z._Z.RAT.T,DSI.S122.ERE._Z._Z._Z.RAT.T,DSI.S122.EIM._Z._Z._Z.RAT.T,DSI.S122.ENE._Z._Z._Z.RAT.T,DSI.S122.LT._Z._Z._Z.PRT.T,DSI.S122.LS._Z._Z._Z.RAT.T,DSI.S122.LCR._Z._Z._Z.RAT.T,DSI.S122.NSF._Z._Z._Z.RAT.T
81,DIC 2024,17.52,15.05,15.2,13.55,5.75,2.87,43.69,44.41,1.3,13.96,63.67,49.5,18.68,25.11,180.02,134.91
82,MAR 2025,17.78,15.23,14.8,13.76,5.76,2.82,43.65,44.66,1.4,14.27,62.3,48.32,18.2,24.3,172.38,135.57
83,JUN 2025,17.83,15.34,13.85,13.83,5.78,2.69,43.35,45.72,1.36,14.29,61.32,47.62,18.03,24.01,176.68,135.29
84,FUENTE,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España,Banco de España
85,NOTAS,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


In [92]:
# Header row 2

df.columns = df.iloc[2]

# Remove rows before header and empty rows at the end

df = df.iloc[5:-2]
df = df.reset_index(drop=True)

# Rename date column

lista_columnas = df.columns.tolist()
lista_columnas[0] = "Quarter"

df.columns = [col.strip() for col in lista_columnas]


In [93]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 17 columns):
 #   Column                                                                                                                                                                                                Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                --------------  ----- 
 0   Quarter                                                                                                                                                                                               79 non-null     str   
 1   Indicadores de solidez financiera. Capital regulatorio sobre activos ponderados por riesgo. Sociedades de depósito, excepto banco central. Ratio                                                      79 non-null     object
 2  

In [94]:
# Modify date and numeric data

df['Quarter'] = df['Quarter'].str.replace('DIC', 'DEC')
df['Quarter'] = pd.to_datetime(df['Quarter'], format = '%b %Y',errors='coerce')

for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [95]:
# Simplify numeric column names

df.columns = df.columns.str.lower()

for col in df.columns[1:]:
    new_col_name = col.replace('indicadores de solidez financiera. ', '').strip()
    new_col_name = new_col_name.replace('sociedades de depósito, excepto banco central.', ' ').strip()
    new_col_name = new_col_name.replace('ratio', ' ').strip()
    new_col_name = new_col_name.replace('proporción sobre el total', ' ').strip()
    new_col_name = new_col_name.replace(' ', '_').strip()
    df.rename(columns={col: new_col_name}, inplace=True)

In [96]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 17 columns):
 #   Column                                                                                    Non-Null Count  Dtype         
---  ------                                                                                    --------------  -----         
 0   quarter                                                                                   79 non-null     datetime64[us]
 1   capital_regulatorio_sobre_activos_ponderados_por_riesgo.                                  61 non-null     float64       
 2   capital_de_nivel_1_sobre_activos_ponderados_por_riesgo.                                   61 non-null     float64       
 3   préstamos_dudosos_netos_de_provisiones_sobre_el_capital.                                  47 non-null     float64       
 4   capital_ordinario_de_nivel_1_sobre_activos_ponderados_por_riesgo.                         13 non-null     float64       
 5   capital_de_nivel_1_so

In [97]:
# Save df and column names for later comparison

df_es = df.copy()
col_es = df.columns.to_list()

##### Comparación y agrupación

In [98]:
# Visualize column names of both datasets

col_IT

['Date',
 'Attività liquide su passività a breve termine',
 'Rapporto tra prestiti deteriorati al netto delle rettifiche e totale dei mezzi propri',
 'Rapporto tra prestiti deteriorati lordi e totale prestiti',
 'Return on assets',
 'Tier 1 su totale attività',
 'Tier 1 su attività ponderate per il rischio']

In [99]:
col_es

['quarter',
 'capital_regulatorio_sobre_activos_ponderados_por_riesgo.',
 'capital_de_nivel_1_sobre_activos_ponderados_por_riesgo.',
 'préstamos_dudosos_netos_de_provisiones_sobre_el_capital.',
 'capital_ordinario_de_nivel_1_sobre_activos_ponderados_por_riesgo.',
 'capital_de_nivel_1_sobre_activos.',
 'préstamos_dudosos_sobre_el_total_de_préstamos_brutos.',
 'indicador_de_concentración_de_préstamos_en_las_tres_actividades_económicas_mayoritarias.',
 'provisiones_sobre_préstamos_dudosos.',
 'rendimiento_sobre_el_activo.',
 'rendimiento_sobre_el_capital.',
 'margen_de_intereses_sobre_ingresos_brutos.',
 'gastos_no_financieros_sobre_ingresos_brutos.',
 'activos_líquidos_sobre_activos.',
 'activos_líquidos_sobre_pasivos_a_corto_plazo.',
 'coeficiente_de_cobertura_de_liquidez.',
 'coeficiente_de_financiación_estable_neta.']

We maintain the column of the Italian dataset, since there's a lower amount of information.

Mapping the equivalent indexes:

- IT[1] = ES[14]
- IT[2] = ES[3]
- IT[3] = ES[6]
- IT[4] = ES[9]
- IT[5] = ES[5]
- IT[6] = ES[4]

In [100]:
# Remove columns from df_es that are not in df_it and reorder according to df_it

col_es_to_keep = [0,14,3,6,9,5,4]

for i in range(len(col_es)):
    if i not in col_es_to_keep:
        df_es = df_es.drop(columns=col_es[i])

cols_in_order = [col_es[i] for i in col_es_to_keep]
df_es = df_es[cols_in_order]

df_es.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 7 columns):
 #   Column                                                             Non-Null Count  Dtype         
---  ------                                                             --------------  -----         
 0   quarter                                                            79 non-null     datetime64[us]
 1   activos_líquidos_sobre_pasivos_a_corto_plazo.                      34 non-null     float64       
 2   préstamos_dudosos_netos_de_provisiones_sobre_el_capital.           47 non-null     float64       
 3   préstamos_dudosos_sobre_el_total_de_préstamos_brutos.              47 non-null     float64       
 4   rendimiento_sobre_el_activo.                                       47 non-null     float64       
 5   capital_de_nivel_1_sobre_activos.                                  47 non-null     float64       
 6   capital_ordinario_de_nivel_1_sobre_activos_ponderados_por_riesgo.  13 non-null  

In [101]:
# Comparing data range

df_es.head()

,quarter,activos_líquidos_sobre_pasivos_a_corto_plazo.,préstamos_dudosos_netos_de_provisiones_sobre_el_capital.,préstamos_dudosos_sobre_el_total_de_préstamos_brutos.,rendimiento_sobre_el_activo.,capital_de_nivel_1_sobre_activos.,capital_ordinario_de_nivel_1_sobre_activos_ponderados_por_riesgo.
0,2005-12-01,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-03-01,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-06-01,NaN,NaN,NaN,NaN,NaN,NaN
3,2006-09-01,NaN,NaN,NaN,NaN,NaN,NaN
4,2006-12-01,NaN,NaN,NaN,NaN,NaN,NaN


We observe various nan values, we separate those to have a more representative idea of the information in the dataset.

In [102]:
# Dropping nan

df_es.dropna(inplace=True)
df_es.head()

,quarter,activos_líquidos_sobre_pasivos_a_corto_plazo.,préstamos_dudosos_netos_de_provisiones_sobre_el_capital.,préstamos_dudosos_sobre_el_total_de_préstamos_brutos.,rendimiento_sobre_el_activo.,capital_de_nivel_1_sobre_activos.,capital_ordinario_de_nivel_1_sobre_activos_ponderados_por_riesgo.
66,2022-06-01,30.78,19.03,3.18,0.87,5.23,13.07
67,2022-09-01,29.85,18.58,3.11,0.89,5.22,13.02
68,2022-12-01,25.30,17.82,3.06,0.86,5.52,13.20
69,2023-03-01,24.56,17.45,3.02,1.00,5.59,13.29
70,2023-06-01,24.62,17.42,3.02,1.07,5.59,13.29


Analysing the italian df

In [103]:
# Dropping nan in df_it

df_it.dropna(inplace=True)
df_it.head()

,Date,Attività liquide su passività a breve termine,Rapporto tra prestiti deteriorati al netto delle rettifiche e totale dei mezzi propri,Rapporto tra prestiti deteriorati lordi e totale prestiti,Return on assets,Tier 1 su totale attività,Tier 1 su attività ponderate per il rischio
0,2025-09-30,114.2,12.3,2.6,0.9,6.5,17.1
1,2025-06-30,117.4,12.2,2.6,0.7,6.8,17.4
2,2025-03-31,119.4,12.2,2.7,0.3,6.9,17.5
3,2024-12-31,124.0,12.7,2.7,1.2,6.8,17.2
4,2024-09-30,118.1,13.0,2.8,0.9,6.7,17.5


The date range is uniform to df_es being quarterly assesments.

We observe anyway that df_it utilizes the last day of the month, whereas df_es the first.

We uniform this behaviour to guarantee a proper integration.

In [104]:
# Unify date format to month level for easier comparison

df_it['Date'] = df_it['Date'].dt.to_period('M').dt.to_timestamp()
df_it.head()

,Date,Attività liquide su passività a breve termine,Rapporto tra prestiti deteriorati al netto delle rettifiche e totale dei mezzi propri,Rapporto tra prestiti deteriorati lordi e totale prestiti,Return on assets,Tier 1 su totale attività,Tier 1 su attività ponderate per il rischio
0,2025-09-01,114.2,12.3,2.6,0.9,6.5,17.1
1,2025-06-01,117.4,12.2,2.6,0.7,6.8,17.4
2,2025-03-01,119.4,12.2,2.7,0.3,6.9,17.5
3,2024-12-01,124.0,12.7,2.7,1.2,6.8,17.2
4,2024-09-01,118.1,13.0,2.8,0.9,6.7,17.5


In [105]:
# Evaluate date range of both datasets

print(f"Date range for df_it:{df_it['Date'].describe()}")
print(f"Date range for df_es:{df_es['quarter'].describe()}")

Date range for df_it:count                            51
mean     2019-04-12 08:28:14.117647
min             2011-06-01 00:00:00
25%             2016-04-16 00:00:00
50%             2019-06-01 00:00:00
75%             2022-07-17 00:00:00
max             2025-09-01 00:00:00
Name: Date, dtype: object
Date range for df_es:count                            13
mean     2023-12-01 01:50:46.153846
min             2022-06-01 00:00:00
25%             2023-03-01 00:00:00
50%             2023-12-01 00:00:00
75%             2024-09-01 00:00:00
max             2025-06-01 00:00:00
Name: quarter, dtype: object


In [106]:
# Filter both datasets for the date range 2022-06-01 (Minimum ES) to 2025-06-01 (Maximum ES)

start_date = '2022-06-01'
end_date = '2025-06-01'

df_it_filtered = df_it[(df_it['Date'] >= start_date) & (df_it['Date'] <= end_date)]
df_es_filtered = df_es[(df_es['quarter'] >= start_date) & (df_es['quarter'] <= end_date)]

Datasets consolidation

In [107]:
# Marking each df with the corrisponding country code and reordering columns

col_order = [0,7,1,2,3,4,5,6]

df_it_filtered['Country'] = 'IT'

it_col = df_it_filtered.columns.to_list()
col_in_order = [it_col[i] for i in col_order]
df_it_filtered = df_it_filtered[col_in_order]

df_es_filtered['Country'] = 'ES'

es_col = df_es_filtered.columns.to_list()
col_in_order = [es_col[i] for i in col_order]
df_es_filtered = df_es_filtered[col_in_order]

In [108]:
# Unify column names for concatenation

df_it_filtered.columns = df_es_filtered.columns

In [109]:
# Combine the datasets

df_merged = pd.concat([df_it_filtered, df_es_filtered], ignore_index=True)
df_merged = pd.melt(df_merged, id_vars=['Country', 'quarter'], var_name='Indicator', value_name='Value')

df_merged['Index_category'] = 'Financial_soundness'
df_merged = df_merged[['quarter','Country','Index_category','Indicator','Value']]
df_merged.columns = ['Date', 'Country', 'Index_category', 'Index', 'Value']

df_merged.head()

,Date,Country,Index_category,Index,Value
0,2025-06-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,117.4
1,2025-03-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,119.4
2,2024-12-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,124.0
3,2024-09-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,118.1
4,2024-06-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,115.9


#### Interest rates

##### IT_tipos_interes

In [110]:
# Importo excel

df = pd.read_excel(files_dict['IT_Tipos_interes.xlsx'])

c:\Users\albet\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [111]:
# Exploring dataset structure

df.head(20)

,BAM_MIR.M.1300010.MIR5411.9.950.1000.SBI77.EUR.110.997,Tassi d'interesse armonizzati - prestiti non c/c - società non finanziarie - flussi,Unnamed: 2
0,SERIE STORICA,NaN,Tassi d'interesse armonizzati - prestiti non c...
1,Data dell'osservazione,NaN,Valore
2,2025-ott,NaN,3.5206
3,2025-set,NaN,3.3835
4,2025-ago,NaN,3.3882
5,2025-lug,NaN,3.4983
6,2025-giu,NaN,3.6047
7,2025-mag,NaN,3.66
8,2025-apr,NaN,3.7563
9,2025-mar,NaN,3.9191


In [112]:
df.tail(10)

,BAM_MIR.M.1300010.MIR5411.9.950.1000.SBI77.EUR.110.997,Tassi d'interesse armonizzati - prestiti non c/c - società non finanziarie - flussi,Unnamed: 2
363,1995-set,NaN,11.1836
364,1995-ago,NaN,11.2641
365,1995-lug,NaN,11.2117
366,1995-giu,NaN,11.0312
367,1995-mag,NaN,10.6196
368,1995-apr,NaN,10.9591
369,1995-mar,NaN,10.063
370,1995-feb,NaN,10.0676
371,1995-gen,NaN,10.0957
372,Prospetto estratto il 02-01-2026 00:54,NaN,NaN


In [113]:
# Dropping columns 2
df.drop(df.columns[1], axis=1, inplace=True)

# Dropping rows without information
df = df.iloc[2:]
df = df.iloc[:-1]
df.reset_index(drop=True, inplace=True)

# Modifying column names
df.columns = ['Date', 'Interest_rate']
df.head()

,Date,Interest_rate
0,2025-ott,3.5206
1,2025-set,3.3835
2,2025-ago,3.3882
3,2025-lug,3.4983
4,2025-giu,3.6047


In [114]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 370 entries, 0 to 369
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Date           370 non-null    str   
 1   Interest_rate  370 non-null    object
dtypes: object(1), str(1)
memory usage: 8.8+ KB


In [115]:
    # Converting date and numeric data (Italian month abbreviations -> numeric month)

month_map_it = {
    'gen': '01', 'feb': '02', 'mar': '03', 'apr': '04',
    'mag': '05', 'giu': '06', 'lug': '07', 'ago': '08',
    'set': '09', 'ott': '10', 'nov': '11', 'dic': '12'
}

if not pd.api.types.is_datetime64_any_dtype(df['Date']):
    date_clean = (
        df['Date']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace('.', '', regex=False)
    )

    # Convert formats like YYYY-mon (it) to YYYY-MM
    parts = date_clean.str.split('-', n=1, expand=True)
    date_norm = parts[0] + '-' + parts[1].map(month_map_it)

    df['Date'] = pd.to_datetime(date_norm, format='%Y-%m', errors='coerce')

# Handle possible decimal commas before converting to numeric
df['Interest_rate'] = pd.to_numeric(
    df['Interest_rate'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
)

df.head()

,Date,Interest_rate
0,2025-10-01,3.5206
1,2025-09-01,3.3835
2,2025-08-01,3.3882
3,2025-07-01,3.4983
4,2025-06-01,3.6047


In [116]:
# Save the cleaned dataframe for further analysis

df_tipos_it = df.copy()

##### ES_Tipos_interes

In [117]:
# Importing excel

df = pd.read_excel(files_dict['ES_Tipos_interes.xlsx'])

In [118]:
# Exploring dataset structure

df.head(20)

,CÓDIGO DE LA SERIE,DN_1TI2T0115,DN_1TI2T0119,DN_1TI2T0120,DN_1TI2T0121,DN_1TI2T0122,DN_1TI2T0123,DN_1TI2T0124,DN_1TI2T0125,DN_1TI2T0126,DN_1TI2T0127,DN_1TI2T0128,DN_1TI2T0129,DN_1TI2T0130,DN_1TI2T0131
0,NÚMERO SECUENCIAL,1843382,2806187,2806210,2806211,2806212,2806213,2806214,2806224,2806215,2806216,2806217,2806218,2806219,2806220
1,ALIAS DE LA SERIE,BE_19_5.1,BE_19_5.2,BE_19_5.3,BE_19_5.4,BE_19_5.5,BE_19_5.6,BE_19_5.7,BE_19_5.8,BE_19_5.9,BE_19_5.10,BE_19_5.11,BE_19_5.12,BE_19_5.13,BE_19_5.14
2,DESCRIPCIÓN DE LA SERIE,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....,Tipo de interés. Nuevas operaciones. EC y EFC....
3,DESCRIPCIÓN DE LAS UNIDADES,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje,Porcentaje
4,FRECUENCIA,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL,MENSUAL
5,ENE 2003,16.5984,-,-,-,-,-,-,-,-,-,-,-,-,-
6,FEB 2003,19.8096,-,-,-,-,-,-,-,-,-,-,-,-,-
7,MAR 2003,16.4381,-,-,-,-,-,-,-,-,-,-,-,-,-
8,ABR 2003,18.6185,-,-,-,-,-,-,-,-,-,-,-,-,-
9,MAY 2003,15.2351,-,-,-,-,-,-,-,-,-,-,-,-,-


In [119]:
df.tail(10)

,CÓDIGO DE LA SERIE,DN_1TI2T0115,DN_1TI2T0119,DN_1TI2T0120,DN_1TI2T0121,DN_1TI2T0122,DN_1TI2T0123,DN_1TI2T0124,DN_1TI2T0125,DN_1TI2T0126,DN_1TI2T0127,DN_1TI2T0128,DN_1TI2T0129,DN_1TI2T0130,DN_1TI2T0131
271,MAR 2025,3.866,17.2658,3.5861,3.5437,4.3612,4.1022,3.4078,3.4575,3.0248,3.1383,3.5085,3.6022,2.2527,3.5324
272,ABR 2025,3.7924,17.0097,3.5349,3.4965,4.5139,4.0628,3.3084,3.3399,3.0597,3.1302,3.4553,3.4805,3.1579,3.3261
273,MAY 2025,3.7417,17.2561,3.4027,3.3456,4.6962,4.1726,3.1575,3.162,3.1138,3.1345,3.3852,3.3556,2.9233,4.2966
274,JUN 2025,3.6828,17.2397,3.2677,3.2065,4.5934,4.1709,3.0391,3.03,3.0898,3.1216,3.1272,3.2641,2.1827,2.8045
275,JUL 2025,3.5395,16.9302,3.2446,3.1944,4.5363,4.0269,3.0772,3.0723,3.0955,3.1382,3.3397,3.35,3.1283,3.4563
276,AGO 2025,3.5194,17.3325,3.2609,3.2109,4.794,4.3141,3.1107,3.0988,3.3917,3.1247,3.262,3.2613,3.3241,3.1558
277,SEP 2025,3.5047,16.9218,3.2034,3.1397,4.7934,4.1555,3.0989,3.0919,3.2462,3.0969,3.2063,3.2053,3.4151,2.799
278,OCT 2025,3.5663,16.9655,3.284,3.218,4.7986,4.1681,3.1058,3.0969,3.2037,3.1701,3.1975,3.2305,3.2678,2.7217
279,FUENTE,-,-,-,-,-,-,-,-,-,-,-,-,-,-
280,NOTAS,-,-,-,-,-,-,-,-,-,-,-,-,-,-


In [120]:
# Heading row 2

df.columns = df.iloc[2]

# Remove rows without information
df = df.iloc[5:-2]
df.reset_index(drop=True, inplace=True)

df.head()

2,DESCRIPCIÓN DE LA SERIE,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Crédito. Descubiertos y líneas de crédito,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Tarjetas de crédito de pago aplazado,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos hasta 250 mil euros. Tipo medio ponderado,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos hasta 250 mil euros. Hasta 1 año,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos hasta 250 mil euros. A más de 1 año y hasta 5 años,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos hasta 250 mil euros. A más de 5 años,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos entre 250 mil y 1 millón de euros. Tipo medio ponderado,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos entre 250 mil y 1 millón de euros. Hasta 1 año,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos entre 250 mil y 1 millón euros. Más 1 año y hasta 5 años,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos entre 250 mil y 1 millón de euros. A más de 5 años,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos más de 1 millón de euros. Tipo medio ponderado,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos más de 1 millón de euros. Hasta 1 año,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos más de 1 millón de euros. A más de 1 año y hasta 5 años,Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos más de 1 millón de euros. A más de 5 años
0,ENE 2003,16.5984,-,-,-,-,-,-,-,-,-,-,-,-,-
1,FEB 2003,19.8096,-,-,-,-,-,-,-,-,-,-,-,-,-
2,MAR 2003,16.4381,-,-,-,-,-,-,-,-,-,-,-,-,-
3,ABR 2003,18.6185,-,-,-,-,-,-,-,-,-,-,-,-,-
4,MAY 2003,15.2351,-,-,-,-,-,-,-,-,-,-,-,-,-


In [121]:
# Select column for comparison with IT types

df = df[['DESCRIPCIÓN DE LA SERIE','Tipo de interés. Nuevas operaciones. EC y EFC. TEDR. SNF. Otros créditos entre 250 mil y 1 millón de euros. Tipo medio ponderado']]
df.columns = ['Date', 'Interest_rate']

df.head()

,Date,Interest_rate
0,ENE 2003,-
1,FEB 2003,-
2,MAR 2003,-
3,ABR 2003,-
4,MAY 2003,-


In [122]:
# Dropping nan

df.dropna(inplace=True)
df = df.query('Interest_rate != "-"')

# Convert date and numeric data (Spanish abbreviations -> numeric month)
month_map_es = {
    'ENE': '01', 'FEB': '02', 'MAR': '03', 'ABR': '04',
    'MAY': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08',
    'SEP': '09', 'OCT': '10', 'NOV': '11', 'DIC': '12'
}

if not pd.api.types.is_datetime64_any_dtype(df['Date']):
    date_clean = (
        df['Date']
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace('.', '', regex=False)
    )

    # Convert formats like YYYY-mon (es) to YYYY-MM
    parts = date_clean.str.split(' ', n=1, expand=True)
    date_norm = parts[1] + '-' + parts[0].map(month_map_es)

    df['Date'] = pd.to_datetime(date_norm, format='%Y-%m', errors='coerce')

# Handle possible decimal commas before converting to numeric

df['Interest_rate'] = pd.to_numeric(
    df['Interest_rate'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
)

df.head()

,Date,Interest_rate
89,2010-06-01,3.2382
90,2010-07-01,3.3126
91,2010-08-01,3.3418
92,2010-09-01,3.2952
93,2010-10-01,3.4816


In [123]:
# Save the cleaned dataframe for later analysis

df_tipos_es = df.copy()

##### Comparison and consolidation

In [124]:
# Evaluating data range for both datasets

print(f"Date range for df_tipos_it:{df_tipos_it['Date'].describe()}")
print(f"Date range for df_tipos_es:{df_tipos_es['Date'].describe()}")

Date range for df_tipos_it:count                           370
mean     2010-05-17 03:30:09.729729
min             1995-01-01 00:00:00
25%             2002-09-08 12:00:00
50%             2010-05-16 12:00:00
75%             2018-01-24 06:00:00
max             2025-10-01 00:00:00
Name: Date, dtype: object
Date range for df_tipos_es:count                           185
mean     2018-01-30 14:39:34.054054
min             2010-06-01 00:00:00
25%             2014-04-01 00:00:00
50%             2018-02-01 00:00:00
75%             2021-12-01 00:00:00
max             2025-10-01 00:00:00
Name: Date, dtype: object


In [125]:
# Filter IT dataset for the dates covered by the ES dataset

start_date = df_tipos_es['Date'].min()
df_tipos_it = df_tipos_it[df_tipos_it['Date'] >= start_date]

In [126]:
# Combining the datasets

df_tipos_it['Country'] = 'IT'
df_tipos_es['Country'] = 'ES'

df_tipos = pd.concat([df_tipos_it, df_tipos_es], ignore_index=True)
df_tipos = df_tipos[['Date', 'Country', 'Interest_rate']]

df_tipos['Index_category'] = 'Interest_rates'
df_tipos = df_tipos[['Date', 'Country', 'Index_category', 'Interest_rate']]
df_tipos.columns = ['Date', 'Country', 'Index_category', 'Weighted_average_interest_rate']

df_tipos.head()

,Date,Country,Index_category,Weighted_average_interest_rate
0,2025-10-01,IT,Interest_rates,3.5206
1,2025-09-01,IT,Interest_rates,3.3835
2,2025-08-01,IT,Interest_rates,3.3882
3,2025-07-01,IT,Interest_rates,3.4983
4,2025-06-01,IT,Interest_rates,3.6047


In [127]:
# Unifying with financial solidity data

df_tipos = pd.melt(df_tipos, id_vars=['Date', 'Country', 'Index_category'], var_name='Index', value_name='Value')
df_merged = pd.concat([df_merged, df_tipos], ignore_index=True)

df_merged.head()

,Date,Country,Index_category,Index,Value
0,2025-06-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,117.4
1,2025-03-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,119.4
2,2024-12-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,124.0
3,2024-09-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,118.1
4,2024-06-01,IT,Financial_soundness,activos_líquidos_sobre_pasivos_a_corto_plazo.,115.9


In [128]:
# Export final dataset for later analysis

df_merged.to_csv(export_path / "financial_indicators_it_es.csv", index=False)

#### Financial institution list

##### IT_Listado_instituciones_financieras

In [129]:
# Import excel

df = pd.read_excel(files_dict['IT_Listado_instituciones_financieras.xlsx'])

In [130]:
# Exploring dataset structure

df.head(10)

,CÓDIGO EUROPEO,LEI,NOMBRE,CATEGORÍA,DIRECCIÓN,INFORME,ENTIDAD MATRIZ
0,IT0001117976504,815600508DD477037C23,'BANCA CENTRO EMILIA - CREDITO COOPERATIVO' SO...,Entidad de crédito,"VIA STATALE, 39, 44042, CENTO",Sí,...
1,IT0000105626685,8156004D7FDDF887AA66,"'BANCA DI CREDITO POPOLARE', SOCIETA' COOPERAT...",Entidad de crédito,"CORSO V. EMANUELE, 92/100, 80059, TORRE DEL GRECO",Sí,...
2,IT0001627501111,NaN,AAREAL BANK AG,Entidad de crédito,"VIA SAVERIO MERCADANTE, 12/14, 00198, ROMA",Sí,DE03472 ...
3,IT0002367758548,NaN,ALLFUNDS BANK S.A.,Entidad de crédito,"VIA BOCCHETTO 6, 20123, MILANO",Sí,ES0011 ...
4,IT0000639755447,529900T32UL0CP1FZA06,ALLIANZ BANK FINANCIAL ADVISORS S.P.A.,Entidad de crédito,"PIAZZA TRE TORRI, 3, 20145, MILANO",Sí,...
5,IT0003512424662,549300FT58TC1XSOHW53,ANIMA LIQUIDITA' EURO,Fondo del mercado monetario,"CORSO GARIBALDI 99, 20121, MILANO",Sí,...
6,IT0005425460987,549300NIFGVIO1HDLR89,ANIMA TESORERIA,Fondo del mercado monetario,"CORSO GARIBALDI 99, 20121, MILANO",Sí,...
7,IT0004916351812,NaN,ARAB BANKING CORPORATION SA,Entidad de crédito,"VIA AMEDEI 8, 20123, MILANO",Sí,FR18979 ...
8,IT0003749450531,NaN,ATTIJARIWAFA BANK EUROPE SEDE SECONDARIA ITALIANA,Entidad de crédito,"VIA ANGELO POLIZIANO, 1, 20154, MILANO",Sí,FR23890 ...
9,IT0000100778167,81560076F9CF35569687,B.C.C. DEL GARDA - BANCA DI CREDITO COOPERATIV...,Entidad de crédito,"VIA TRIESTE, 62, 25018, MONTICHIARI",Sí,...


In [131]:
df.tail(10)

,CÓDIGO EUROPEO,LEI,NOMBRE,CATEGORÍA,DIRECCIÓN,INFORME,ENTIDAD MATRIZ
420,IT0003866026681,NaN,UNION BANCAIRE PRIVEE (EUROPE) S.A.,Entidad de crédito,"VIA BRERA,5, 20121, MILANO",Sí,LUB00038 ...
421,IT0005207681448,8156008C81E431B3E772,UNIPOLPAY S.P.A.,Otra institución,"VIA STALINGRADO 37, 40128, BOLOGNA",No,...
422,IT0000643845485,5299009VRTVEID33F522,VALPOLICELLA BENACO BANCA CREDITO COOPERATIVO ...,Entidad de crédito,"VIA ALCIDE DE GASPERI, 11, 37010, COSTERMANO S...",Sí,...
423,IT0002891614730,815600C6F7496F9B4023,VIVIBANCA S.P.A.,Entidad de crédito,"VIA GIOVANNI GIOLITTI 15, 10123, TORINO",Sí,...
424,IT0001232771625,NaN,VOLKSWAGEN BANK GMBH,Entidad de crédito,"VIA PRIVATA GROSIO 10/4, 20151, MILANO",Sí,DE03402 ...
425,IT0004215764223,NaN,WESTERN UNION INTERNATIONAL BANK GMBH,Entidad de crédito,"VIA BARBERINI, 68, 00187, ROMA",Sí,AT0000067035186 ...
426,IT0004162343594,8156007AE27B7429B185,WISE DIALOG BANK S.P.A.,Entidad de crédito,"VIA MESSINA, 38 - TORRE D, 20154, MILANO",Sí,...
427,IT0004440793059,81560058E659F533DB13,YOUNITED,Entidad de crédito,"VIA SARDEGNA 40, 00187, ROMA",Sí,FR16488 ...
428,IT0000425034251,815600810B937CD30406,ZKB ZADRU¿NA KRA¿KA BANKA TRST GORICA ZADRUGA ...,Entidad de crédito,"VIA DEL RICREATORIO, 2, 34151, TRIESTE",Sí,...
429,IT0005232560224,815600914B5015427B51,ZURICH ITALY BANK S.P.A.,Entidad de crédito,"VIA BENIGNO CRESPI, 23, 20159, MILANO",Sí,...


In [132]:
df_describe(df)

DataFrame shape: (430, 7)

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 430 entries, 0 to 429
Data columns (total 7 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [133]:
# Adding country column

df['Country'] = 'IT'

# Removing columns with irrelevant information

df.columns = df.columns.str.strip()
df = df[['CÓDIGO EUROPEO', 'NOMBRE', 'CATEGORÍA', 'DIRECCIÓN', 'Country']]

# Splitting DIRECCION into address, postal code, and city
split_cols = (
    df['DIRECCIÓN']
    .astype(str)
    .str.rsplit(', ', n=2, expand=True)
    .reindex(columns=range(3))
)
split_cols.columns = ['Address', 'ZipCode', 'City']
split_cols = split_cols.apply(lambda col: col.str.strip())

df[['Address', 'ZipCode', 'City']] = split_cols

# Reordering columns

df = df[['CÓDIGO EUROPEO', 'NOMBRE', 'CATEGORÍA', 'Address', 'ZipCode', 'City', 'Country']]
df.head()

,CÓDIGO EUROPEO,NOMBRE,CATEGORÍA,Address,ZipCode,City,Country
0,IT0001117976504,'BANCA CENTRO EMILIA - CREDITO COOPERATIVO' SO...,Entidad de crédito,"VIA STATALE, 39",44042,CENTO,IT
1,IT0000105626685,"'BANCA DI CREDITO POPOLARE', SOCIETA' COOPERAT...",Entidad de crédito,"CORSO V. EMANUELE, 92/100",80059,TORRE DEL GRECO,IT
2,IT0001627501111,AAREAL BANK AG,Entidad de crédito,"VIA SAVERIO MERCADANTE, 12/14",00198,ROMA,IT
3,IT0002367758548,ALLFUNDS BANK S.A.,Entidad de crédito,VIA BOCCHETTO 6,20123,MILANO,IT
4,IT0000639755447,ALLIANZ BANK FINANCIAL ADVISORS S.P.A.,Entidad de crédito,"PIAZZA TRE TORRI, 3",20145,MILANO,IT


In [134]:
# Integrating autonomous community information with "gi_comuni_cap" file

# Import file

df_ciudades_it = pd.read_excel(files_dict['gi_comuni_cap.xlsx'])
df_ciudades_it.head()

,Database Comuni Cap by gardainformatica.it,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,codice_istat,denominazione_ita_altra,denominazione_ita,denominazione_altra,cap,sigla_provincia,denominazione_provincia,tipologia_provincia,codice_regione,denominazione_regione,tipologia_regione,ripartizione_geografica,flag_capoluogo,codice_belfiore,lat,lon,superficie_kmq
1,001001,Agliè,Agliè,NaN,10011,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A074,45.363467,7.768606,13.2407
2,001002,Airasca,Airasca,NaN,10060,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A109,44.917006,7.484504,16.0327
3,001003,Ala di Stura,Ala di Stura,NaN,10070,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A117,45.314924,7.304367,46.1749
4,001004,Albiano d'Ivrea,Albiano d'Ivrea,NaN,10010,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A157,45.433646,7.949504,11.677


In [135]:
# Header row 1

df_ciudades_it.columns = df_ciudades_it.iloc[0]
df_ciudades_it = df_ciudades_it[1:]
df_ciudades_it.head()

,codice_istat,denominazione_ita_altra,denominazione_ita,denominazione_altra,cap,sigla_provincia,denominazione_provincia,tipologia_provincia,codice_regione,denominazione_regione,tipologia_regione,ripartizione_geografica,flag_capoluogo,codice_belfiore,lat,lon,superficie_kmq
1,001001,Agliè,Agliè,NaN,10011,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A074,45.363467,7.768606,13.2407
2,001002,Airasca,Airasca,NaN,10060,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A109,44.917006,7.484504,16.0327
3,001003,Ala di Stura,Ala di Stura,NaN,10070,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A117,45.314924,7.304367,46.1749
4,001004,Albiano d'Ivrea,Albiano d'Ivrea,NaN,10010,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A157,45.433646,7.949504,11.677
5,001006,Almese,Almese,NaN,10040,TO,Torino,Citta metropolitana,01,Piemonte,statuto ordinario,Nord-ovest,NO,A218,45.117665,7.395186,17.9879


In [136]:
# Adding region and lat/lon information to df based on postal code (cap)

df_ciudades_it = df_ciudades_it[['cap','denominazione_regione','lat','lon']]

df = df.merge(df_ciudades_it, left_on='ZipCode', right_on='cap', how='left')
df = df.drop(columns=['cap'])

# Renaming and reordering columns

df = df[['CÓDIGO EUROPEO','NOMBRE','CATEGORÍA','Address','City','denominazione_regione','Country','lat','lon']]
df.columns = ['European_code','Name','Category','Address','City','Region','Country','Lat','Lon']

df.head()


,European_code,Name,Category,Address,City,Region,Country,Lat,Lon
0,IT0001117976504,'BANCA CENTRO EMILIA - CREDITO COOPERATIVO' SO...,Entidad de crédito,"VIA STATALE, 39",CENTO,Emilia-Romagna,IT,44.727547,11.290563
1,IT0000105626685,"'BANCA DI CREDITO POPOLARE', SOCIETA' COOPERAT...",Entidad de crédito,"CORSO V. EMANUELE, 92/100",TORRE DEL GRECO,Campania,IT,40.786445,14.365858
2,IT0001627501111,AAREAL BANK AG,Entidad de crédito,"VIA SAVERIO MERCADANTE, 12/14",ROMA,Lazio,IT,41.892984,12.483536
3,IT0002367758548,ALLFUNDS BANK S.A.,Entidad de crédito,VIA BOCCHETTO 6,MILANO,Lombardia,IT,45.466647,9.190648
4,IT0000639755447,ALLIANZ BANK FINANCIAL ADVISORS S.P.A.,Entidad de crédito,"PIAZZA TRE TORRI, 3",MILANO,Lombardia,IT,45.466647,9.190648


In [137]:
df_inst_IT = df.copy()

##### ES_Listado_instituciones_financieras.xlsx

In [138]:
# Importing excel

df = pd.read_excel(files_dict['ES_Listado_instituciones_financieras.xlsx'])

In [139]:
# Exploring dataset structure

df.head(10)

,CÓDIGO EUROPEO,LEI,NOMBRE,CATEGORÍA,DIRECCIÓN,INFORME,ENTIDAD MATRIZ,CÓDIGO DE SUPERVISOR
0,ES0241,959800QKRKCC19MR2G52,"A&G BANCO, S.A.",Entidad de crédito,"Paseo de la Castellana, 92, 28046, Madrid",No,NaN,0241 ...
1,ES2080,54930056IRBXK0Q1FP96,"Abanca Corporacion Bancaria, S.A.",Entidad de crédito,"CI Cantón Claudino Pita, 2, 15300, Betanzos",Sí,NaN,2080 ...
2,ES8620,549300ORPMRMXB2JO334,"Abanca Servicios Financieros E.F.C., S.A.",Otra institución,"CL RÚA NUEVA, 30, 15004, A CORUÑA",No,NaN,8620 ...
3,ES1535,NaN,"AKF Bank GmbH & Co Kg, Sucursal en España",Entidad de crédito,"AV DE EUROPA, 12, 28108, ALCOBENDAS",No,DE03439,1535 ...
4,ES0011,95980020140005844330,"Allfunds Bank, S.A.",Entidad de crédito,"CL PADRES DOMINICOS, 7, 28109, MADRID",Sí,NaN,0011 ...
5,ES6730,NaN,ALPHA FX EUROPE LIMITED SUCURSAL EN ESPAÑA,Otra institución,"PS CASTELLANA, 53, 28046, MADRID",No,MTA213000,6730 ...
6,ES0200,95980020140005218292,"ANDBANK ESPAÑA BANCA PRIVADA, S.A.",Entidad de crédito,"PS DE LA CASTELLANA, 55, 28046, MADRID",Sí,NaN,0200 ...
7,ES0136,95980020140005658381,"Aresbank, S.A.",Entidad de crédito,"Ps de la Castellana, 257, 28046, Madrid",No,NaN,0136 ...
8,ES3183,959800AQXRU3780MW094,"Arquia Bank, S.A.",Entidad de crédito,"CL TUTOR, 16, 28008, MADRID",Sí,NaN,3183 ...
9,ES1541,9598002BMCMBU7G30F66,"Attijariwafa Bank Europe, Sucursal en España",Entidad de crédito,"Cl Bravo Murillo, 210, 28020, Madrid",No,FR23890,1541 ...


In [140]:
df.tail(10)

,CÓDIGO EUROPEO,LEI,NOMBRE,CATEGORÍA,DIRECCIÓN,INFORME,ENTIDAD MATRIZ,CÓDIGO DE SUPERVISOR
233,ES8769,959800RYLG2JBD5F8835,"Unión Financiera Asturiana, S.A., E.F.C.",Otra institución,"CL PELAYO, 15, 33003, OVIEDO",No,NaN,8769 ...
234,ES6719,959800NZ0JPD1HVKVW84,"Unnax Regulatory Services, EDE, SL",Otra institución,"PZ DE EUROPA, 22-24, 08902, L' HOSPITALET DE L...",No,NaN,6719 ...
235,ES6709,959800HUFA92RBVJQJ56,"Up Aganea EDE, SA",Otra institución,"AV. EUROPA, 16, 28108, ALCOBENDAS",No,NaN,6709 ...
236,ES8806,959800SRYS1VBV8YJ703,"VFS Financial Services Spain E.F.C., S.A.",Otra institución,"CL GOBELAS, 41 Y 45, 28023, MADRID",No,NaN,8806 ...
237,ES6731,NaN,VIVID MONEY SA SUCURSAL EN ESPAÑA,Otra institución,"RD SANT PERE, 52, 08010, BARCELONA",No,LUW00015,6731 ...
238,ES1480,NaN,"Volkswagen Bank GmbH, Sucursal en España",Entidad de crédito,"AV. DE BRUSELAS, 34, 28108, ALCOBENDAS",Sí,DE03402,1480 ...
239,ES1575,NaN,"Western Union International Bank GMBH, Sucursa...",Entidad de crédito,"BEATRIZ DE BOBADILLA, 14, 28040, MADRID",No,AT0000067035186,1575 ...
240,ES0229,549300Q17EOFMB1AGS79,"Wizink Bank, S.A.",Entidad de crédito,"Cl Ulises, 16-18, 28043, Madrid",Sí,NaN,0229 ...
241,ES8840,9598000WTZEQB8MJGF25,"Xfera Consumer Finance, E.F.C. S.A.",Otra institución,"PS DE LOS MELANCOLICOS, 14, 28005, MADRID",No,NaN,8840 ...
242,ES1560,NaN,"Younited, Sucursal en España",Entidad de crédito,"Carrer de la Caravel, 12, 08017, Barcelona",No,FR16488,1560 ...


In [141]:
df_describe(df)

DataFrame shape: (243, 8)

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 8 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

The dataset has the same exact structure of the prior one, we proceed with the same cleaning strategy.

In [142]:
# Adding country column

df['Country'] = 'ES'

# Eliminating columns without relevant information

df.columns = df.columns.str.strip()
df = df[['CÓDIGO EUROPEO', 'NOMBRE', 'CATEGORÍA', 'DIRECCIÓN', 'Country']]

# Splitting DIRECCION into address, postal code, and city
split_cols = (
    df['DIRECCIÓN']
    .astype(str)
    .str.rsplit(', ', n=2, expand=True)
    .reindex(columns=range(3))
)

split_cols.columns = ['Dirección', 'Código Postal', 'Ciudad']
df = df.drop(columns=['DIRECCIÓN']).join(split_cols)
df['Ciudad'] = df['Ciudad'].str.upper()

# Reordering columns

df = df[['CÓDIGO EUROPEO', 'NOMBRE', 'CATEGORÍA', 'Dirección', 'Ciudad', 'Código Postal', 'Country']]
df.columns = ['European_code','Name','Category','Address','City','zip_code','Country']
df.head()

,European_code,Name,Category,Address,City,zip_code,Country
0,ES0241,"A&G BANCO, S.A.",Entidad de crédito,"Paseo de la Castellana, 92",MADRID,28046,ES
1,ES2080,"Abanca Corporacion Bancaria, S.A.",Entidad de crédito,"CI Cantón Claudino Pita, 2",BETANZOS,15300,ES
2,ES8620,"Abanca Servicios Financieros E.F.C., S.A.",Otra institución,"CL RÚA NUEVA, 30",A CORUÑA,15004,ES
3,ES1535,"AKF Bank GmbH & Co Kg, Sucursal en España",Entidad de crédito,"AV DE EUROPA, 12",ALCOBENDAS,28108,ES
4,ES0011,"Allfunds Bank, S.A.",Entidad de crédito,"CL PADRES DOMINICOS, 7",MADRID,28109,ES


In [143]:
# Importing file of populations and provinces for geographical relevance analysis

df_poblaciones_es = pd.read_csv(files_dict['MUNICIPIOS.csv'],sep=';',encoding='latin1',decimal=',')
df_poblaciones_es.head()

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25_ETRS89,LONGITUD_ETRS89,LATITUD_ETRS89,ORIGENCOOR,ALTITUD,ORIGENALTITUD
0,1001000000,1010014,1010,1,Araba/Álava,Alegría-Dulantzi,2975,1994.5872,35069,1001000101,Alegría-Dulantzi,2860,0113-3,-2.512437,42.839812,Mapa,568.0,MDT
1,1002000000,1010029,1020,1,Araba/Álava,Amurrio,10313,9629.6800,65381,1002000201,Amurrio,9238,0086-4,-3.000073,43.054278,Mapa,219.0,MDT
2,1003000000,1010035,1030,1,Araba/Álava,Aramaio,1409,7308.9600,42097,1003000601,Ibarra,758,0087-4,-2.565400,43.051197,Mapa,333.0,MDT
3,1004000000,1010040,1040,1,Araba/Álava,Artziniega,1832,2728.7300,22886,1004000101,Artziniega,1697,0086-1,-3.127917,43.120844,Mapa,210.0,MDT
4,1006000000,1010066,1060,1,Araba/Álava,Armiñón,232,1297.2700,24707,1006000101,Armiñón,113,0137-4,-2.871835,42.723262,Mapa,467.0,MDT


In [144]:
df_provincias_es = pd.read_csv(files_dict['PROVINCIAS.csv'],sep=';',encoding='latin1',decimal=',')
df_provincias_es.head()

,COD_PROV,PROVINCIA,COD_CA,COMUNIDAD_AUTONOMA,CAPITAL
0,1,Araba/Álava,16,País Vasco/Euskadi,Vitoria-Gasteiz
1,2,Albacete,8,Castilla-La Mancha,Albacete
2,3,Alacant/Alicante,10,Comunitat Valenciana,Alacant/Alicante
3,4,Almería,1,Andalucía,Almería
4,5,Ávila,7,Castilla y León,Ávila


In [145]:
# Eliminating columns without relevant information

col_to_keep = ['COD_PROV','NOMBRE_ACTUAL','LONGITUD_ETRS89','LATITUD_ETRS89']
df_poblaciones_es = df_poblaciones_es[col_to_keep]

col_to_keep = ['COD_PROV','COMUNIDAD_AUTONOMA']
df_provincias_es = df_provincias_es[col_to_keep]

# Adding region and lat/lon information to df based on postal code

df_poblaciones_es['NOMBRE_ACTUAL'] = df_poblaciones_es['NOMBRE_ACTUAL'].str.upper()

df_poblaciones_es = df_poblaciones_es.merge(df_provincias_es, on='COD_PROV', how='left')
df_poblaciones_es = df_poblaciones_es.drop(columns=['COD_PROV'])

df_poblaciones_es.columns = ['City', 'Lon', 'Lat', 'Region']

df_poblaciones_es.head()

,City,Lon,Lat,Region
0,ALEGRÍA-DULANTZI,-2.512437,42.839812,País Vasco/Euskadi
1,AMURRIO,-3.000073,43.054278,País Vasco/Euskadi
2,ARAMAIO,-2.565400,43.051197,País Vasco/Euskadi
3,ARTZINIEGA,-3.127917,43.120844,País Vasco/Euskadi
4,ARMIÑÓN,-2.871835,42.723262,País Vasco/Euskadi


In [146]:
# Integrating regional information to the list of financial institution

df_merged_es = df.merge(df_poblaciones_es, on='City', how='left')

# Excluding columns without relevant information and formatting

df_merged_es = df_merged_es[['European_code','Name','Category','Address','City','Region','Country','Lat','Lon']]
df_merged_es['City'] = df_merged_es['City'].str.title()
df_merged_es.columns = ['European_code','Name','Category','Address','City','Region','Country','Lat','Lon']

df_merged_es.head()

,European_code,Name,Category,Address,City,Region,Country,Lat,Lon
0,ES0241,"A&G BANCO, S.A.",Entidad de crédito,"Paseo de la Castellana, 92",Madrid,Comunidad de Madrid,ES,40.408412,-3.687601
1,ES2080,"Abanca Corporacion Bancaria, S.A.",Entidad de crédito,"CI Cantón Claudino Pita, 2",Betanzos,Galicia,ES,43.279114,-8.210832
2,ES8620,"Abanca Servicios Financieros E.F.C., S.A.",Otra institución,"CL RÚA NUEVA, 30",A Coruña,Galicia,ES,43.371266,-8.395502
3,ES1535,"AKF Bank GmbH & Co Kg, Sucursal en España",Entidad de crédito,"AV DE EUROPA, 12",Alcobendas,Comunidad de Madrid,ES,40.541042,-3.632402
4,ES0011,"Allfunds Bank, S.A.",Entidad de crédito,"CL PADRES DOMINICOS, 7",Madrid,Comunidad de Madrid,ES,40.408412,-3.687601


In [147]:
# Visualizing missing region data

df_merged_es[df_merged_es['Region'].isna()]

,European_code,Name,Category,Address,City,Region,Country,Lat,Lon
18,ES0240,Banco de Crédito Social Cooperativo,Entidad de crédito,"CL CIUDAD FINANCIERA, 1",Almeria,NaN,ES,NaN,NaN
27,ES0186,"Banco Mediolanum, S.A.",Entidad de crédito,"CL BARCAS, 10",Valencia,NaN,ES,NaN,NaN
55,ES0038,"CACEIS Bank Spain, S.A.",Entidad de crédito,PQ.EMPRESARIAL LA FINCA-P.CLUB DEPORTIVO,Pozuelo De Alarcon,NaN,ES,NaN,NaN
61,ES3162,"Caixa Rural Benicarló, S. Coop. de Credit V.",Entidad de crédito,"AV JOAN CARLES I, 18",Benicarlo,NaN,ES,NaN,NaN
62,ES3117,"Caixa Rural D'Algemesí, S. Coop. V. de Crédit",Entidad de crédito,"CL SAN JOSÉ DE CALASANZ, 6",Algemesi,NaN,ES,NaN,NaN
63,ES3105,"Caixa Rural de Callosa d'en Sarrià, Cooperativ...",Entidad de crédito,"AV JAIME I, 1",Callosa D'En Sarria,NaN,ES,NaN,NaN
64,ES3096,"Caixa Rural de L'Alcudia, Sociedad Cooperativa...",Entidad de crédito,AV VERGE DE L'ORETO 2,L' Alcudia,NaN,ES,NaN,NaN
65,ES3123,"Caixa Rural de Turís, Cooperativa de Crédito V...",Entidad de crédito,"PZ CONSTITUCIÓN, 2",Turis,NaN,ES,NaN,NaN
67,ES3111,"Caixa Rural La Vall 'San Isidro', Sociedad Coo...",Entidad de crédito,"AV CORAZÓN DE JESÚS, 3",La Vall D'Uixo,NaN,ES,NaN,NaN
68,ES3166,"Caixa Rural Les Coves de Vinromá, S. Coop. de ...",Entidad de crédito,"CL SAN ANTONIO, 27",Les Coves De Vinroma,NaN,ES,NaN,NaN


In [148]:
# Manually filling missing regions

df_merged_es.iloc[18,5] = 'Andalucía'
df_merged_es.iloc[27,5] = 'Comunitat Valenciana'
df_merged_es.iloc[55,5] = 'Comunidad de Madrid'
df_merged_es.iloc[61,5] = 'Comunitat Valenciana'
df_merged_es.iloc[62,5] = 'Comunitat Valenciana'
df_merged_es.iloc[63,5] = 'Cataluña/Catalunya'
df_merged_es.iloc[64,5] = 'Comunitat Valenciana'
df_merged_es.iloc[65,5] = 'Comunitat Valenciana'
df_merged_es.iloc[67,5] = 'Comunitat Valenciana'
df_merged_es.iloc[68,5] = 'Comunitat Valenciana'
df_merged_es.iloc[70,5] = 'Comunitat Valenciana'
df_merged_es.iloc[73,5] = 'Comunitat Valenciana'
df_merged_es.iloc[76,5] = 'Comunitat Valenciana'
df_merged_es.iloc[79,5] = 'País Vasco/Euskadi'
df_merged_es.iloc[92,5] = 'Castilla-La Mancha' 
df_merged_es.iloc[95,5] = 'Principado de Asturias' 
df_merged_es.iloc[98,5] = 'Andalucía'
df_merged_es.iloc[111,5] = 'Región de Murcia'
df_merged_es.iloc[112,5] = 'Comunitat Valenciana' 
df_merged_es.iloc[113,5] = 'Comunitat Valenciana'
df_merged_es.iloc[114,5] = 'Comunitat Valenciana' 
df_merged_es.iloc[115,5] = 'Comunitat Valenciana'
df_merged_es.iloc[122,5] = 'Comunidad de Madrid'
df_merged_es.iloc[129,5] = 'Cataluña/Catalunya'
df_merged_es.iloc[137,5] = 'Andalucía'
df_merged_es.iloc[143,5] = 'Cataluña/Catalunya'
df_merged_es.iloc[160,5] = 'Cataluña/Catalunya'
df_merged_es.iloc[175,5] = 'País Vasco/Euskadi'
df_merged_es.iloc[202,5] = 'Comunitat Valenciana'
df_merged_es.iloc[203,5] = 'Cataluña/Catalunya'
df_merged_es.iloc[219,5] = 'Andalucía'
df_merged_es.iloc[225,5] = 'Comunidad de Madrid'
df_merged_es.iloc[235,5] = 'Cataluña/Catalunya'


In [149]:
# copying df for later merging with IT dataframe

df_inst_ES = df_merged_es.copy()

##### Comparison and consolidation

In [150]:
# Joining IT and ES datasets and exporting for later analysis

df_inst_merged = pd.concat([df_inst_IT, df_inst_ES], ignore_index=True)
df_inst_merged['Country'].value_counts()

Country
IT    1035
ES     244
Name: count, dtype: int64

In [151]:
# Exporting the combined dataset

df_inst_merged.to_csv(export_path / "financial_institutions_it_es.csv", index=False)

### Economics

In [154]:
# Loading files in this category

df = pd.read_excel(excel_path_bio, sheet_name="Raw", header=0)
df.query('Category == "Economics"')['Name'].values

<ArrowStringArray>
['PIB_Comunidades_autonomas.xlsx', 'I_D_Comunidades_autonomas.xlsx',
      'PIB_Regioni_Italiane.xlsx', 'PIB_Regioni_Italiane_total.csv',
      'I_D_Regioni_Italiane.xlsx']
Length: 5, dtype: str

#### GDP

##### PIB_Comunidades_autonomas.xlsx

In [155]:
# Importing excel

df = pd.read_excel(files_dict['PIB_Comunidades_autonomas.xlsx'])

In [156]:
# Exploring dataset structure

df.head(20)

,Series,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75
0,Serie 2000-2024 por comunidades y ciudades aut...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P.I.B. a precios de mercado y valor añadido br...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Unidades: miles de euros y porcentajes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,,Valor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,,2024(A),2023(P),2022.0,2021.0,2020.0,2019.0,2018.0,2017.0,2016.0,...,2009.0,2008.0,2007.0,2006.0,2005.0,2004.0,2003.0,2002.0,2001.0,2000.0
7,Andalucía,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"A. Agricultura, ganadería, silvicultura y ...",13498327,11791799,10264709.0,11156574.0,9678629.0,9344931.0,10325706.0,10896805.0,9670056.0,...,4.7,4.8,4.9,4.7,5.4,5.9,6.5,6.4,7.0,6.8
9,"B_E. Industrias extractivas, industria man...",22564579,21682646,22820010.0,18397711.0,15702698.0,17554177.0,17389075.0,17296189.0,16113913.0,...,10.1,11.2,11.3,11.7,12.2,12.2,12.5,12.7,12.8,13.0


In [157]:
df.tail(10)

,Series,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75
306,PRODUCTO INTERIOR BRUTO A PRECIOS DE MERCADO,1594330000,1497761000,1.375863e+09,1.235474e+09,1.129214e+09,1.253710e+09,1.212276e+09,1.170024e+09,1.122967e+09,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
307,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,Notas:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
310,(P) Estimación provisional.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
312,(A) Estimación avance.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
313,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314,Fuente:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
315,Instituto Nacional de Estadística,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We notice several issues in the file structure, the data seems to be in a sequencial order with subtotals for each region and GDP category.

In [158]:
# Creating list of regions from provincias file to map with series column in df

df_provincias_es = pd.read_csv(files_dict['PROVINCIAS.csv'], sep=';', encoding='latin1', decimal=',')
comunidades = df_provincias_es['COMUNIDAD_AUTONOMA'].dropna().astype(str).str.strip().unique().tolist()

# Marking each row with the corresponding autonomous community

def assign_region(df, comunidades):
    comunidades_set = {c.strip().casefold() for c in comunidades}
    series_norm = df['Series'].astype(str).str.strip().str.casefold()
    df['Region'] = series_norm.isin(comunidades_set).astype(int)
    return df

df['Series'] = df['Series'].astype(str).str.strip()
df = assign_region(df, comunidades)

df_filtered = df[['Series', 'Region']]
df_filtered.query('Region == 1')

,Series,Region
7,Andalucía,1
22,Aragón,1
67,Canarias,1
82,Cantabria,1
97,Castilla y León,1
142,Comunitat Valenciana,1
157,Extremadura,1
172,Galicia,1


In [159]:
# Not being able to mark all autonomous communities, we try to detect a pattern in the represented indicators

# We notice a difference between rows that suggests a regular block of 15
# We mark the first row of the block (start_idx) and then every 15 positions

start_idx = 7
step = 15
n_regions = len(comunidades)

df['Region'] = 0

pattern_positions = start_idx + step * np.arange(n_regions)
pattern_positions = pattern_positions[pattern_positions < len(df)]

df.iloc[pattern_positions, df.columns.get_loc('Region')] = 1

# Quick view of rows marked by pattern
df_filtered = df[['Series', 'Region']]
df_filtered.query('Region == 1').head(30)

,Series,Region
7,Andalucía,1
22,Aragón,1
37,"Asturias, Principado de",1
52,"Balears, Illes",1
67,Canarias,1
82,Cantabria,1
97,Castilla y León,1
112,Castilla - La Mancha,1
127,Cataluña,1
142,Comunitat Valenciana,1


In [160]:
# Saving list of regions in the PIB file to unify names of autonomous communities

Comunidades_PIB = (
    df.loc[df['Region'] == 1, 'Series']
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

def normalize_text(txt):
    txt = str(txt).strip().lower()
    txt = unicodedata.normalize('NFKD', txt)
    txt = ''.join(ch for ch in txt if not unicodedata.combining(ch))
    txt = txt.replace('/', ' ')
    txt = re.sub(r'[^a-z0-9\s]', ' ', txt)
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt


# Alias for known variants between both sources
alias_to_canonical = {
    'madrid comunidad de': 'comunidad de madrid',
    'murcia region de': 'region de murcia',
    'navarra comunidad foral de': 'comunidad foral de navarra',
    'rioja la': 'la rioja',
    'asturias principado de': 'principado de asturias',
    'balears illes': 'illes balears',
    'pais vasco euskadi': 'pais vasco',
    'cataluna catalunya': 'cataluna',
    'ciudad autonoma de ceuta': 'ceuta',
    'ciudad autonoma de melilla': 'melilla',
}


def canonical_key(txt):
    key = normalize_text(txt)
    return alias_to_canonical.get(key, key)


# Dataframes of reference and PIB with normalized key
comunidades_validas = pd.DataFrame({'Comunidad': sorted(set(comunidades))})
comunidades_validas['key'] = comunidades_validas['Comunidad'].map(canonical_key)

comunidades_pib = pd.DataFrame({'Comunidad_PIB': sorted(set(Comunidades_PIB))})
comunidades_pib['key'] = comunidades_pib['Comunidad_PIB'].map(canonical_key)

# Main correspondence
comunidades_merge = comunidades_validas.merge(
    comunidades_pib[['Comunidad_PIB', 'key']],
    on='key',
    how='left'
)

# Quality checks
aun_sin_match_ref = comunidades_merge[comunidades_merge['Comunidad_PIB'].isna()].copy()
pib_sin_match_ref = comunidades_pib[
    ~comunidades_pib['key'].isin(comunidades_validas['key'])
].copy()

# Fuzzy suggestions only if there are communities without match
if not aun_sin_match_ref.empty:
    pib_keys = comunidades_pib['key'].dropna().unique().tolist()

    def best_match(k):
        best = max(pib_keys, key=lambda cand: SequenceMatcher(None, k, cand).ratio())
        score = SequenceMatcher(None, k, best).ratio()
        return pd.Series({'key_sugerida': best, 'score_sugerencia': round(score, 3)})

    aun_sin_match_ref = aun_sin_match_ref.join(aun_sin_match_ref['key'].apply(best_match))
    key_to_pib = comunidades_pib.drop_duplicates('key').set_index('key')['Comunidad_PIB']
    aun_sin_match_ref['Comunidad_PIB_sugerida'] = aun_sin_match_ref['key_sugerida'].map(key_to_pib)

# Final dictionary PIB -> reference and standardized column in df
map_pib_to_ref = comunidades_merge.dropna(subset=['Comunidad_PIB']).set_index('Comunidad_PIB')['Comunidad'].to_dict()
df['Comunidad_match'] = df['Series'].map(map_pib_to_ref)

# Assign an ID to each row of autonomous community in the order they appear in the PIB df

dict_ID_comunidad = {comunidad: idx for idx, comunidad in enumerate(df.loc[df['Region'] == 1, 'Series'].tolist())}
df['Comunidad_ID'] = df['Series'].map(dict_ID_comunidad)
comunidades_merge['ID'] = comunidades_merge['Comunidad_PIB'].map(dict_ID_comunidad)+1

comunidades_merge

,Comunidad,key,Comunidad_PIB,ID
0,Andalucía,andalucia,Andalucía,1
1,Aragón,aragon,Aragón,2
2,Canarias,canarias,Canarias,5
3,Cantabria,cantabria,Cantabria,6
4,Castilla y León,castilla y leon,Castilla y León,7
5,Castilla-La Mancha,castilla la mancha,Castilla - La Mancha,8
6,Cataluña/Catalunya,cataluna,Cataluña,9
7,Ciudad Autónoma de Ceuta,ceuta,Ceuta,18
8,Ciudad Autónoma de Melilla,melilla,Melilla,19
9,Comunidad Foral de Navarra,comunidad foral de navarra,"Navarra, Comunidad Foral de",15


In [161]:
# Unifying names of autonomous communities in the PIB dataset

# Creating ID column for autonomous communities in PIB by summing the previous check

df['Comunidad_ID'] = df['Region'].astype(int).cumsum()

# Unifying names of autonomous communities in the PIB dataset using the created mapping
df = pd.merge(df,comunidades_merge[['Comunidad','ID']],left_on='Comunidad_ID',right_on='ID',how='left')

# Removing join columns

df.drop(['ID','Comunidad_match','Comunidad_ID'], axis=1, inplace=True)

In [162]:
# Now we act on the structure of the file, removing rows without relevant information and renaming columns

# Remove the first 5 rows and modify headers

df.columns = df.iloc[6]
df = df.iloc[7:].reset_index(drop=True)

# Rename and order columns

df.columns = df.columns.astype(str).str.strip()
df_columns = df.columns.tolist()

df_columns[0] = 'series'
df_columns[76] = 'check'
df_columns[77] = 'region'
df.columns = df_columns

col_ord = [0,77,76] + list(range(1,76))
df = df.iloc[:, col_ord]

# Clean df of rows without information and check data quality
df.dropna(inplace=True)
df[df['check']==1]['region'].value_counts()
df = df.drop(columns=['check'])
df.groupby('region')['series'].count()

region
Andalucía                     14
Aragón                        14
Canarias                      14
Cantabria                     14
Castilla y León               14
Castilla-La Mancha            14
Cataluña/Catalunya            14
Ciudad Autónoma de Ceuta      14
Ciudad Autónoma de Melilla    28
Comunidad Foral de Navarra    14
Comunidad de Madrid           14
Comunitat Valenciana          14
Extremadura                   14
Galicia                       14
Illes Balears                 14
La Rioja                      14
País Vasco/Euskadi            14
Principado de Asturias        14
Región de Murcia              14
Name: series, dtype: int64

In [163]:
# Separating GDP indicators and quality check

indicadores_PIB_ES = df['series'].unique().tolist()
indicadores_PIB_ES

['A. Agricultura, ganadería, silvicultura y pesca',
 'B_E. Industrias extractivas, industria manufacturera, suministro de energía eléctrica, gas, vapor y aire acondicionado, suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación',
 'C. - De las cuales: Industria manufacturera',
 'F. Construcción',
 'G_I. Comercio al por mayor y al por menor, reparación de vehículos de motor y motocicletas, transporte y almacenamiento, hostelería',
 'J. Información y comunicaciones',
 'K. Actividades financieras y de seguros',
 'L. Actividades inmobiliarias',
 'M_N. Actividades profesionales, científicas y técnicas, actividades administrativas y servicios auxiliares',
 'O_Q. Administración pública y defensa, seguridad social obligatoria, educación, actividades sanitarias y de servicios sociales',
 'R_U. Actividades artísticas, recreativas y de entretenimiento, reparación de artículos de uso doméstico y otros servicios',
 'Valor añadido bruto total',
 'Impuestos netos sobr

In [164]:
# Check format and values of columns

len(df.columns)

df_columns = df.columns.to_frame()
df_columns

df_columns.value_counts()


0      
2024(A)    3
2023(P)    3
2022.0     3
2021.0     3
2020.0     3
2019.0     3
2018.0     3
2017.0     3
2016.0     3
2015.0     3
2014.0     3
2013.0     3
2012.0     3
2011.0     3
2010.0     3
2009.0     3
2008.0     3
2007.0     3
2006.0     3
2005.0     3
2004.0     3
2003.0     3
2002.0     3
2001.0     3
2000.0     2
series     1
region     1
2000       1
Name: count, dtype: int64

Hay columnas repetidas, averiguamos en el df cuales no contien información relevante o repitida

In [165]:
# We see that each year column is repeated, we isolate an autonomous community and year to analyze its format

df_filtered = df.query('region == "Andalucía"')
df_filtered.iloc[:,0:28].head()


,series,region,2024(A),2023(P),2022.0,2021.0,2020.0,2019.0,2018.0,2017.0,...,2008.0,2007.0,2006.0,2005.0,2004.0,2003.0,2002.0,2001.0,2000.0,2024(A)
1,"A. Agricultura, ganadería, silvicultura y pesca",Andalucía,13498327,11791799,10264709.0,11156574.0,9678629.0,9344931.0,10325706.0,10896805.0,...,7334586.0,7361115.0,6485941.0,6993833.0,7069399.0,7187088.0,6455862.0,6537935.0,5901807.0,14.5
2,"B_E. Industrias extractivas, industria manufac...",Andalucía,22564579,21682646,22820010.0,18397711.0,15702698.0,17554177.0,17389075.0,17296189.0,...,17093403.0,16865975.0,16239306.0,15781887.0,14497975.0,13764727.0,12881204.0,12054859.0,11309912.0,4.1
3,C. - De las cuales: Industria manufacturera,Andalucía,14415090,13097590,12440246.0,11293938.0,9589054.0,11033185.0,10969900.0,11287757.0,...,12514902.0,12662622.0,12416502.0,11990284.0,11302884.0,10775641.0,10096016.0,9592713.0,9054775.0,10.1
4,F. Construcción,Andalucía,12383956,11817420,10980066.0,9711981.0,9345940.0,11013161.0,9931243.0,9193378.0,...,19401572.0,19537545.0,18805020.0,16921764.0,14658435.0,13106529.0,11598963.0,10167472.0,8764235.0,4.8
5,"G_I. Comercio al por mayor y al por menor, rep...",Andalucía,45740761,42275350,37743719.0,31983427.0,26895746.0,35207628.0,33827656.0,32746238.0,...,29080602.0,27702357.0,26302678.0,24636971.0,23656921.0,22236036.0,21023977.0,19472794.0,18244929.0,8.2


In [166]:
# The date range is from 2024 to 2020, we notice that after the 2020 column, 2024-2020 is repeated but with a different format

df_filtered.iloc[:,27:52].head()

# We notice negative numbers, so we assume it is an interannual variation

,2024(A),2023(P),2022.0,2021.0,2020.0,2019.0,2018.0,2017.0,2016.0,2015.0,...,2009.0,2008.0,2007.0,2006.0,2005.0,2004.0,2003.0,2002.0,2001.0,2000
1,14.5,14.9,-8.0,15.3,3.6,-9.5,-5.2,12.7,1.5,29.1,...,-6.5,-0.4,13.5,-7.3,-1.1,-1.6,11.3,-1.3,10.8,..
2,4.1,-5,24.0,17.2,-10.5,0.9,0.5,7.3,3.1,4.1,...,-14.2,1.3,3.9,2.9,8.9,5.3,6.9,6.9,6.6,..
3,10.1,5.3,10.1,17.8,-13.1,0.6,-2.8,8.5,3.3,5.8,...,-19.2,-1.2,2.0,3.6,6.1,4.9,6.7,5.2,5.9,..
4,4.8,7.6,13.1,3.9,-15.1,10.9,8.0,3.3,2.9,7.5,...,-10.6,-0.7,3.9,11.1,15.4,11.8,13.0,14.1,16.0,..
5,8.2,12,18.0,18.9,-23.6,4.1,3.3,4.6,3.1,5.6,...,-1.1,5.0,5.3,6.8,4.1,6.4,5.8,8.0,6.7,..


In [167]:
# Analyze the last columns

df_filtered.iloc[:,52:77].head()

# Again, we assume % variations or subtotals of community/country

,2024(A),2023(P),2022.0,2021.0,2020.0,2019.0,2018.0,2017.0,2016.0,2015.0,...,2009.0,2008.0,2007.0,2006.0,2005.0,2004.0,2003.0,2002.0,2001.0,2000.0
1,6.4,5.9,5.6,6.7,6.4,5.6,6.4,6.9,6.5,6.5,...,4.7,4.8,4.9,4.7,5.4,5.9,6.5,6.4,7.0,6.8
2,10.6,10.9,12.4,11.1,10.4,10.5,10.7,11.0,10.7,10.7,...,10.1,11.2,11.3,11.7,12.2,12.2,12.5,12.7,12.8,13.0
3,6.8,6.6,6.7,6.8,6.4,6.6,6.8,7.2,6.9,6.9,...,6.9,8.2,8.5,8.9,9.3,9.5,9.8,10.0,10.2,10.4
4,5.8,5.9,6.0,5.8,6.2,6.6,6.1,5.9,5.9,5.9,...,11.9,12.7,13.1,13.5,13.1,12.3,11.9,11.5,10.8,10.1
5,21.5,21.2,20.5,19.2,17.9,21.1,20.9,20.9,20.9,20.7,...,19.7,19.1,18.6,18.9,19.1,19.9,20.2,20.8,20.7,21.0


In [168]:
# We decide to keep only the first columns, as other ratios or variations can be calculated later

df = df.iloc[:,0:27]

# Rename columns

col_years = df.columns[2:27].to_list()
col_years = [col.strip() for col in col_years]
col_years = [col.replace('(A)', ' ') for col in col_years]
col_years = [col.replace('(P)', ' ') for col in col_years]
col_years = [col.replace('.0', ' ') for col in col_years]
col_years = [col.strip() for col in col_years]

df.columns = ['Index', 'Region'] + col_years

In [169]:
df.head()

,Index,Region,2024,2023,2022,2021,2020,2019,2018,2017,...,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
1,"A. Agricultura, ganadería, silvicultura y pesca",Andalucía,13498327,11791799,10264709.0,11156574.0,9678629.0,9344931.0,10325706.0,10896805.0,...,6855327.0,7334586.0,7361115.0,6485941.0,6993833.0,7069399.0,7187088.0,6455862.0,6537935.0,5901807.0
2,"B_E. Industrias extractivas, industria manufac...",Andalucía,22564579,21682646,22820010.0,18397711.0,15702698.0,17554177.0,17389075.0,17296189.0,...,14670626.0,17093403.0,16865975.0,16239306.0,15781887.0,14497975.0,13764727.0,12881204.0,12054859.0,11309912.0
3,C. - De las cuales: Industria manufacturera,Andalucía,14415090,13097590,12440246.0,11293938.0,9589054.0,11033185.0,10969900.0,11287757.0,...,10116536.0,12514902.0,12662622.0,12416502.0,11990284.0,11302884.0,10775641.0,10096016.0,9592713.0,9054775.0
4,F. Construcción,Andalucía,12383956,11817420,10980066.0,9711981.0,9345940.0,11013161.0,9931243.0,9193378.0,...,17353574.0,19401572.0,19537545.0,18805020.0,16921764.0,14658435.0,13106529.0,11598963.0,10167472.0,8764235.0
5,"G_I. Comercio al por mayor y al por menor, rep...",Andalucía,45740761,42275350,37743719.0,31983427.0,26895746.0,35207628.0,33827656.0,32746238.0,...,28750938.0,29080602.0,27702357.0,26302678.0,24636971.0,23656921.0,22236036.0,21023977.0,19472794.0,18244929.0


In [170]:
df.info()

<class 'pandas.DataFrame'>
Index: 280 entries, 1 to 299
Data columns (total 27 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Index   280 non-null    str    
 1   Region  280 non-null    str    
 2   2024    280 non-null    object 
 3   2023    280 non-null    object 
 4   2022    280 non-null    float64
 5   2021    280 non-null    float64
 6   2020    280 non-null    float64
 7   2019    280 non-null    float64
 8   2018    280 non-null    float64
 9   2017    280 non-null    float64
 10  2016    280 non-null    float64
 11  2015    280 non-null    float64
 12  2014    280 non-null    float64
 13  2013    280 non-null    float64
 14  2012    280 non-null    float64
 15  2011    280 non-null    float64
 16  2010    280 non-null    float64
 17  2009    280 non-null    float64
 18  2008    280 non-null    float64
 19  2007    280 non-null    float64
 20  2006    280 non-null    float64
 21  2005    280 non-null    float64
 22  2004    280 non-nu

In [171]:
# Clean data and prepare for subsequent grouping with df_IT

# Convert all year columns to numeric

cols = df.columns[2:27]
df[cols] = (df[cols].apply(pd.to_numeric, errors="coerce").astype("Int64"))

# Divide by 1000 to homogenize with IT data

df[cols] = df[cols] / 1000

df['Country'] = 'ES'

col_ord = [27,1,0] + list(range(2,27))
df = df.iloc[:, col_ord]
df.head()

,Country,Region,Index,2024,2023,2022,2021,2020,2019,2018,...,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
1,ES,Andalucía,"A. Agricultura, ganadería, silvicultura y pesca",13498.327,11791.799,10264.709,11156.574,9678.629,9344.931,10325.706,...,6855.327,7334.586,7361.115,6485.941,6993.833,7069.399,7187.088,6455.862,6537.935,5901.807
2,ES,Andalucía,"B_E. Industrias extractivas, industria manufac...",22564.579,21682.646,22820.01,18397.711,15702.698,17554.177,17389.075,...,14670.626,17093.403,16865.975,16239.306,15781.887,14497.975,13764.727,12881.204,12054.859,11309.912
3,ES,Andalucía,C. - De las cuales: Industria manufacturera,14415.09,13097.59,12440.246,11293.938,9589.054,11033.185,10969.9,...,10116.536,12514.902,12662.622,12416.502,11990.284,11302.884,10775.641,10096.016,9592.713,9054.775
4,ES,Andalucía,F. Construcción,12383.956,11817.42,10980.066,9711.981,9345.94,11013.161,9931.243,...,17353.574,19401.572,19537.545,18805.02,16921.764,14658.435,13106.529,11598.963,10167.472,8764.235
5,ES,Andalucía,"G_I. Comercio al por mayor y al por menor, rep...",45740.761,42275.35,37743.719,31983.427,26895.746,35207.628,33827.656,...,28750.938,29080.602,27702.357,26302.678,24636.971,23656.921,22236.036,21023.977,19472.794,18244.929


In [172]:
# Restructure: years to rows and Index values to columns
id_cols = ["Country", "Region", "Index"]

df_long = df.melt(
    id_vars=id_cols,
    var_name="year",
    value_name="value"
 )

# If there are duplicates by key, pivot_table resolves them with aggfunc
dup_keys = ["Country", "Region", "year", "Index"]
n_dups = df_long.duplicated(dup_keys).sum()

df_out = (
    df_long.pivot_table(
        index=["Country", "Region", "year"],
        columns="Index",
        values="value",
        aggfunc="first"  # change to 'mean' or 'sum' if you prefer another logic
    )
    .reset_index()
 )

df_out["year"] = pd.to_datetime(df_out["year"], errors="coerce")
df_out.columns.name = None

In [173]:
df_PIB_ES = df_out.copy()

In [174]:
df_PIB_ES

,Country,Region,year,"A. Agricultura, ganadería, silvicultura y pesca","B_E. Industrias extractivas, industria manufacturera, suministro de energía eléctrica, gas, vapor y aire acondicionado, suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación",C. - De las cuales: Industria manufacturera,F. Construcción,"G_I. Comercio al por mayor y al por menor, reparación de vehículos de motor y motocicletas, transporte y almacenamiento, hostelería",Impuestos netos sobre los productos,J. Información y comunicaciones,K. Actividades financieras y de seguros,L. Actividades inmobiliarias,"M_N. Actividades profesionales, científicas y técnicas, actividades administrativas y servicios auxiliares","O_Q. Administración pública y defensa, seguridad social obligatoria, educación, actividades sanitarias y de servicios sociales",PRODUCTO INTERIOR BRUTO A PRECIOS DE MERCADO,"R_U. Actividades artísticas, recreativas y de entretenimiento, reparación de artículos de uso doméstico y otros servicios",Valor añadido bruto total
0,ES,Andalucía,2000-01-01,5901.807,11309.912,9054.775,8764.235,18244.929,8038.233,3059.692,2822.459,4949.915,4923.978,15421.081,86760.722,3324.481,78722.489
1,ES,Andalucía,2001-01-01,6537.935,12054.859,9592.713,10167.472,19472.794,8454.758,3170.399,3219.959,5518.105,5271.152,16515.446,93964.317,3581.438,85509.559
2,ES,Andalucía,2002-01-01,6455.862,12881.204,10096.016,11598.963,21023.977,9122.035,3373.307,3524.13,6528.347,5445.426,17469.108,101269.401,3847.042,92147.366
3,ES,Andalucía,2003-01-01,7187.088,13764.727,10775.641,13106.529,22236.036,10395.439,3425.752,3747.18,7594.783,5738.303,18780.117,110093.641,4117.687,99698.202
4,ES,Andalucía,2004-01-01,7069.399,14497.975,11302.884,14658.435,23656.921,11845.59,3455.224,4076.708,9074.558,5963.743,20229.51,119026.615,4498.552,107181.025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,ES,Región de Murcia,2020-01-01,1499.188,5590.5,3815.341,1663.424,6184.319,2691.062,433.607,944.844,2834.299,1799.318,5736.877,30440.306,1062.868,27749.244
471,ES,Región de Murcia,2021-01-01,1573.468,6523.471,4498.201,1739.534,7050.579,3234.815,475.564,949.412,2908.133,2043.25,6038.382,33603.543,1066.935,30368.728
472,ES,Región de Murcia,2022-01-01,1507.202,8197.657,5792.727,1946.984,7993.656,3344.731,519.419,1078.797,3073.625,2139.3,6328.302,37380.806,1251.133,34036.075
473,ES,Región de Murcia,2023-01-01,2310.557,7516.343,5675.708,2125.461,8801.105,3532.245,569.459,1512.339,3340.152,2273.883,6562.904,39838.741,1294.293,36306.496


##### PIB_Regioni_Italiane.xlsx

In [175]:
# Importing excel

df = pd.read_excel(files_dict['PIB_Regioni_Italiane.xlsx'])

In [176]:
# Exploring data structure

df.head(20)

,Valore aggiunto per branca di attività,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,Frequenza: Annuale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aggregato: Valore aggiunto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Valutazione: Valori concatenati con anno di ri...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Correzione: Dati grezzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Tipologia di prezzo: Prezzi base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Edizione: Dic-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Tempo,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
8,Branca di attività economica (ATECO 2007),,,,,,,,,,
9,Territorio:Italia,,,,,,,,,,


In [177]:
df.tail(10)

,Valore aggiunto per branca di attività,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
1031,Attività amministrative e di servizi di suppor...,0,0,0,0,0,0,0,0,0,..
1032,"Amministrazione pubblica e difesa, assicurazio...",370.6,380.8,377.8,423,421.8,450.4,464.3,446.7,486.2,486.1
1033,"Amministrazione pubblica e difesa, assicurazio...",370.6,380.8,377.8,423,421.8,450.4,464.3,446.7,486.2,..
1034,"Amministrazione pubblica e difesa, assicurazio...",370.6,380.8,377.8,423,421.8,450.4,464.3,446.7,486.2,..
1035,Istruzione,0,0,0,0,0,0,0,0,0,..
1036,Sanità e assistenza sociale,0,0,0,0,0,0,0,0,0,..
1037,"Attività artistiche, di intrattenimento e dive...",0,0,0,0,0,0,0,0,0,..
1038,"Attività artistiche, di intrattenimento e dive...",0,0,0,0,0,0,0,0,0,..
1039,Altre attività di servizi,0,0,0,0,0,0,0,0,0,..
1040,Attività di famiglie e convivenze come datori ...,0,0,0,0,0,0,0,0,0,..


We notice heading in row 7. We also notice that the file present a final section with national totals. The section is market with "territory", we'll use a similar logic as before.

In [178]:
# Heading row 7

df.columns = df.iloc[7]
df = df.iloc[9:].reset_index(drop=True)
df.columns = df.columns.astype(str).str.strip()

# Mark rows with territory information, create ID_Regione column, and create a unique list of territories

df['Check'] = df['Tempo'].apply(lambda x: 1 if 'territorio' in str(x).lower() else 0)
df['ID_Regione'] = df['Check'].cumsum()

df_regio_PIB = df[df['Check'] == 1][['Tempo', 'ID_Regione']].reset_index(drop=True)
df_regio_PIB['Tempo'] = df_regio_PIB['Tempo'].str.replace('Territorio:',' ')
df_regio_PIB['Tempo'] = df_regio_PIB['Tempo'].str.strip()
df_regio_PIB


7,Tempo,ID_Regione
0,Italia,1
1,Piemonte,2
2,Valle d'Aosta / Vallée d'Aoste,3
3,Liguria,4
4,Lombardia,5
5,Trentino Alto Adige / Südtirol,6
6,Provincia Autonoma Bolzano / Bozen,7
7,Provincia Autonoma Trento,8
8,Veneto,9
9,Friuli-Venezia Giulia,10


In [179]:
# Call df Italian regions to standardize names

regioni = df_ciudades_it['denominazione_regione'].unique().tolist()
regioni

['Piemonte',
 "Valle d'Aosta/Vallée d'Aoste",
 'Liguria',
 'Lombardia',
 'Trentino-Alto Adige/Südtirol',
 'Veneto',
 'Friuli-Venezia Giulia',
 'Emilia-Romagna',
 'Marche',
 'Toscana',
 'Umbria',
 'Lazio',
 'Campania',
 'Abruzzo',
 'Molise',
 'Puglia',
 'Basilicata',
 'Calabria',
 'Sicilia',
 'Sardegna']

In [180]:
# Unify PIB region list and region list

df_regioni = pd.DataFrame({'Region': regioni})
df_regioni['Region_key'] = df_regioni['Region'].apply(canonical_key)

df_regio_PIB['Region_key'] = df_regio_PIB['Tempo'].apply(canonical_key)
df_regio_merge = df_regio_PIB.merge(df_regioni[['Region', 'Region_key']], on='Region_key', how='left')

df_regio_merge

,Tempo,ID_Regione,Region_key,Region
0,Italia,1,italia,NaN
1,Piemonte,2,piemonte,Piemonte
2,Valle d'Aosta / Vallée d'Aoste,3,valle d aosta vallee d aoste,Valle d'Aosta/Vallée d'Aoste
3,Liguria,4,liguria,Liguria
4,Lombardia,5,lombardia,Lombardia
5,Trentino Alto Adige / Südtirol,6,trentino alto adige sudtirol,Trentino-Alto Adige/Südtirol
6,Provincia Autonoma Bolzano / Bozen,7,provincia autonoma bolzano bozen,NaN
7,Provincia Autonoma Trento,8,provincia autonoma trento,NaN
8,Veneto,9,veneto,Veneto
9,Friuli-Venezia Giulia,10,friuli venezia giulia,Friuli-Venezia Giulia


The main differences are the aggregate "Italy" and "Extra-regio", which we'll eliminate. In the df_PIB we also notice like the row 6-7 cannot be compared, since we're speaking of the provinces belonging to the region "Trentino Alto Adige".

We'll eliminate these rows from df_PIB.

In [181]:
# We remove rows corresponding to aggregates that we decided not to consider in the analysis and add the name of the autonomous community

df = df[~df['ID_Regione'].isin([1,7,8,24])]
df = df[~df['Check'].isin([1])]
df = pd.merge(df,df_regio_merge,left_on='ID_Regione',right_on='ID_Regione',how='left')
df = df.drop(columns=['ID_Regione','Region_key','Tempo_y','Check'])

# Add country column for later integration

df['Country'] = 'IT'

# Reorder and rename columns

col_order = [12,11,0,1,2,3,4,5,6,7,8,9,10]
df = df.iloc[:,col_order]

df_columns = df.columns.astype(str).str.strip().tolist()
df_columns[2] = 'Index'
df.columns = df_columns

df.head()

,Country,Region,Index,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,IT,Piemonte,Totale attività economiche,117052.5,118577.2,122326.8,123446.3,123083.3,111721.9,121751.8,126269,128542.1,129987.5
1,IT,Piemonte,"Agricoltura, silvicoltura e pesca",1977.4,2036.7,1896.3,1948.7,1901,1744.3,1629.9,1560.6,1577.6,1596.6
2,IT,Piemonte,"Produzioni vegetali e animali, caccia e serviz...",1972.9,2032.5,1892.1,1944.4,1897,1740.3,1625.9,1557.4,1574.6,..
3,IT,Piemonte,Pesca e acquicoltura,4.5,4.2,4.2,4.3,4.1,4,4,3.2,3.1,..
4,IT,Piemonte,"Attività estrattiva, attività manifatturiere, ...",33425,34465.3,36018.5,36432.3,35872.9,31420.3,36621,37125.2,38950.5,38666.8


In [182]:
# Extract unique list of indicators

indicadores_PIB_IT = df['Index'].dropna().unique().tolist()
indicadores_PIB_IT

['Totale attività economiche  ',
 'Agricoltura, silvicoltura e pesca  ',
 'Produzioni vegetali e animali, caccia e servizi connessi, silvicultura  ',
 'Pesca e acquicoltura  ',
 'Attività estrattiva, attività manifatturiere, fornitura di energia elettrica, gas, vapore e aria condizionata, fornitura di acqua, reti fognarie, attività di trattamento dei rifiuti e risanamento, costruzioni  ',
 'Attività estrattiva, attività manifatturiere, fornitura di energia elettrica, gas, vapore e aria condizionata, fornitura di acqua, reti fognarie, attività di trattamento dei rifiuti e risanamento  ',
 'Industria estrattiva  ',
 'Industria manifatturiera  ',
 'Industrie alimentari, delle bevande e del tabacco  ',
 'Industrie tessili, confezione di articoli di abbigliamento e di articoli in pelle e simili  ',
 'Industria del legno, della carta, editoria  ',
 'Fabbricazione di coke e prodotti derivanti dalla raffinazione del petrolio, fabbricazione di prodotti chimici e farmaceutici  ',
 'Fabbricazione

In [183]:
# Analyze data types and clean data

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Country  840 non-null    str   
 1   Region   840 non-null    str   
 2   Index    840 non-null    str   
 3   2015     840 non-null    object
 4   2016     840 non-null    object
 5   2017     840 non-null    object
 6   2018     840 non-null    object
 7   2019     840 non-null    object
 8   2020     840 non-null    object
 9   2021     840 non-null    object
 10  2022     840 non-null    object
 11  2023     840 non-null    object
 12  2024     840 non-null    object
dtypes: object(10), str(3)
memory usage: 161.7+ KB


In [184]:
# Transform variables to int

df_num_col = list(df.columns[3:])
df[df_num_col] = df[df_num_col].apply(pd.to_numeric, errors='coerce', downcast='integer')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Country  840 non-null    str    
 1   Region   840 non-null    str    
 2   Index    840 non-null    str    
 3   2015     840 non-null    float64
 4   2016     840 non-null    float64
 5   2017     840 non-null    float64
 6   2018     840 non-null    float64
 7   2019     840 non-null    float64
 8   2020     840 non-null    float64
 9   2021     840 non-null    float64
 10  2022     840 non-null    float64
 11  2023     840 non-null    float64
 12  2024     180 non-null    float64
dtypes: float64(10), str(3)
memory usage: 161.7 KB


In [185]:
# Restructure: years to rows and Index values to columns
id_cols = ["Country", "Region", "Index"]

df_long = df.melt(
    id_vars=id_cols,
    var_name="year",
    value_name="value"
 )

# If there are duplicates by key, pivot_table resolves them with aggfunc
dup_keys = ["Country", "Region", "year", "Index"]
n_dups = df_long.duplicated(dup_keys).sum()

df_out = (
    df_long.pivot_table(
        index=["Country", "Region", "year"],
        columns="Index",
        values="value",
        aggfunc="first"  # change to 'mean' or 'sum' if you prefer another logic
    )
    .reset_index()
 )

df_out["year"] = pd.to_datetime(df_out["year"], errors="coerce")
df_out.columns.name = None
df_out.head()

,Country,Region,year,"Agricoltura, silvicoltura e pesca",Altre attività di servizi,"Amministrazione pubblica e difesa, assicurazione sociale obbligatoria","Amministrazione pubblica e difesa, assicurazione sociale obbligatoria, istruzione, sanità e assistenza sociale","Amministrazione pubblica e difesa, assicurazione sociale obbligatoria, istruzione, sanità e assistenza sociale, attività artistiche, di intrattenimento e divertimento, riparazione di beni per la casa e altri servizi",Attività amministrative e di servizi di supporto,"Attività artistiche, di intrattenimento e divertimento",...,"Industrie tessili, confezione di articoli di abbigliamento e di articoli in pelle e simili",Istruzione,Pesca e acquicoltura,"Produzioni vegetali e animali, caccia e servizi connessi, silvicultura",Sanità e assistenza sociale,Servizi,Servizi di alloggio e di ristorazione,Servizi di informazione e comunicazione,Totale attività economiche,Trasporti e magazzinaggio
0,IT,Abruzzo,2015-01-01,876.9,591.9,2975.7,6482.9,7694.0,823.9,315.1,...,496.6,1591.8,86.9,835.6,1908.2,20879.8,1110.8,396.5,29177.1,1208.8
1,IT,Abruzzo,2016-01-01,919.3,560.3,2918.3,6416.0,7605.3,835.2,342.5,...,497.3,1593.9,76.6,878.9,1902.4,20861.7,1145.4,418.0,29377.0,1250.5
2,IT,Abruzzo,2017-01-01,896.7,565.4,2807.4,6300.1,7497.8,937.2,357.8,...,497.1,1584.4,59.7,862.2,1912.8,20992.5,1175.7,417.2,29564.1,1340.1
3,IT,Abruzzo,2018-01-01,891.7,583.8,2766.5,6263.3,7457.8,958.1,346.1,...,533.0,1593.4,48.2,861.1,1912.3,21076.0,1226.5,433.0,29507.7,1341.3
4,IT,Abruzzo,2019-01-01,910.1,553.5,2750.5,6137.5,7252.8,947.5,303.7,...,530.0,1551.8,31.8,886.6,1837.3,21034.7,1216.9,441.8,29771.0,1379.8


In [186]:
# Check if the indicator "Totale attività economiche" corresponds to the total calculated from the sum of the indicators

df_out.columns = df_out.columns.astype(str).str.strip()

df_col = df_out.columns.to_list()
df_col_sum = [col for col in df_col if col not in ['Country', 'Region', 'year','Totale attività economiche']]

df_out['Totale_Indicatori'] = df_out[df_col_sum].sum(axis=1)
df_out['var'] = df_out['Totale attività economiche'] - df_out['Totale_Indicatori']

df_out.groupby('year')[['Totale attività economiche', 'Totale_Indicatori', 'var']].sum().head(20)

,Totale attività economiche,Totale_Indicatori,var
year,,,
2015-01-01,1553850.5,5629526.8,-4075676.3
2016-01-01,1574532.0,5706150.4,-4131618.4
2017-01-01,1603428.2,5819495.8,-4216067.6
2018-01-01,1619331.1,5877303.9,-4257972.8
2019-01-01,1630950.9,5915857.2,-4284906.3
2020-01-01,1495469.5,5392135.6,-3896666.1
2021-01-01,1629096.7,5887991.0,-4258894.3
2022-01-01,1710594.9,6182707.7,-4472112.8
2023-01-01,1727122.4,6232798.2,-4505675.8


We notice a significant difference between the total in the dataset and the one calculated, which points to the presence of subtotals and grouping of indicators. We explore the original dataset in order to map subtotals and details indicators.

In [187]:
# Creating list of breakdown indicators to select them for later analysis and comparison with the total indicator

indicadores_PIB_IT = ['Produzioni vegetali e animali, caccia e servizi connessi, silvicultura','Pesca e acquicoltura','Industria estrattiva','Industria manifatturiera',
                       'Fornitura di energia elettrica, gas, vapore e aria condizionata','Fornitura di acqua, reti fognarie, attività di trattamento dei rifiuti e risanamento',
                       'Costruzioni',"Commercio all'ingrosso e al dettaglio, riparazione di autoveicoli e motocicli",'Trasporti e magazzinaggio','Servizi di alloggio e di ristorazione',
                       'Servizi di informazione e comunicazione','Attività finanziarie e assicurative','Attività immobiliari','Attività professionali, scientifiche e tecniche',
                       'Attività amministrative e di servizi di supporto','Amministrazione pubblica e difesa, assicurazione sociale obbligatoria','Istruzione',
                       'Sanità e assistenza sociale','Attività artistiche, di intrattenimento e divertimento','Altre attività di servizi','Attività di famiglie e convivenze come datori di lavoro per personale domestico, produzione di beni e servizi indifferenziati per uso proprio da parte di famiglie e convivenze']

# Select breakdown indicators and the total indicator for further analysis

df_out = df_out[['Country', 'Region', 'year','Totale attività economiche'] + [col for col in df_out.columns if col in indicadores_PIB_IT]]
df_out['Totale_Indicatori'] = df_out[[col for col in df_out.columns if col in indicadores_PIB_IT]].sum(axis=1)
df_out['var'] = df_out['Totale attività economiche'] - df_out['Totale_Indicatori']

# Check quality

df_out.groupby('year')[['Totale attività economiche', 'Totale_Indicatori']].sum().head(20)

,Totale attività economiche,Totale_Indicatori
year,,
2015-01-01,1553850.5,1558657.4
2016-01-01,1574532.0,1577970.8
2017-01-01,1603428.2,1606389.9
2018-01-01,1619331.1,1622079.0
2019-01-01,1630950.9,1633232.4
2020-01-01,1495469.5,1495469.3
2021-01-01,1629096.7,1629096.7
2022-01-01,1710594.9,1710435.0
2023-01-01,1727122.4,1728008.9


The breakdown seems valid, we notice a difference in 2024, which we suppose is due to a lack of breakdown information. We filter out these data.

In [188]:
# Eliminating 2024 data due to lack of information

df_out = df_out[df_out['year'] < '2024-01-01']
df_out.reset_index(drop=True, inplace=True)

In [189]:
# Save variable

df_PIB_IT = df_out.copy()

We notice that in this file there's not information regarding the total GDP like DF_ES.

We derive this information from another file: PIB_Regioni_Italiane_total.csv

In [190]:
# Importing excel

df_PIB_IT_total = pd.read_csv(files_dict['PIB_Regioni_Italiane_total.csv'], decimal=',')

In [191]:
# Exploring dataset structure

df_PIB_IT_total.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE_AGGR,Aggregato,VALUATION,Valutazione,ADJUSTMENT,Correzione,...,NOTE_ADJUSTMENT,Correzione (NOTE_ADJUSTMENT),NOTE_EDITION,Edizione (NOTE_EDITION),BASE_PER,Anno base,UNIT_MEAS,Unità di misura,UNIT_MULT,Unità di moltiplicazione
0,A,Annuale,ITC1,Piemonte,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
1,A,Annuale,ITC1,Piemonte,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
2,A,Annuale,ITC1,Piemonte,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
3,A,Annuale,ITC1,Piemonte,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
4,A,Annuale,ITC1,Piemonte,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni


In [192]:
df_PIB_IT_total.tail()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE_AGGR,Aggregato,VALUATION,Valutazione,ADJUSTMENT,Correzione,...,NOTE_ADJUSTMENT,Correzione (NOTE_ADJUSTMENT),NOTE_EDITION,Edizione (NOTE_EDITION),BASE_PER,Anno base,UNIT_MEAS,Unità di misura,UNIT_MULT,Unità di moltiplicazione
195,A,Annuale,ITG2,Sardegna,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
196,A,Annuale,ITG2,Sardegna,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
197,A,Annuale,ITG2,Sardegna,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
198,A,Annuale,ITG2,Sardegna,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni
199,A,Annuale,ITG2,Sardegna,B1GQ_B_W2_S1,Prodotto interno lordo ai prezzi di mercato,V,Prezzi correnti,N,Dati grezzi,...,NaN,NaN,NaN,NaN,NaN,NaN,EURO,Euro,6,Milioni


In [193]:
# Visualizing columns

df_PIB_IT_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 32 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   FREQ                             200 non-null    str    
 1   Frequenza                        200 non-null    str    
 2   REF_AREA                         200 non-null    str    
 3   Territorio                       200 non-null    str    
 4   DATA_TYPE_AGGR                   200 non-null    str    
 5   Aggregato                        200 non-null    str    
 6   VALUATION                        200 non-null    str    
 7   Valutazione                      200 non-null    str    
 8   ADJUSTMENT                       200 non-null    str    
 9   Correzione                       200 non-null    str    
 10  EDITION                          200 non-null    str    
 11  Edizione                         200 non-null    str    
 12  TIME_PERIOD                      

In [194]:
# Eliminating columns without relevant information and renaming

df_PIB_IT_total.columns = df_PIB_IT_total.columns.astype(str).str.strip()

df_PIB_IT_total = df_PIB_IT_total[['Territorio', 'TIME_PERIOD', 'Osservazione']]
df_PIB_IT_total.columns = ['Region', 'year', 'Total_GDP']

df_PIB_IT_total.head()

,Region,year,Total_GDP
0,Piemonte,2015,124970.1
1,Piemonte,2016,128292.3
2,Piemonte,2017,132625.4
3,Piemonte,2018,135216.6
4,Piemonte,2019,135773.8


In [195]:
df_PIB_IT['Region'].unique()

<ArrowStringArray>
[                     'Abruzzo',                   'Basilicata',
                     'Calabria',                     'Campania',
               'Emilia-Romagna',        'Friuli-Venezia Giulia',
                        'Lazio',                      'Liguria',
                    'Lombardia',                       'Marche',
                       'Molise',                     'Piemonte',
                       'Puglia',                     'Sardegna',
                      'Sicilia',                      'Toscana',
 'Trentino-Alto Adige/Südtirol',                       'Umbria',
 'Valle d'Aosta/Vallée d'Aoste',                       'Veneto']
Length: 20, dtype: str

In [196]:
# Unifying region names in the PIB dataset with the list of regions in the df of financial institutions for later merging

regioni_map = {
    'Piemonte':'Piemonte',
    '\'Valle d"\'Aosta / Vallée d"\'Aoste\'':"Valle d'Aosta/Vallée d'Aoste",
    'Liguria':'Liguria',
    'Lombardia':'Lombardia',
    'Trentino Alto Adige / Südtirol':'Trentino-Alto Adige/Südtirol',
    'Veneto':'Veneto',
    'Friuli-Venezia Giulia':'Friuli-Venezia Giulia',
    'Emilia-Romagna':'Emilia-Romagna',
    'Toscana':'Toscana',
    'Umbria':'Umbria',
    'Marche':'Marche',
    'Lazio':'Lazio',
    'Abruzzo':'Abruzzo',
    'Molise':'Molise',
    'Campania':'Campania',
    'Puglia':'Puglia',
    'Basilicata':'Basilicata',
    'Calabria':'Calabria',
    'Sicilia':'Sicilia',
    'Sardegna':'Sardegna'
}

df_PIB_IT_total['Region'] = df_PIB_IT_total['Region'].map(regioni_map)


In [197]:
# Transforming types

df_PIB_IT_total['Total_GDP'] = pd.to_numeric(df_PIB_IT_total['Total_GDP'], errors='coerce')
df_PIB_IT_total['year'] = pd.to_datetime(df_PIB_IT_total['year'].astype(str), errors='coerce')
df_PIB_IT_total['Region'] = df_PIB_IT_total['Region'].astype(str).str.strip()
df_PIB_IT_total.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Region     200 non-null    str           
 1   year       200 non-null    datetime64[us]
 2   Total_GDP  200 non-null    float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 6.9 KB


In [198]:
# Mergin with the breakdown indicators dataset for later comparison and analysis

df_PIB_IT = pd.merge(df_PIB_IT, df_PIB_IT_total, on=['Region','year'], how='left')

In [199]:
# Updating indicators list

indicadores_PIB_IT = df_PIB_IT.columns.to_list()
indicadores_PIB_IT.remove('Country')
indicadores_PIB_IT.remove('Region')
indicadores_PIB_IT.remove('year')
indicadores_PIB_IT.remove('var')
indicadores_PIB_IT.remove('Totale_Indicatori')

##### Comparison and consolidation

We try to reduce heavily the number of indicators and unfying between the 2 datasets.

The idea is to group variables in order to have a clearer idea of the GDP composition for each region.

In [200]:
# Analizamos los indicatore de PIB_ES

indicadores_PIB_ES

['A. Agricultura, ganadería, silvicultura y pesca',
 'B_E. Industrias extractivas, industria manufacturera, suministro de energía eléctrica, gas, vapor y aire acondicionado, suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación',
 'C. - De las cuales: Industria manufacturera',
 'F. Construcción',
 'G_I. Comercio al por mayor y al por menor, reparación de vehículos de motor y motocicletas, transporte y almacenamiento, hostelería',
 'J. Información y comunicaciones',
 'K. Actividades financieras y de seguros',
 'L. Actividades inmobiliarias',
 'M_N. Actividades profesionales, científicas y técnicas, actividades administrativas y servicios auxiliares',
 'O_Q. Administración pública y defensa, seguridad social obligatoria, educación, actividades sanitarias y de servicios sociales',
 'R_U. Actividades artísticas, recreativas y de entretenimiento, reparación de artículos de uso doméstico y otros servicios',
 'Valor añadido bruto total',
 'Impuestos netos sobr

The consolidation proposal is the following:

1. Primary sector
2. Secundary sector
3. Tertiary sector
4. Tertiary sector: financial activities
5. Public sector
6. Other
7. Total PIB

In [201]:
# Analizying indicator C in the df_es

print(df_PIB_ES['B_E. Industrias extractivas, industria manufacturera, suministro de energía eléctrica, gas, vapor y aire acondicionado, suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación'].describe())
print(df_PIB_ES['C. - De las cuales: Industria manufacturera'].describe())


count          475.0
mean     8608.490526
std      8972.498349
min           20.298
25%        2337.5795
50%           5590.5
75%        13699.503
max        51390.788
Name: B_E. Industrias extractivas, industria manufacturera, suministro de energía eléctrica, gas, vapor y aire acondicionado, suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación, dtype: Float64
count          475.0
mean         6705.24
std      7565.091883
min            11.74
25%         1502.018
50%          4270.73
75%       10358.4595
max          42894.3
Name: C. - De las cuales: Industria manufacturera, dtype: Float64


It seems that the C indicator is contained in the B_E indicator, we create a new breakdown indicator to mark the difference and correctly categorise it.

In [202]:
# Creating a new indicator to separate manufacturing industry from the rest of industries

df_PIB_ES['B_E_Industrias_extractivas_suministro_energia_agua'] = (
    df_PIB_ES['B_E. Industrias extractivas, industria manufacturera, suministro de energía eléctrica, gas, vapor y aire acondicionado, suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación'] - 
    df_PIB_ES['C. - De las cuales: Industria manufacturera']
)

In [203]:
# Creating indicators categories

Primary = ["A. Agricultura, ganadería, silvicultura y pesca","B_E_Industrias_extractivas_suministro_energia_agua"]
Secundary = ["C. - De las cuales: Industria manufacturera","F. Construcción"]
Tertiary = ["G_I. Comercio al por mayor y al por menor, reparación de vehículos de motor y motocicletas, transporte y almacenamiento, hostelería",
            "J. Información y comunicaciones","L. Actividades inmobiliarias"]
Tertiary_financial = ["K. Actividades financieras y de seguros"]
Public = ["O_Q. Administración pública y defensa, seguridad social obligatoria, educación, actividades sanitarias y de servicios sociales"]
Other = ["M_N. Actividades profesionales, científicas y técnicas, actividades administrativas y servicios auxiliares","R_U. Actividades artísticas, recreativas y de entretenimiento, reparación de artículos de uso doméstico y otros servicios"]

# Creating indicator category columns and total

df_PIB_ES['Primary'] = df_PIB_ES[Primary].sum(axis=1)
df_PIB_ES['Secundary'] = df_PIB_ES[Secundary].sum(axis=1)
df_PIB_ES['Tertiary'] = df_PIB_ES[Tertiary].sum(axis=1)
df_PIB_ES['Tertiary_financial'] = df_PIB_ES[Tertiary_financial].sum(axis=1)
df_PIB_ES['Public'] = df_PIB_ES[Public].sum(axis=1)
df_PIB_ES['Other'] = df_PIB_ES[Other].sum(axis=1)
df_PIB_ES['Total_GDP'] = df_PIB_ES['PRODUCTO INTERIOR BRUTO A PRECIOS DE MERCADO']

# Visualizing df_PIB_ES with categories

df_PIB_ES = df_PIB_ES[['Country', 'Region', 'year', 'Primary', 'Secundary', 'Tertiary', 'Tertiary_financial', 'Public', 'Other','Total_GDP']]
df_PIB_ES.head()

,Country,Region,year,Primary,Secundary,Tertiary,Tertiary_financial,Public,Other,Total_GDP
0,ES,Andalucía,2000-01-01,8156.944,17819.01,26254.536,2822.459,15421.081,8248.459,86760.722
1,ES,Andalucía,2001-01-01,9000.081,19760.185,28161.298,3219.959,16515.446,8852.59,93964.317
2,ES,Andalucía,2002-01-01,9241.05,21694.979,30925.631,3524.13,17469.108,9292.468,101269.401
3,ES,Andalucía,2003-01-01,10176.174,23882.17,33256.571,3747.18,18780.117,9855.99,110093.641
4,ES,Andalucía,2004-01-01,10264.49,25961.319,36186.703,4076.708,20229.51,10462.295,119026.615


In [204]:
# Analizying the indicators of PIB_IT

indicadores_PIB_IT

['Totale attività economiche',
 'Altre attività di servizi',
 'Amministrazione pubblica e difesa, assicurazione sociale obbligatoria',
 'Attività amministrative e di servizi di supporto',
 'Attività artistiche, di intrattenimento e divertimento',
 'Attività di famiglie e convivenze come datori di lavoro per personale domestico, produzione di beni e servizi indifferenziati per uso proprio da parte di famiglie e convivenze',
 'Attività finanziarie e assicurative',
 'Attività immobiliari',
 'Attività professionali, scientifiche e tecniche',
 "Commercio all'ingrosso e al dettaglio, riparazione di autoveicoli e motocicli",
 'Costruzioni',
 'Fornitura di acqua, reti fognarie, attività di trattamento dei rifiuti e risanamento',
 'Fornitura di energia elettrica, gas, vapore e aria condizionata',
 'Industria estrattiva',
 'Industria manifatturiera',
 'Istruzione',
 'Pesca e acquicoltura',
 'Produzioni vegetali e animali, caccia e servizi connessi, silvicultura',
 'Sanità e assistenza sociale',


The list of indicators is much more detailled, but we notice an order according to the specific sector, we operate utilizing the list of indicators we just created.

In [205]:
len(indicadores_PIB_IT)

23

In [206]:
# Creating indicator categories

Primary = ['Industria estrattiva','Pesca e acquicoltura','Produzioni vegetali e animali, caccia e servizi connessi, silvicultura']
Secundary = ['Costruzioni','Fornitura di acqua, reti fognarie, attività di trattamento dei rifiuti e risanamento','Fornitura di energia elettrica, gas, vapore e aria condizionata','Industria manifatturiera']
Tertiary = ['Altre attività di servizi','Attività amministrative e di servizi di supporto','Attività immobiliari',"Commercio all'ingrosso e al dettaglio, riparazione di autoveicoli e motocicli",'Servizi di alloggio e di ristorazione','Servizi di informazione e comunicazione','Trasporti e magazzinaggio']
Tertiary_financial = ['Attività finanziarie e assicurative']
Public = ['Amministrazione pubblica e difesa, assicurazione sociale obbligatoria','Sanità e assistenza sociale']
Other = ['Attività artistiche, di intrattenimento e divertimento','Attività di famiglie e convivenze come datori di lavoro per personale domestico, produzione di beni e servizi indifferenziati per uso proprio da parte di famiglie e convivenze','Attività professionali, scientifiche e tecniche','Istruzione']

# Creaming indicator category columns and total

df_PIB_IT['Primary'] = df_PIB_IT[Primary].sum(axis=1)
df_PIB_IT['Secundary'] = df_PIB_IT[Secundary].sum(axis=1)
df_PIB_IT['Tertiary'] = df_PIB_IT[Tertiary].sum(axis=1)
df_PIB_IT['Tertiary_financial'] = df_PIB_IT[Tertiary_financial].sum(axis=1)
df_PIB_IT['Public'] = df_PIB_IT[Public].sum(axis=1)
df_PIB_IT['Other'] = df_PIB_IT[Other].sum(axis=1)

# Visualizing df_PIB_IT with categories

df_PIB_IT = df_PIB_IT[['Country', 'Region', 'year', 'Primary', 'Secundary', 'Tertiary', 'Tertiary_financial', 'Public', 'Other', 'Total_GDP']]
df_PIB_IT.head()

,Country,Region,year,Primary,Secundary,Tertiary,Tertiary_financial,Public,Other,Total_GDP
0,IT,Abruzzo,2015-01-01,985.7,7390.7,11194.2,1102.0,4883.9,3712.0,31791.1
1,IT,Abruzzo,2016-01-01,1033.1,7535.0,11256.5,1109.0,4820.7,3692.4,31926.0
2,IT,Abruzzo,2017-01-01,995.4,7620.1,11468.7,1080.8,4720.2,3741.3,32733.3
3,IT,Abruzzo,2018-01-01,976.2,7490.1,11644.1,1065.7,4678.8,3710.2,32798.6
4,IT,Abruzzo,2019-01-01,991.8,7775.1,11832.8,1035.4,4587.8,3593.1,33030.0


In [207]:
# Comparing date ranges of the two dfs

print(f"Date range for df_PIB_IT:{df_PIB_IT['year'].describe()}")
print(f"Date range for df_PIB_ES:{df_PIB_ES['year'].describe()}")

Date range for df_PIB_IT:count                    180
mean     2019-01-01 02:40:00
min      2015-01-01 00:00:00
25%      2017-01-01 00:00:00
50%      2019-01-01 00:00:00
75%      2021-01-01 00:00:00
max      2023-01-01 00:00:00
Name: year, dtype: object
Date range for df_PIB_ES:count                    475
mean     2012-01-01 08:38:24
min      2000-01-01 00:00:00
25%      2006-01-01 00:00:00
50%      2012-01-01 00:00:00
75%      2018-01-01 00:00:00
max      2024-01-01 00:00:00
Name: year, dtype: object


In [208]:
# Filtering df_PIB_ES for the initial date of df_PIB_IT (Keeping df_PIB_ES data for integration with R&D data)

start_date = df_PIB_IT['year'].min()
df_PIB_ES = df_PIB_ES[df_PIB_ES['year'] >= start_date]

In [209]:
# Unifying df_PIB_IT and df_PIB_ES

df_PIB_merged = pd.concat([df_PIB_IT, df_PIB_ES], ignore_index=True)
df_PIB_merged.head()

,Country,Region,year,Primary,Secundary,Tertiary,Tertiary_financial,Public,Other,Total_GDP
0,IT,Abruzzo,2015-01-01,985.7,7390.7,11194.2,1102.0,4883.9,3712.0,31791.1
1,IT,Abruzzo,2016-01-01,1033.1,7535.0,11256.5,1109.0,4820.7,3692.4,31926.0
2,IT,Abruzzo,2017-01-01,995.4,7620.1,11468.7,1080.8,4720.2,3741.3,32733.3
3,IT,Abruzzo,2018-01-01,976.2,7490.1,11644.1,1065.7,4678.8,3710.2,32798.6
4,IT,Abruzzo,2019-01-01,991.8,7775.1,11832.8,1035.4,4587.8,3593.1,33030.0


In [210]:
# Transforming the shape of the dataset
df_PIB_final = df_PIB_merged.melt(
    id_vars=['Country', 'Region', 'year'],
    var_name='index',
    value_name='value'
)
df_PIB_final.head()

,Country,Region,year,index,value
0,IT,Abruzzo,2015-01-01,Primary,985.7
1,IT,Abruzzo,2016-01-01,Primary,1033.1
2,IT,Abruzzo,2017-01-01,Primary,995.4
3,IT,Abruzzo,2018-01-01,Primary,976.2
4,IT,Abruzzo,2019-01-01,Primary,991.8


#### R&D

##### I_D_Comunidades_autonomas.xlsx

In [211]:
# Importing excel

df = pd.read_excel(files_dict['I_D_Comunidades_autonomas.xlsx'])


In [212]:
# Exploring dataset structure

df.head(20)

,Resultados detallados 2024,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,Resultados por comunidades autónomas,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,Sector Empresas. Gasto y personal en I+D inter...,NaN,NaN,NaN
3,Unidades: Especificada en las variables,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,,Gasto en I+D interna (miles de euros),Gasto en I+D interna (%),Personal en EJC: Total
6,Total Nacional,23930881,100,139436.6
7,01 Andalucía,2309973,6.5,10336.5
8,02 Aragón,569365,2.4,4142.8
9,"03 Asturias, Principado de",307867,1.4,2119.1


In [213]:
df.tail(10)

,Resultados detallados 2024,Unnamed: 1,Unnamed: 2,Unnamed: 3
24,18 Ceuta,5426,..,..
25,19 Melilla,8635,..,..
26,NaN,NaN,NaN,NaN
27,NaN,NaN,NaN,NaN
28,Notas:,NaN,NaN,NaN
29,1) '..'=dato protegido por secreto estadístico,NaN,NaN,NaN
30,2) EJC: equivalencia a jornada completa,NaN,NaN,NaN
31,NaN,NaN,NaN,NaN
32,Fuente:,NaN,NaN,NaN
33,Instituto Nacional de Estadística,NaN,NaN,NaN


In [214]:
# Eliminating rows without relevant information and renaming columns

df.columns = df.iloc[5]
df.columns = df.columns.astype(str).str.strip()

df = df.iloc[7:-8].reset_index(drop=True)

df_columns = df.columns.tolist()
df_columns[0] = 'Region'
df_columns[1] = 'R&D Spending'

df.columns = df_columns

# We keep only the total R&D spending by autonomous community

df = df[['Region', 'R&D Spending']]

# Adding country and year columns for later integration

df['Country'] = 'ES'
df['Year'] = 2024

# Reordering columns

df = df[['Country', 'Region', 'Year', 'R&D Spending']]
df.head()

,Country,Region,Year,R&D Spending
0,ES,01 Andalucía,2024,2309973
1,ES,02 Aragón,2024,569365
2,ES,"03 Asturias, Principado de",2024,307867
3,ES,"04 Balears, Illes",2024,176351
4,ES,05 Canarias,2024,305907


In [215]:
# Renaming regions to unify with df_PIB_ES

df_PIB_ES['Region'] = df_PIB_ES['Region'].str.strip()
df['Region'] = df['Region'].str.strip()

region_mapping = {
    '01 Andalucía':                   'Andalucía',
    '02 Aragón':                      'Aragón',
    '03 Asturias, Principado de':     'Principado de Asturias',
    '04 Balears, Illes':              'Illes Balears',
    '05 Canarias':                    'Canarias',
    '06 Cantabria':                   'Cantabria',
    '07 Castilla y León':             'Castilla y León',
    '08 Castilla - La Mancha':        'Castilla-La Mancha',
    '09 Cataluña':                    'Cataluña/Catalunya',
    '10 Comunitat Valenciana':        'Comunitat Valenciana',
    '11 Extremadura':                 'Extremadura',
    '12 Galicia':                     'Galicia',
    '13 Madrid, Comunidad de':        'Comunidad de Madrid',
    '14 Murcia, Región de':           'Región de Murcia',
    '15 Navarra, Comunidad Foral de': 'Comunidad Foral de Navarra',
    '16 País Vasco':                  'País Vasco/Euskadi',
    '17 Rioja, La':                   'La Rioja',
    '18 Ceuta':                       'Ciudad Autónoma de Ceuta',
    '19 Melilla':                     'Ciudad Autónoma de Melilla',
}

df['Region'] = df['Region'].map(region_mapping)
df.head()

,Country,Region,Year,R&D Spending
0,ES,Andalucía,2024,2309973
1,ES,Aragón,2024,569365
2,ES,Principado de Asturias,2024,307867
3,ES,Illes Balears,2024,176351
4,ES,Canarias,2024,305907


In [216]:
# Eliminating rows without information and cleaning types

df.dropna(inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Country       19 non-null     str   
 1   Region        19 non-null     str   
 2   Year          19 non-null     int64 
 3   R&D Spending  19 non-null     object
dtypes: int64(1), object(1), str(2)
memory usage: 1.1+ KB


In [217]:
# Changing types

df['Year'] = pd.to_datetime(df['Year'], format='%Y', errors='coerce')
df['R&D Spending'] = pd.to_numeric(df['R&D Spending'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Country       19 non-null     str           
 1   Region        19 non-null     str           
 2   Year          19 non-null     datetime64[us]
 3   R&D Spending  19 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 1.1 KB


In [218]:
# Saving df in a variable for later integration

df_R_D_ES = df.copy()

##### I_D_Regioni_Italiane.xlsx

In [219]:
# Importing excel

df = pd.read_excel(files_dict['I_D_Regioni_Italiane.xlsx'])

In [220]:
# Exploring dataset structure

df.head(20)

,Spesa - reg.,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,Frequenza: Annuale,NaN,NaN,NaN
1,Indicatore: Spesa per ricerca e sviluppo intra...,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Tempo,2023,2024,2025
4,Settore istituzionale,,,
5,Territorio:Italia,,,
6,Totale economia,29399402,..,..
7,Imprese,17155883,17365769,18067173
8,Istituzioni pubbliche (escluse università pubb...,4370870,4659747,4996060
9,Università (pubbliche e private),7362530,..,..


In [221]:
df.tail(10)

,Spesa - reg.,Unnamed: 1,Unnamed: 2,Unnamed: 3
163,Imprese,371552,..,..
164,Istituzioni pubbliche (escluse università pubb...,224997,..,..
165,Università (pubbliche e private),436862,..,..
166,Istituzioni private non profit (escluse univer...,18510,..,..
167,Territorio:Sardegna,,,
168,Totale economia,348692,..,..
169,Imprese,55693,..,..
170,Istituzioni pubbliche (escluse università pubb...,89986,..,..
171,Università (pubbliche e private),201515,..,..
172,Istituzioni private non profit (escluse univer...,1498,..,..


We notice information for 3 years: 2023, 2024, 2025. We keep only the information about 2024 to compare it with Spain.

We examine the first column and we notice a grouping by territory and category: total, companies, public sector etc.

We clean the dataset to only keep the information about each region and total.

In [222]:
# Eliminating rows without relevant information and renaming columns

df.columns = df.iloc[3]
df = df.iloc[17:].reset_index(drop=True)

df.head()

3,Tempo,2023,2024,2025
0,Territorio:Piemonte,,,
1,Totale economia,3319321,..,..
2,Imprese,2575936,..,..
3,Istituzioni pubbliche (escluse università pubb...,122690,..,..
4,Università (pubbliche e private),562360,..,..


We notice there's no information for year 2024 and 2025, we'll then keep the 2023 data, which we'll compare at % level with Spain.

In [223]:
# Eliminating columns without relevant information, adding country and year columns, and reordering for later integration

df.columns = df.columns.astype(str).str.strip()

df = df[['Tempo', '2023']]

df['Country'] = 'IT'
df['Year'] = 2023

df.columns = ['Region', 'R&D Spending', 'Country', 'Year']
df = df[['Country', 'Region', 'Year', 'R&D Spending']]

df.head()

,Country,Region,Year,R&D Spending
0,IT,Territorio:Piemonte,2023,
1,IT,Totale economia,2023,3319321
2,IT,Imprese,2023,2575936
3,IT,Istituzioni pubbliche (escluse università pubb...,2023,122690
4,IT,Università (pubbliche e private),2023,562360


In [224]:
# We try to keep only the rows with territorial information by filtering for non-null values in the R&D spending column

i = df['R&D Spending'].loc[0]

Territories_list = df[df['R&D Spending'] == i]
Territories_list = Territories_list['Region'].tolist()
Territories_list

['Territorio:Piemonte',
 "Territorio:Valle d'Aosta / Vallée d'Aoste",
 'Territorio:Liguria',
 'Territorio:Lombardia',
 'Territorio:Nord-est',
 'Territorio:Trentino Alto Adige / Südtirol',
 'Territorio:Provincia Autonoma Bolzano / Bozen',
 'Territorio:Provincia Autonoma Trento',
 'Territorio:Veneto',
 'Territorio:Friuli-Venezia Giulia',
 'Territorio:Emilia-Romagna',
 'Territorio:Centro',
 'Territorio:Toscana',
 'Territorio:Umbria',
 'Territorio:Marche',
 'Territorio:Lazio',
 'Territorio:Sud',
 'Territorio:Abruzzo',
 'Territorio:Molise',
 'Territorio:Campania',
 'Territorio:Puglia',
 'Territorio:Basilicata',
 'Territorio:Calabria',
 'Territorio:Isole',
 'Territorio:Sicilia',
 'Territorio:Sardegna']

In [225]:
# Creating regions maps to compare with the list of regions in df_PIB_IT and unify names

territories_check = df_PIB_IT['Region'].unique()
territories_check = [r.strip() for r in territories_check]

territories_map = {
    'Territorio:Piemonte': 'Piemonte',
    'Territorio:Valle d\'Aosta / Vallée d\'Aoste': 'Valle d\'Aosta/Vallée d\'Aoste',
    'Territorio:Liguria': 'Liguria',
    'Territorio:Lombardia': 'Lombardia',
    'Territorio:Nord-est': 'Territorio:Nord-est',
    'Territorio:Trentino Alto Adige / Südtirol': 'Trentino-Alto Adige/Südtirol',
    'Territorio:Provincia Autonoma Bolzano / Bozen': 'Provincia Autonoma Bolzano / Bozen',
    'Territorio:Provincia Autonoma Trento': 'Provincia Autonoma Trento',
    'Territorio:Veneto': 'Veneto',
    'Territorio:Friuli-Venezia Giulia': 'Friuli-Venezia Giulia',
    'Territorio:Emilia-Romagna': 'Emilia-Romagna',
    'Territorio:Centro':'Territorio:Centro',
    'Territorio:Toscana': 'Toscana',
    'Territorio:Umbria': 'Umbria',
    'Territorio:Marche': 'Marche',
    'Territorio:Lazio': 'Lazio',
    'Territorio:Sud':'Territorio:Sud',
    'Territorio:Abruzzo': 'Abruzzo',
    'Territorio:Molise': 'Molise',
    'Territorio:Campania': 'Campania',
    'Territorio:Puglia': 'Puglia',
    'Territorio:Basilicata': 'Basilicata',
    'Territorio:Calabria': 'Calabria',
    'Territorio:Isole':'Territorio:Isole',
    'Territorio:Sicilia': 'Sicilia',
    'Territorio:Sardegna': 'Sardegna'
}

In [226]:
# Separating indicators and regions in the df before substituting, we eliminate nan values to keep only region and respective total indicator

k = i = df['R&D Spending'].loc[0]

df['Check_region'] = [1 if df['R&D Spending'][idx] == k else 0 for idx in range(len(df))]
df['Region_ID'] = df['Check_region'].cumsum()
df['Check_index'] = [1 if df['Check_region'].shift(1)[idx] == 1 else 0 for idx in range(len(df))]
df['Check_region'] = df['Check_region'].replace({0: np.nan})
df['Check_index'] = df['Check_index'].replace({0: np.nan})

df = df.dropna(subset=['Check_index'], how='all').reset_index(drop=True)

df.drop(columns=['Check_region', 'Check_index','Region'], inplace=True)

df.head(50)

,Country,Year,R&D Spending,Region_ID
0,IT,2023,3319321,1
1,IT,2023,30966,2
2,IT,2023,842076,3
3,IT,2023,5838424,4
4,IT,2023,7810293,5
5,IT,2023,616772,6
6,IT,2023,213081,7
7,IT,2023,403691,8
8,IT,2023,2361473,9
9,IT,2023,761679,10


In [227]:
# We now assign region_ID to each row of region according to the order of appearance in the df, we eliminate rows of subtotals and substitute region names using the created map

df['Region_ID'] = df['Region_ID'].astype(str)

ID_territories_map = {
    '1' : 'Piemonte',
    '2' : 'Valle d\'Aosta/Vallée d\'Aoste',
    '3' : 'Liguria',
    '4' : 'Lombardia',
    '5' : 'Territorio:Nord-est',
    '6' : 'Trentino-Alto Adige/Südtirol',
    '7' : 'Provincia Autonoma Bolzano / Bozen',
    '8' : 'Provincia Autonoma Trento',
    '9' : 'Veneto',
    '10' : 'Friuli-Venezia Giulia',
    '11' : 'Emilia-Romagna',
    '12' : 'Territorio:Centro',
    '13' : 'Toscana',
    '14' : 'Umbria',
    '15' : 'Marche',
    '16' : 'Lazio',
    '17' : 'Territorio:Sud',
    '18' : 'Abruzzo',
    '19' : 'Molise',
    '20' : 'Campania',
    '21' : 'Puglia',
    '22' : 'Basilicata',
    '23' : 'Calabria',
    '24' : 'Territorio:Isole',
    '25' : 'Sicilia',
    '26' : 'Sardegna'
}

# Eliminating rows of subtotals: ID 5, 7, 8, 12, 17, 24

df = df[~df['Region_ID'].isin(['5', '7', '8', '12', '17', '24'])].reset_index(drop=True)

# Mapping regions

df['Region_ID'] = df['Region_ID'].map(ID_territories_map)
df.head(21)

,Country,Year,R&D Spending,Region_ID
0,IT,2023,3319321,Piemonte
1,IT,2023,30966,Valle d'Aosta/Vallée d'Aoste
2,IT,2023,842076,Liguria
3,IT,2023,5838424,Lombardia
4,IT,2023,616772,Trentino-Alto Adige/Südtirol
5,IT,2023,2361473,Veneto
6,IT,2023,761679,Friuli-Venezia Giulia
7,IT,2023,4070369,Emilia-Romagna
8,IT,2023,1911194,Toscana
9,IT,2023,237183,Umbria


In [228]:
# Renaming and reordering columns, checking and modifying types

df.columns = ['Country', 'Year', 'R&D Spending', 'Region']
df = df[['Country', 'Region', 'Year', 'R&D Spending']]

df.dropna(inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Country       20 non-null     str   
 1   Region        20 non-null     str   
 2   Year          20 non-null     int64 
 3   R&D Spending  20 non-null     object
dtypes: int64(1), object(1), str(2)
memory usage: 1.0+ KB


In [229]:
# Change year format and R&D spending format

df['Year'] = pd.to_datetime(df['Year'], format='%Y', errors='coerce')
df['R&D Spending'] = pd.to_numeric(df['R&D Spending'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Country       20 non-null     str           
 1   Region        20 non-null     str           
 2   Year          20 non-null     datetime64[us]
 3   R&D Spending  20 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 1.0 KB


In [230]:
# Saving variable

df_R_D_IT = df.copy()

#### Comparison and consolidation

In [231]:
# Grouping R&D datasets of IT and ES

df_R_D_merged = pd.concat([df_R_D_IT, df_R_D_ES], ignore_index=True)
df_R_D_merged.head() 

,Country,Region,Year,R&D Spending
0,IT,Piemonte,2023-01-01,3319321
1,IT,Valle d'Aosta/Vallée d'Aoste,2023-01-01,30966
2,IT,Liguria,2023-01-01,842076
3,IT,Lombardia,2023-01-01,5838424
4,IT,Trentino-Alto Adige/Südtirol,2023-01-01,616772


In [232]:
# Changing shape to combine with df_PIB

df_R_D_merged = df_R_D_merged.melt(
    id_vars=['Country', 'Region', 'Year'],
    var_name='index',
    value_name='value'
)

df_R_D_merged.columns = ['Country', 'Region', 'year', 'index', 'value']
df_R_D_merged.head()

,Country,Region,year,index,value
0,IT,Piemonte,2023-01-01,R&D Spending,3319321
1,IT,Valle d'Aosta/Vallée d'Aoste,2023-01-01,R&D Spending,30966
2,IT,Liguria,2023-01-01,R&D Spending,842076
3,IT,Lombardia,2023-01-01,R&D Spending,5838424
4,IT,Trentino-Alto Adige/Südtirol,2023-01-01,R&D Spending,616772


In [233]:
# We now evaluate the grouping with the PIB df for later analysis

df_PIB_final.head()

,Country,Region,year,index,value
0,IT,Abruzzo,2015-01-01,Primary,985.7
1,IT,Abruzzo,2016-01-01,Primary,1033.1
2,IT,Abruzzo,2017-01-01,Primary,995.4
3,IT,Abruzzo,2018-01-01,Primary,976.2
4,IT,Abruzzo,2019-01-01,Primary,991.8


In [234]:
# Merging the PIB and R&D datasets for later analysis

df_economy_final = pd.concat([df_PIB_final, df_R_D_merged], ignore_index=True)


In [235]:
# Check quality

df_economy_final['index'].value_counts()

index
Primary               370
Secundary             370
Tertiary              370
Tertiary_financial    370
Public                370
Other                 370
Total_GDP             370
R&D Spending           39
Name: count, dtype: int64

In [236]:
# Exporting csv

df_economy_final.to_csv(export_path / "GDP_R_D_Evolution_it_es.csv", index=False)

### Geography

In [239]:
# Calling list of files in this category

df = pd.read_excel(excel_path_bio, sheet_name="Raw", header=0)
df.query('Category == "Geography"')['Name'].values

<ArrowStringArray>
['recintos_autonomicas_inspire_peninbal_etrs89.shp',
                        'limits_IT_regions.geojson',
                               'gi_comuni_cap.xlsx',
                                   'MUNICIPIOS.csv',
                                   'PROVINCIAS.csv']
Length: 5, dtype: str

The three files "gi_comuni_cap", "Municipios" y "Provincias" are in reality only been used as reference to unify regions names and extract lat and lon data.

The other two files contains the GEO limits for viz.

### Demographics

In [242]:
# Calling list of files in this category

df = pd.read_excel(excel_path_bio, sheet_name="Raw", header=0)
df.query('Category == "Demography"')['Name'].values

<ArrowStringArray>
[]
Length: 0, dtype: str

#### Población_Comunidades_autonomas.xlsx

In [243]:
# Importing excel

df = pd.read_excel(files_dict['Población_Comunidades_autonomas.xlsx'])

c:\Users\albet\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [244]:
# Exploring file structure

df.head(20)

,Media de los cuatro trimestres del año,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,Resultados por comunidades autónomas\t,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Población por grupo de edad, sexo y comunidad ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Unidades: Miles Personas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,,Total,NaN,NaN,Menores de 16,NaN,NaN,De 16 a 19 años,NaN,NaN,...,NaN,De 45 a 54 años,NaN,NaN,De 55 a 64 años,NaN,NaN,65 y más años,NaN,NaN
6,,2023,2022.0,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0,...,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0
7,Ambos sexos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Total Nacional,47589.2,47018.1,46835.5,7067.7,7093.0,7181.3,2046.6,1995.4,1943.8,...,6914.5,7800.1,7700.7,7630.4,6763.9,6620.9,6482.9,9482,9268.2,9098.4
9,01 Andalucía,8535,8466.0,8436.5,1346.6,1358.2,1382.1,388.9,383.0,371.0,...,1259.3,1378.9,1365.6,1356.2,1221.5,1193.3,1163.2,1556.4,1514.6,1482.2


In [245]:
df.tail(10)

,Media de los cuatro trimestres del año,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
66,16 País Vasco,1114.3,1109.6,1107.9,150.4,152.7,152.6,41.6,40.2,41.3,...,146.7,176.5,175.3,174.1,168.2,167.2,165.6,285.4,281.2,277.0
67,"17 Rioja, La",159.7,158.4,158.0,22.6,23.1,22.7,6.7,6.2,6.6,...,22.6,25.4,25.1,24.8,23.1,22.7,22.3,36.9,36.2,35.7
68,18 Ceuta,42.1,41.4,41.8,9.6,9.4,8.7,2.4,3.1,2.7,...,5.3,6,5.4,5.4,5.1,5.0,5.5,5.2,6.2,6.8
69,19 Melilla,40.3,40.4,40.5,7.8,8.2,9.5,1.9,1.9,2.3,...,6.2,5.2,5.6,5.7,5.1,5.1,4.5,6.2,4.6,3.8
70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
71,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,Notas:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,Fuente:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,Instituto Nacional de Estadística,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The first 4 and last 5 rows don't contain relevant values.

The data is distributed for region in the first column and years and population class in row 5 and 6 respectively.

There's no imeddiate evidence of a distribution by sex, but we evaluate the file closer.

In [246]:
# Exploring content of row 5 to identify categories

categories = df.iloc[5].dropna().tolist()
categories

[' ',
 'Total',
 'Menores de 16',
 'De 16 a 19 años',
 'De 20 a 24 años',
 'De 25 a 34 años',
 'De 35 a 44 años',
 'De 45 a 54 años',
 'De 55 a 64 años',
 '65 y más años']

In [247]:
# Eliminating first 4 and last 6 rows without relevant information

df = df.iloc[5:-6].reset_index(drop=True, inplace=False)
df.head()

,Media de los cuatro trimestres del año,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,,Total,NaN,NaN,Menores de 16,NaN,NaN,De 16 a 19 años,NaN,NaN,...,NaN,De 45 a 54 años,NaN,NaN,De 55 a 64 años,NaN,NaN,65 y más años,NaN,NaN
1,,2023,2022.0,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0,...,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0
2,Ambos sexos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Total Nacional,47589.2,47018.1,46835.5,7067.7,7093.0,7181.3,2046.6,1995.4,1943.8,...,6914.5,7800.1,7700.7,7630.4,6763.9,6620.9,6482.9,9482,9268.2,9098.4
4,01 Andalucía,8535,8466.0,8436.5,1346.6,1358.2,1382.1,388.9,383.0,371.0,...,1259.3,1378.9,1365.6,1356.2,1221.5,1193.3,1163.2,1556.4,1514.6,1482.2


In [248]:
# Filling empty cells in row 5 of categories with the corresponding value

categorias = df.iloc[0].to_list()

for i in range(len(categorias)):
    if pd.isna(categorias[i]):
        categorias[i] = categorias[i-1]

df = df.astype(object)
df.loc[0] = categorias

df.head()

,Media de los cuatro trimestres del año,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,,Total,Total,Total,Menores de 16,Menores de 16,Menores de 16,De 16 a 19 años,De 16 a 19 años,De 16 a 19 años,...,De 35 a 44 años,De 45 a 54 años,De 45 a 54 años,De 45 a 54 años,De 55 a 64 años,De 55 a 64 años,De 55 a 64 años,65 y más años,65 y más años,65 y más años
1,,2023,2022.0,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0,...,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0,2023,2022.0,2021.0
2,Ambos sexos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Total Nacional,47589.2,47018.1,46835.5,7067.7,7093.0,7181.3,2046.6,1995.4,1943.8,...,6914.5,7800.1,7700.7,7630.4,6763.9,6620.9,6482.9,9482,9268.2,9098.4
4,01 Andalucía,8535,8466.0,8436.5,1346.6,1358.2,1382.1,388.9,383.0,371.0,...,1259.3,1378.9,1365.6,1356.2,1221.5,1193.3,1163.2,1556.4,1514.6,1482.2


In [249]:
# Exploring first column

df_columns = df.columns.to_list()
df_columns[0] = 'Region'

df.columns = df_columns

df['Region'] = df['Region'].astype(str).str.strip()
df['Region'].value_counts()

Region
Total Nacional                    3
01 Andalucía                      3
02 Aragón                         3
03 Asturias, Principado de        3
04 Balears, Illes                 3
05 Canarias                       3
06 Cantabria                      3
07 Castilla y León                3
08 Castilla - La Mancha           3
09 Cataluña                       3
10 Comunitat Valenciana           3
11 Extremadura                    3
12 Galicia                        3
13 Madrid, Comunidad de           3
14 Murcia, Región de              3
15 Navarra, Comunidad Foral de    3
16 País Vasco                     3
17 Rioja, La                      3
18 Ceuta                          3
19 Melilla                        3
                                  2
Ambos sexos                       1
Hombres                           1
Mujeres                           1
Name: count, dtype: int64

We notice 3 values that don't correspond the a region: national total, both sexes, male and female.

We try to understand how the data is distributed to identify correctly the right values and order.

In [250]:
df_filter = df[df['Region'].isin(['Total Nacional','Ambos sexos','Hombres','Mujeres'])]
df_filter

,Region,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
2,Ambos sexos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Total Nacional,47589.2,47018.1,46835.5,7067.7,7093.0,7181.3,2046.6,1995.4,1943.8,...,6914.5,7800.1,7700.7,7630.4,6763.9,6620.9,6482.9,9482,9268.2,9098.4
23,Hombres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,Total Nacional,23313.1,23044.0,22974.7,3641.6,3657.6,3703.9,1058.1,1027.3,1000.0,...,3441.2,3892.7,3850.0,3820.9,3302.6,3234.6,3168.2,4185.3,4084.4,4005.5
44,Mujeres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,Total Nacional,24276.1,23974.1,23860.8,3426.1,3435.4,3477.3,988.6,968.1,943.7,...,3473.3,3907.4,3850.7,3809.4,3461.2,3386.4,3314.6,5296.7,5183.8,5092.9


The data is distributed in categories: both sexes, male, female. After the national total, we find a detail for each region.

In [251]:
# Eliminating rows of total national, we will calculate this data as the sum of the info by autonomous community

df = df[df['Region']!='Total Nacional']
df_filter

,Region,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
2,Ambos sexos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Total Nacional,47589.2,47018.1,46835.5,7067.7,7093.0,7181.3,2046.6,1995.4,1943.8,...,6914.5,7800.1,7700.7,7630.4,6763.9,6620.9,6482.9,9482,9268.2,9098.4
23,Hombres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,Total Nacional,23313.1,23044.0,22974.7,3641.6,3657.6,3703.9,1058.1,1027.3,1000.0,...,3441.2,3892.7,3850.0,3820.9,3302.6,3234.6,3168.2,4185.3,4084.4,4005.5
44,Mujeres,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,Total Nacional,24276.1,23974.1,23860.8,3426.1,3435.4,3477.3,988.6,968.1,943.7,...,3473.3,3907.4,3850.7,3809.4,3461.2,3386.4,3314.6,5296.7,5183.8,5092.9


In [252]:
# Creating check_category and ID_category columns to mark the data by autonomous community accordingly, then we eliminate the data for both
# sexes, which we will calculate as the total of the disaggregated data

df['Check_category'] = df['Region'].apply(lambda x: 1 if x in ['Ambos sexos','Hombres','Mujeres'] else 0)
df['ID_category'] = df['Check_category'].cumsum()

category_map = {
    1 : 'Ambos sexos',
    2 : 'Hombres',
    3 : 'Mujeres'
}

df['Category'] = df['ID_category'].map(category_map)

df = df[df['ID_category']!=1].reset_index(drop=True)

df = df[df['Region'] != 'Hombres'].reset_index(drop=True)
df = df[df['Region'] != 'Mujeres'].reset_index(drop=True)

df = df.drop(columns=['Check_category','ID_category'])

# Rearrange and reorder columns

df = df[['Region', 'Category'] + [col for col in df.columns if col not in ['Region', 'Category']]]
df.rename(columns={'Category':'Sex'}, inplace=True)

# Quality check: 2 values for each region

df['Region'].value_counts()

Region
                                  2
01 Andalucía                      2
02 Aragón                         2
03 Asturias, Principado de        2
04 Balears, Illes                 2
05 Canarias                       2
06 Cantabria                      2
07 Castilla y León                2
08 Castilla - La Mancha           2
09 Cataluña                       2
10 Comunitat Valenciana           2
11 Extremadura                    2
12 Galicia                        2
13 Madrid, Comunidad de           2
14 Murcia, Región de              2
15 Navarra, Comunidad Foral de    2
16 País Vasco                     2
17 Rioja, La                      2
18 Ceuta                          2
19 Melilla                        2
Name: count, dtype: int64

We now try to shift to colums the information about each year and age class.

In [253]:
# Modifying dataset shape to have years in rows and categories in columns

categories = df.iloc[0, 2:].tolist()
years = df.iloc[1, 2:].tolist()

data = df.iloc[2:].copy().reset_index(drop=True)

data_vals = data.iloc[:, 2:].copy()
data_vals.columns = pd.MultiIndex.from_arrays([categories, years], names=['Category', 'Year'])
data_vals.insert(0, 'Region', data['Region'].values)
data_vals.insert(1, 'Sex', data['Sex'].values)

df_long = data_vals.set_index(['Region', 'Sex']).stack(level='Year').reset_index()
df_long

Category,Region,Sex,Year,Total,Menores de 16,De 16 a 19 años,De 20 a 24 años,De 25 a 34 años,De 35 a 44 años,De 45 a 54 años,De 55 a 64 años,65 y más años
0,01 Andalucía,Hombres,2023,4203.6,692.5,200.8,242.4,491.7,597.5,687.6,597.4,693.7
1,01 Andalucía,Hombres,2022.0,4172.7,697.4,198.7,237.1,489.7,610.2,681.5,584.4,673.6
2,01 Andalucía,Hombres,2021.0,4159.9,713.1,189.1,233.0,492.0,627.4,677.3,570.1,657.9
3,02 Aragón,Hombres,2023,642.1,99,28.1,34.3,68.7,85.1,106.3,94.5,126.1
4,02 Aragón,Hombres,2022.0,634.9,98.1,28.0,32.9,66.9,87.1,105.0,93.3,123.6
...,...,...,...,...,...,...,...,...,...,...,...,...
109,18 Ceuta,Mujeres,2022.0,41.4,9.4,3.1,2.0,4.5,5.6,5.4,5.0,6.2
110,18 Ceuta,Mujeres,2021.0,41.8,8.7,2.7,2.6,4.8,5.3,5.4,5.5,6.8
111,19 Melilla,Mujeres,2023,40.3,7.8,1.9,3,6.1,5,5.2,5.1,6.2
112,19 Melilla,Mujeres,2022.0,40.4,8.2,1.9,3.0,6.2,5.7,5.6,5.1,4.6


In [254]:
# Mapping of region names to unify with the ones used in previous dfs

territories = df_long['Region'].unique().tolist()
territories = [r.strip() for r in territories]
territories

['01 Andalucía',
 '02 Aragón',
 '03 Asturias, Principado de',
 '04 Balears, Illes',
 '05 Canarias',
 '06 Cantabria',
 '07 Castilla y León',
 '08 Castilla - La Mancha',
 '09 Cataluña',
 '10 Comunitat Valenciana',
 '11 Extremadura',
 '12 Galicia',
 '13 Madrid, Comunidad de',
 '14 Murcia, Región de',
 '15 Navarra, Comunidad Foral de',
 '16 País Vasco',
 '17 Rioja, La',
 '18 Ceuta',
 '19 Melilla']

In [255]:
# List of regions in df_PIB_ES for comparison

territories_check = df_PIB_ES['Region'].unique().tolist()
territories_check = [r.strip() for r in territories_check]
territories_check

['Andalucía',
 'Aragón',
 'Canarias',
 'Cantabria',
 'Castilla y León',
 'Castilla-La Mancha',
 'Cataluña/Catalunya',
 'Ciudad Autónoma de Ceuta',
 'Ciudad Autónoma de Melilla',
 'Comunidad Foral de Navarra',
 'Comunidad de Madrid',
 'Comunitat Valenciana',
 'Extremadura',
 'Galicia',
 'Illes Balears',
 'La Rioja',
 'País Vasco/Euskadi',
 'Principado de Asturias',
 'Región de Murcia']

In [256]:
# Creating map

territories_map = {
 '01 Andalucía':'Andalucía',
 '02 Aragón':'Aragón',
 '03 Asturias, Principado de':'Principado de Asturias',
 '04 Balears, Illes':'Illes Balears',
 '05 Canarias':'Canarias',
 '06 Cantabria':'Cantabria',
 '07 Castilla y León':'Castilla y León',
 '08 Castilla - La Mancha':'Castilla-La Mancha',
 '09 Cataluña':'Cataluña/Catalunya',
 '10 Comunitat Valenciana':'Comunitat Valenciana',
 '11 Extremadura':'Extremadura',
 '12 Galicia':'Galicia',
 '13 Madrid, Comunidad de':'Comunidad de Madrid',
 '14 Murcia, Región de':'Región de Murcia',
 '15 Navarra, Comunidad Foral de':'Comunidad Foral de Navarra',
 '16 País Vasco':'País Vasco/Euskadi',
 '17 Rioja, La':'La Rioja',
 '18 Ceuta':'Ciudad Autónoma de Ceuta',
 '19 Melilla':'Ciudad Autónoma de Melilla'
 }

# Aplicamos mapa

df_long['Region'] = df_long['Region'].map(territories_map)
df_long['Region'].value_counts()

Region
Andalucía                     6
Aragón                        6
Principado de Asturias        6
Illes Balears                 6
Canarias                      6
Cantabria                     6
Castilla y León               6
Castilla-La Mancha            6
Cataluña/Catalunya            6
Comunitat Valenciana          6
Extremadura                   6
Galicia                       6
Comunidad de Madrid           6
Región de Murcia              6
Comunidad Foral de Navarra    6
País Vasco/Euskadi            6
La Rioja                      6
Ciudad Autónoma de Ceuta      6
Ciudad Autónoma de Melilla    6
Name: count, dtype: int64

In [257]:
# Preparing df for later merging

# Adding country column

df_long['Country'] = 'ES'

# Rearrange and rename columns

df_long = df_long[['Country','Region','Sex','Year','Menores de 16','De 16 a 19 años','De 20 a 24 años','De 25 a 34 años','De 35 a 44 años','De 45 a 54 años','De 55 a 64 años','65 y más años','Total']]

df_columns = df_long.columns.to_list()
len(df_columns)

df_columns[4] = 'Under_16'
df_columns[5] = '16_19'
df_columns[6] = '20_24'
df_columns[7] = '25_34'
df_columns[8] = '35_44'
df_columns[9] = '45_54'
df_columns[10] = '55_64'
df_columns[11] = 'Over_65'

df_long.columns = df_columns
df_long.head()

# Uniform year format and translating to english

df_long['Year'] = df_long['Year'].astype(str).str.replace('2022.0','2022')
df_long['Year'] = df_long['Year'].astype(str).str.replace('2021.0','2021')
df_long['Year'] = pd.to_datetime(df_long['Year'], format='%Y', errors='coerce')

df_long['Sex'] = df_long['Sex'].str.replace('Hombre','Male')
df_long['Sex'] = df_long['Sex'].str.replace('Mujer','Female')

# Transform remaining columns to numeric
# INE data uses '.' as a thousands separator, we remove it before converting

cols = df_long.columns[4:]
df_long[cols] = df_long[cols].astype(int)

df_long.info()


<class 'pandas.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Country   114 non-null    str           
 1   Region    114 non-null    str           
 2   Sex       114 non-null    str           
 3   Year      114 non-null    datetime64[us]
 4   Under_16  114 non-null    int64         
 5   16_19     114 non-null    int64         
 6   20_24     114 non-null    int64         
 7   25_34     114 non-null    int64         
 8   35_44     114 non-null    int64         
 9   45_54     114 non-null    int64         
 10  55_64     114 non-null    int64         
 11  Over_65   114 non-null    int64         
 12  Total     114 non-null    int64         
dtypes: datetime64[us](1), int64(9), str(3)
memory usage: 14.5 KB


In [258]:
# Save df_long as a variable for later integration

df_población_ES = df_long.copy()

#### Población_Regioni_Italiane.xlsx

In [259]:
# Importing excel

df = pd.read_excel(files_dict['Población_Regioni_Italiane.xlsx'])

c:\Users\albet\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [260]:
# Exploring structure

df.head(20)

,"Italia, regioni, province",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,Frequenza: Annuale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Indicatore: Popolazione al 1º gennaio,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Stato civile: Totale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Tempo,,2020,2020,2020,2021,2021,2021,2022,2022,2022,2023,2023,2023,2024,2024,2024,2025,2025,2025
5,Sesso,,Maschi,Femmine,Totale,Maschi,Femmine,Totale,Maschi,Femmine,Totale,Maschi,Femmine,Totale,Maschi,Femmine,Totale,Maschi,Femmine,Totale
6,Territorio,Età,,,,,,,,,,,,,,,,,,
7,Piemonte,0 anni,14145,13640,27785,14031,13139,27170,13729,13029,26758,13265,12689,25954,12987,12209,25196,12591,12067,24658
8,Piemonte,1 anni,14705,14291,28996,14390,13819,28209,14131,13240,27371,13859,13179,27038,13367,12852,26219,13137,12360,25497
9,Piemonte,2 anni,15653,15180,30833,14883,14444,29327,14501,13853,28354,14266,13401,27667,14029,13345,27374,13586,12975,26561


In [261]:
df.tail(10)

,"Italia, regioni, province",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
2241,Sardegna,92 anni,921,2235,3156,1032,2389,3421,1053,2416,3469,1124,2480,3604,1175,2309,3484,1212,2480,3692
2242,Sardegna,93 anni,767,1835,2602,733,1852,2585,816,1979,2795,788,2005,2793,922,2072,2994,944,1931,2875
2243,Sardegna,94 anni,480,1334,1814,590,1482,2072,558,1523,2081,605,1579,2184,627,1665,2292,721,1696,2417
2244,Sardegna,95 anni,374,1127,1501,356,1050,1406,415,1154,1569,412,1162,1574,440,1249,1689,470,1347,1817
2245,Sardegna,96 anni,279,754,1033,274,848,1122,254,847,1101,287,858,1145,295,866,1161,322,966,1288
2246,Sardegna,97 anni,198,579,777,188,572,760,197,640,837,173,587,760,196,635,831,211,658,869
2247,Sardegna,98 anni,126,377,503,134,428,562,128,416,544,132,436,568,119,428,547,134,491,625
2248,Sardegna,99 anni,82,306,388,72,281,353,101,295,396,78,278,356,85,316,401,78,304,382
2249,Sardegna,100 anni e più,97,361,458,122,447,569,118,480,598,124,470,594,116,461,577,125,547,672
2250,Sardegna,Totale,791696,819925,1611621,778110,811934,1590044,778670,808743,1587413,774245,803901,1578146,771282,799171,1570453,767895,794486,1562381


The file is structured as follows:

Columns:

- Years from 2020 to 2025
- Sex and total

Rows:

- Age (Each year in a separate row)
- Marital status? (Need to investigate; mentioned at the beginning of the file)
- Region

Additional Notes

The number of inhabitants does not appear to be grouped by thousands, unlike the ES file.
The header is located in row 4.
There are no columns with irrelevant information at the end of the file.

In [262]:
# Exploring values of first column to understand categorization

df.columns = df.columns.astype(str).str.strip()
df['Italia, regioni, province'] = df['Italia, regioni, province'].astype(str).str.strip()

categories = df['Italia, regioni, province'].unique()
categories

<ArrowStringArray>
[                   'Frequenza: Annuale',
 'Indicatore: Popolazione al 1º gennaio',
                  'Stato civile: Totale',
                                     nan,
                                 'Tempo',
                                 'Sesso',
                            'Territorio',
                              'Piemonte',
        'Valle d'Aosta / Vallée d'Aoste',
                               'Liguria',
                             'Lombardia',
        'Trentino Alto Adige / Südtirol',
    'Provincia Autonoma Bolzano / Bozen',
             'Provincia Autonoma Trento',
                                'Veneto',
                 'Friuli-Venezia Giulia',
                        'Emilia-Romagna',
                               'Toscana',
                                'Umbria',
                                'Marche',
                                 'Lazio',
                               'Abruzzo',
                                'Molise',
               

In [263]:
# Header row 4

df.columns = df.iloc[4]

In [264]:
# Eliminating rows without relevant information and resetting index

df = df.iloc[5:,]
df.reset_index(drop=True, inplace=True)

df = df[df['Tempo']!='Territorio'].reset_index(drop=True)

# Renaming columns

df_col = df.columns.to_list()

df_col[0] = 'region'
df_col[1] = 'category'
df.columns = df_col

# Eliminating columns without relevant information and adding country column for later integration

col = df.columns.to_list()
col_to_drop = [col for col in col if '2020' in col or '2024' in col or '2025' in col]
df = df.drop(columns=col_to_drop)

col = df.columns.to_list()

col[4] = 'drop'
col[7] = 'drop'
col[10] = 'drop'

df.columns = col

col_to_drop = [col for col in col if 'drop' in col]
df = df.drop(columns=col_to_drop)

In [265]:
# Mapping region names to unify with the ones used in df_PIB_IT for later merging

territories = df['region'].unique().tolist()
territories = [r.strip() for r in territories]

territories_check = df_PIB_IT['Region'].unique().tolist()
territories_check = [r.strip() for r in territories_check]

territories_map = {
    "Valle d'Aosta / Vallée d'Aoste": "Valle d'Aosta/Vallée d'Aoste",
    "Trentino Alto Adige / Südtirol": "Trentino-Alto Adige/Südtirol",
    'Piemonte': 'Piemonte',
    'Liguria': 'Liguria',
    'Lombardia': 'Lombardia',
    'Veneto': 'Veneto',
    'Friuli-Venezia Giulia': 'Friuli-Venezia Giulia',
    'Emilia-Romagna': 'Emilia-Romagna',
    'Toscana': 'Toscana',
    'Umbria': 'Umbria',
    'Marche': 'Marche',
    'Lazio': 'Lazio',
    'Abruzzo': 'Abruzzo',
    'Molise': 'Molise',
    'Campania': 'Campania',
    'Puglia': 'Puglia',
    'Basilicata': 'Basilicata',
    'Calabria': 'Calabria',
    'Sicilia': 'Sicilia',
    'Sardegna': 'Sardegna'
}

df['region'] = df['region'].map(territories_map)

# Eliminating rows without region information, excluding row 1

df = df[df['region'].notna() | (df.index == 0)].reset_index(drop=True)

In [266]:
# Changing shape of the df to have years and sex in rows and categories in columns

# Row 0 contains the sex labels for each year column
sex_labels = df.iloc[0, 2:].tolist()

# Eliminating the sex row (row 0) from the data df
df_data = df.iloc[1:].reset_index(drop=True)

year_cols = df_data.columns[2:].tolist()

# Extracting the year from the column names (first 4 characters)
years_parsed = [str(c)[:4] for c in year_cols]

# Constructing new column names: year_sex
df_data = df_data.copy()
df_data.columns = ['region', 'category'] + [f"{yr}_{sx}" for yr, sx in zip(years_parsed, sex_labels)]

# Converting to long format
df_melted = df_data.melt(id_vars=['region', 'category'], var_name='year_sex', value_name='value')

# Splitting year and sex
df_melted[['year', 'sex']] = df_melted['year_sex'].str.split('_', n=1, expand=True)
df_melted = df_melted.drop(columns='year_sex')

# Pivoting categories into columns
df_out = df_melted.pivot_table(
    index=['region', 'year', 'sex'],
    columns='category',
    values='value',
    aggfunc='first'
).reset_index()
df_out.columns.name = None
df_out.head()

# Reordering column '100 anni e più' to the end and removing 'Totale'

df_out.columns = df_out.columns.astype(str).str.strip()
col_order = df_out.columns.tolist()

col_order = [col for col in df_out.columns if col not in ['100 anni e più', 'Totale']]
col_order.append('100 anni e più')

df_out = df_out[col_order]

In [267]:
# Unifying age classes with the df_ES

# Listing unique age classes

class_ref = df_población_ES.columns.to_list()
class_ref

['Country',
 'Region',
 'Sex',
 'Year',
 'Under_16',
 '16_19',
 '20_24',
 '25_34',
 '35_44',
 '45_54',
 '55_64',
 'Over_65',
 'Total']

In [268]:
# Grouping columns to unify format

df_out['Under_16'] = df_out[['0 anni','1 anni','2 anni','3 anni','4 anni','5 anni','6 anni','7 anni','8 anni','9 anni','10 anni','11 anni','12 anni','13 anni','14 anni','15 anni']].sum(axis=1)
df_out['16_19'] = df_out[['16 anni','17 anni','18 anni','19 anni']].sum(axis=1)
df_out['20_24'] = df_out[['20 anni','21 anni','22 anni','23 anni','24 anni']].sum(axis=1)
df_out['25_34'] = df_out[['25 anni','26 anni','27 anni','28 anni','29 anni','30 anni','31 anni','32 anni','33 anni','34 anni']].sum(axis=1)
df_out['35_44'] = df_out[['35 anni','36 anni','37 anni','38 anni','39 anni','40 anni','41 anni','42 anni','43 anni','44 anni']].sum(axis=1)
df_out['45_54'] = df_out[['45 anni','46 anni','47 anni','48 anni','49 anni','50 anni','51 anni','52 anni','53 anni','54 anni']].sum(axis=1)
df_out['55_64'] = df_out[['55 anni','56 anni','57 anni','58 anni','59 anni','60 anni','61 anni','62 anni','63 anni','64 anni']].sum(axis=1)
df_out['Over_65'] = df_out[['65 anni','66 anni','67 anni','68 anni','69 anni','70 anni','71 anni','72 anni','73 anni','74 anni','75 anni','76 anni','77 anni','78 anni','79 anni','80 anni','81 anni','82 anni','83 anni','84 anni','85 anni','86 anni','87 anni','88 anni','89 anni','90 anni','91 anni','92 anni','93 anni','94 anni','95 anni','96 anni','97 anni','98 anni','99 anni','100 anni e più']].sum(axis=1)

# Keeping only relevant columns for further analysis

df_out = df_out[['region', 'year', 'sex', 'Under_16', '16_19', '20_24', '25_34', '35_44', '45_54', '55_64', 'Over_65']]

# Creating total column

df_out['Total'] = df_out[['Under_16', '16_19', '20_24', '25_34', '35_44', '45_54', '55_64', 'Over_65']].sum(axis=1)

# Creating country column

df_out['Country'] = 'IT'

# Reordering and renaming columns

col_order = ['Country', 'region', 'sex','year', 'Under_16', '16_19', '20_24', '25_34', '35_44', '45_54', '55_64', 'Over_65', 'Total']
df_out = df_out[col_order]

df_columns = df_out.columns.astype(str).str.strip().tolist()
df_columns[1] = 'Region'
df_columns[2] = 'Sex'
df_columns[3] = 'Year'
df_out.columns = df_columns

In [269]:
# Modifying values of sex to unify with df_ES

df_out['Sex'] = df_out['Sex'].astype(str).str.strip()
df_out['Sex'] = df_out['Sex'].replace({'Maschi': 'Male', 'Femmine': 'Female'})

In [270]:
# Quality check on total to determine the granularity of population data and unify with df_ES

df_out.groupby('Year')['Total'].sum()

Year
2021    59236213
2022    59030133
2023    58997201
Name: Total, dtype: object

In [271]:
# Converting values to thousands to unify with df_ES

df_out['Under_16'] = df_out['Under_16'] / 1000
df_out['16_19'] = df_out['16_19'] / 1000
df_out['20_24'] = df_out['20_24'] / 1000
df_out['25_34'] = df_out['25_34'] / 1000
df_out['35_44'] = df_out['35_44'] / 1000
df_out['45_54'] = df_out['45_54'] / 1000
df_out['55_64'] = df_out['55_64'] / 1000
df_out['Over_65'] = df_out['Over_65'] / 1000
df_out['Total'] = df_out['Total'] / 1000

In [272]:
# Check final data types

df_out.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Country   120 non-null    str   
 1   Region    120 non-null    str   
 2   Sex       120 non-null    str   
 3   Year      120 non-null    str   
 4   Under_16  120 non-null    object
 5   16_19     120 non-null    object
 6   20_24     120 non-null    object
 7   25_34     120 non-null    object
 8   35_44     120 non-null    object
 9   45_54     120 non-null    object
 10  55_64     120 non-null    object
 11  Over_65   120 non-null    object
 12  Total     120 non-null    object
dtypes: object(9), str(4)
memory usage: 14.9+ KB


In [273]:
# Strip string values

df_out['Region'] = df_out['Region'].astype(str).str.strip()

# Convert year to date

df_out['Year'] = pd.to_datetime(df_out['Year'], format='%Y', errors='coerce')

# Convert population values to integers

cols = df_out.columns[4:]
df_out[cols] = df_out[cols].astype(int)

df_out.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Country   120 non-null    str           
 1   Region    120 non-null    str           
 2   Sex       120 non-null    str           
 3   Year      120 non-null    datetime64[us]
 4   Under_16  120 non-null    int64         
 5   16_19     120 non-null    int64         
 6   20_24     120 non-null    int64         
 7   25_34     120 non-null    int64         
 8   35_44     120 non-null    int64         
 9   45_54     120 non-null    int64         
 10  55_64     120 non-null    int64         
 11  Over_65   120 non-null    int64         
 12  Total     120 non-null    int64         
dtypes: datetime64[us](1), int64(9), str(3)
memory usage: 14.4 KB


In [274]:
df_out.groupby('Year')['Under_16'].sum()

Year
2021-01-01    8186
2022-01-01    8046
2023-01-01    7900
Name: Under_16, dtype: int64

In [275]:
# Save df_out as a variable

df_población_IT = df_out.copy()

#### Comparison and consolidation

In [276]:
# Combining df_población_IT and df_población_ES for later analysis

df_población_merged = pd.concat([df_población_IT, df_población_ES], ignore_index=True)
df_población_merged.head()

,Country,Region,Sex,Year,Under_16,16_19,20_24,25_34,35_44,45_54,55_64,Over_65,Total
0,IT,Abruzzo,Female,2021-01-01,81,22,29,65,80,102,97,176,655
1,IT,Abruzzo,Male,2021-01-01,86,24,32,69,82,99,91,140,625
2,IT,Abruzzo,Female,2022-01-01,80,22,28,64,78,101,99,177,651
3,IT,Abruzzo,Male,2022-01-01,85,24,31,68,80,98,94,141,624
4,IT,Abruzzo,Female,2023-01-01,79,22,28,63,77,99,101,178,649


In [277]:
df_población_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Country   234 non-null    str           
 1   Region    234 non-null    str           
 2   Sex       234 non-null    str           
 3   Year      234 non-null    datetime64[us]
 4   Under_16  234 non-null    int64         
 5   16_19     234 non-null    int64         
 6   20_24     234 non-null    int64         
 7   25_34     234 non-null    int64         
 8   35_44     234 non-null    int64         
 9   45_54     234 non-null    int64         
 10  55_64     234 non-null    int64         
 11  Over_65   234 non-null    int64         
 12  Total     234 non-null    int64         
dtypes: datetime64[us](1), int64(9), str(3)
memory usage: 28.7 KB


In [278]:
# Quality check on population values by year

df_población_merged.groupby(['Country','Year'])['Under_16'].sum()

Country  Year      
ES       2021-01-01    7164
         2022-01-01    7076
         2023-01-01    7051
IT       2021-01-01    8186
         2022-01-01    8046
         2023-01-01    7900
Name: Under_16, dtype: int64

In [279]:
# Changing shape to long

df_población_final = df_población_merged.melt(
    id_vars=['Country', 'Region', 'Sex', 'Year'],
    var_name='Age_Group',
    value_name='Population'
)

df_población_final = df_población_final[df_población_final['Age_Group'] != 'Total'].reset_index(drop=True)
df_población_final.groupby(['Country','Year','Age_Group'])['Population'].sum()

Country  Year        Age_Group
ES       2021-01-01  16_19         1926
                     20_24         2367
                     25_34         5181
                     35_44         6899
                     45_54         7612
                     55_64         6467
                     Over_65       9078
                     Under_16      7164
         2022-01-01  16_19         1978
                     20_24         2421
                     25_34         5182
                     35_44         6683
                     45_54         7686
                     55_64         6607
                     Over_65       9250
                     Under_16      7076
         2023-01-01  16_19         2029
                     20_24         2521
                     25_34         5296
                     35_44         6565
                     45_54         7785
                     55_64         6750
                     Over_65       9466
                     Under_16      7051
IT       

In [280]:
# Unifying sex values
df_población_final['Sex'] = df_población_final['Sex'].replace('Males', 'Male').replace('Femalees', 'Female')

In [281]:
# Exporting csv

df_población_final.to_csv(export_path / "population_it_es.csv", index=False)